# Kaikōura rescue routing: road widths + landslide risk + A* routing (one notebook)

Runs the three X-Band modules in order, each feeding the next:

| Part | What it does | Hands to the next part |
| --- | --- | --- |
| **1. Road widths** (lidar) | Measures every road's width after the 2016 quake and decides if each vehicle fits | `by_osm_id.csv` (width yes / no / unknown per vehicle) |
| **2. Landslide risk** (Random Forest) | Predicts landslides from post-quake Sentinel-2 imagery and checks which roads run through them; scores the RF against the GNS landslide map | `landslide_by_osm_id_rf.csv` (landslide yes / unknown per road) |
| **3. Routing** (A*) | Finds the fastest route each vehicle can drive, charging extra time for width and landslide concerns | Route maps, blockage tables, GeoPackages |

**Before you press Run all**

All settings you might change are in one place: the **Project inputs** cell (the 2nd code cell). In particular:

1. **Earth Engine project:** set `EE_PROJECT` to your own registered Earth Engine project. Colab will ask you to sign in.
2. **GNS access:** Part 2 scores the Random Forest against the GNS landslide inventory (`GNS_SOURCE_ASSET`, `GNS_DEBRIS_ASSET`). Your Google account needs read access to both Earth Engine assets (ask Mingqi).
3. **Random Forest model file:** downloaded from GitHub (`RF_GITHUB_URL`). If that fails, the notebook stops and asks you to get `rf_150_trees.joblib.gz` from ncha509 and upload it to `/content/kaikoura_unet_roads/`.

Everything else (team road file, Kaikōura AOI shapefile, lidar, NZTA widths, OpenStreetMap roads) downloads automatically.
The U-Net model has been removed from this combined notebook; only the Random Forest is run.

Every blockage on a route is labelled **Width** (the vehicle may not fit) or **Landslide risk** (the satellite model saw a possible slip), in the tables and on the maps.

## Step 0 – Install the tools

In [ ]:
# All three parts' tools in one install (road widths + landslide + routing).
# scikit-learn is pinned to the version the saved Random Forest was built with.
%pip install -q geopandas rasterio shapely pyproj requests matplotlib earthengine-api folium scikit-learn==1.6.1 osmnx

## Project inputs

The only settings most people need to change: vehicle widths and the start / end points of the routes.

**Kaikōura only (for now).** The team road file, post-quake lidar, NZTA widths, landslide imagery and AOI shapefile all cover the Kaikōura study area, so widths and landslide checks only exist there. Start and end points outside it will still route, but every road will be "unverified".

In [ ]:
# =============================================================================
# PROJECT INPUTS - edit here, then Run all
# NOTE: set up for KAIKOURA ONLY at the moment. The road widths (lidar), landslide
# model (Sentinel-2) and AOI shapefile cover the Kaikoura study area only; points
# outside it still route, but those roads have no width or landslide check.
# =============================================================================

# ---- Accounts and data access (Part 2) ------------------------------------
EE_PROJECT = 'geo-761'   # your own registered Google Earth Engine project ID; Colab asks you to sign in
# GNS Kaikoura landslide inventory (V3) on Earth Engine: needs read access for your Google account (ask Mingqi)
GNS_SOURCE_ASSET = 'projects/deep-voyage-479406-j9/assets/kaikoura_source_v3'
GNS_DEBRIS_ASSET = 'projects/deep-voyage-479406-j9/assets/kaikoura_debris_v3'
# Saved Random Forest (rf_150_trees.joblib.gz), downloaded from GitHub. If that fails, ask ncha509 for the file
# and upload it to /content/kaikoura_unet_roads/rf_150_trees.joblib.gz before running Part 2.
RF_GITHUB_URL = 'https://raw.githubusercontent.com/NMC31/random_forest_761/master/rf_150_trees.joblib'

# ---- Landslide penalties in routing (Part 3) --------------------------------
USE_LANDSLIDE = True     # False = route on road widths only (to compare routes with and without landslides)
SLIDE_DELAY_MIN = {      # extra minutes charged per road flagged by the Random Forest (starting values, agree with the team)
    "landslide: predicted run >= 50 m": 60,
    "landslide: predicted overlap < 50 m": 10,
    "landslide: road not fully observed": 2,
}

# ---- Vehicles: width of each vehicle (m). Keep the two names (pinzgauer, mhov);
#      the routing maps and reports are built for these two.
VEHICLE_WIDTHS = {
    "pinzgauer": {"label": "Pinzgauer (NZLOV)", "width_m": 1.8},              # NightHawk / Pinzgauer brochure: 1.8 m
    "mhov":      {"label": "MHOV (Rheinmetall MAN HX60)", "width_m": 2.6},    # Deagel 2.6 m (conservative; Wikipedia 2.5 m for HX77)
}
# Spare room needed beside the vehicle (m): mirrors, steering, passing debris (~0.5 m each side).
# A road "fits" if it is at least width_m + CLEARANCE_M wide, e.g. MHOV needs 2.6 + 1.0 = 3.6 m.
CLEARANCE_M = 1.0

# ---- Main route (Part 3, Phase 3): (longitude, latitude) in WGS84 decimal degrees
ROUTE_START_LONLAT = (173.6810, -42.4030)   # Kaikoura township
ROUTE_END_LONLAT = (173.7230, -42.3330)     # Hapuku, north on SH1

# ---- Long-distance case (Part 3, case 3): builds its own corridor graph if outside the Kaikoura AOI
CASE3_START_LONLAT = (172.8460, -42.7720)   # Culverden township
CASE3_END_LONLAT = (173.6810, -42.4030)     # Kaikoura township

# Quick check: is the main route inside the Kaikoura AOI (Kaikoura.shp extent)?
KAIKOURA_BOUNDS = (173.18, -42.57, 174.07, -41.90)   # min lon, min lat, max lon, max lat
for label, (lon, lat) in (("start", ROUTE_START_LONLAT), ("end", ROUTE_END_LONLAT)):
    if not (KAIKOURA_BOUNDS[0] <= lon <= KAIKOURA_BOUNDS[2] and KAIKOURA_BOUNDS[1] <= lat <= KAIKOURA_BOUNDS[3]):
        print(f"Warning: route {label} {(lon, lat)} is outside the Kaikoura study area; width and landslide checks won't cover it")
print("Vehicles:", {v: d["width_m"] + CLEARANCE_M for v, d in VEHICLE_WIDTHS.items()}, "m of road needed")
print("Main route:", ROUTE_START_LONLAT, "->", ROUTE_END_LONLAT)

---
# Part 1 – Road widths from post-quake lidar

Unchanged from `xband_road_widths_explained`. Output used by routing: `/content/xband_roads/handoff/by_osm_id.csv`.

# How wide were Kaikōura's roads after the 2016 earthquake, and could Army trucks get through?

**X-Band group project · road location, width and accuracy · Hinami & Evan**

---

### What this notebook is about

On 14 November 2016 a magnitude 7.8 earthquake struck Kaikōura. It triggered tens of thousands of landslides ([Massey et al., 2018](https://doi.org/10.1785/0120170305)), which buried or narrowed many of the roads in and out of town. When the New Zealand Defence Force (NZDF) sends trucks into a disaster zone, one of the first questions is very practical: **is the road wide enough for our vehicles?**

Our part of the X-Band project answers that question for 120 km of road around Kaikōura. We:

1. take the road lines the team already agreed on (from OpenStreetMap),
2. check those lines sit where the real roads are,
3. **measure how wide each road was after the earthquake**, using a 3D laser scan of the ground (lidar) flown a few weeks after the quake,
4. test our measurements against official road widths from Waka Kotahi NZTA, and
5. turn the widths into a simple *yes / no / unknown* for each vehicle, which our teammates Craig and Neeravh feed into their routing model.

### Where this sits in the team project

| Part of the project | Who | Question it answers |
| --- | --- | --- |
| Radar change detection, U-Net and Random Forest | Mingqi & Tom | *Where are the landslides?* |
| **This notebook** | **Hinami & Evan** | ***Where are the roads, how wide are they after the quake, and how sure are we?*** |
| Routing (A\*, off-road, risk) | Craig & Neeravh | *What is the best way through?* |

> **A note on our approach.** We first thought about measuring road width from satellite images such as Sentinel-2 or Sentinel-1 radar, because they are free and cover the whole region. The problem is that their pixels are about 10 m × 10 m, and Kaikōura's roads are only about 6–9 m wide, so a whole road fits inside a single pixel. The team's U-Net notebook reached the same conclusion: its 10 m output cannot establish remaining vehicle clearance. Lidar measures the ground every metre, which is fine enough to see the edges of a road. Satellites are still very useful for *finding* likely slips, so we see the two as working together rather than competing.

### A few remote-sensing words

| Word | Meaning |
| --- | --- |
| **Remote sensing** | Measuring the Earth from a distance: satellites, planes, drones. |
| **Lidar** | A laser on an aircraft fires pulses at the ground and times how long they take to bounce back, building a 3D map of the surface. |
| **DEM (digital elevation model)** | A grid of ground heights made from lidar. Ours has one height every 1 m × 1 m, with buildings and trees removed. |
| **Raster vs vector** | A raster is a grid of cells (like a photo, e.g. the DEM). A vector is points, lines and shapes (e.g. a road line). |
| **Coordinate reference system (CRS)** | The system used to give every place a position. We use **NZTM (EPSG:2193)**, which measures position in metres, so distances and widths come out in metres. |
| **Cross-section (transect)** | A short line drawn straight across the road. Reading the ground height along it shows the shape of the road, like slicing a loaf of bread to see inside. |
| **Calibration and validation** | Calibration = tuning a method's settings using known answers. Validation = testing it on *different* known answers it wasn't tuned on, which shows honestly how accurate it is. |

### The big idea in one picture

A road is a **flat strip** cut into the landscape. Slicing across it with the lidar gives a height profile like this:

```
  hillside  \
             \__________________________
              |<--- flat = drivable --->|\
                                          \  drop to the sea
```

The width of that flat strip is the road width. Where landslide debris covers part of the road, the flat strip is narrower, so **we are measuring how much road was left after the earthquake.**

## Data and sources

Every dataset we use is open and free to re-use (with attribution).

| Data | What we use it for | Source |
| --- | --- | --- |
| Team road file (`kai_route.gpkg`), from OpenStreetMap | The road lines we measure | [X-Band GitHub repository](https://github.com/Loading-my-brain/x-band); road data © [OpenStreetMap contributors](https://www.openstreetmap.org/copyright) |
| Kaikōura 2016–2017 lidar, 1 m DEM | Ground heights after the earthquake | Captured by AAM Ltd after the quake, commissioned by LINZ: [OpenTopography record, doi:10.5069/G9HD7SWJ](https://doi.org/10.5069/G9HD7SWJ); files served from [New Zealand Elevation on AWS (LINZ)](https://registry.opendata.aws/nz-elevation/) |
| LINZ national 1 m lidar DEM | Backup heights where the 2016–17 survey has gaps | [New Zealand Elevation on AWS (LINZ)](https://registry.opendata.aws/nz-elevation/) |
| NZTA National Road Centreline – RCA data | Official road widths to test our measurements against | [Waka Kotahi NZTA open data](https://opendata-nzta.opendata.arcgis.com/datasets/NZTA::national-road-centreline-road-controlling-authority-data) ([field descriptions](https://opendata-nzta.opendata.arcgis.com/pages/rcl-data-field-descriptions)), CC BY 4.0 |
| Vehicle widths | Pinzgauer 1.8 m; MHOV 2.6 m | [NightHawk (Pinzgauer/NZLOV)](https://www.nighthawk.nz/index.php/3001-new-zealand-army-to-purchase-bushmaster-vehicles-to-replace-its-armoured-pinzgauer-fleet), [Pinzgauer brochure](https://www.scribd.com/document/691296451/Pinzgauer-Brochure); [Deagel (MAN HX)](https://www.deagel.com/Armies/MAN%20HX/a001978), [Wikipedia: RMMV HX range](https://en.wikipedia.org/wiki/RMMV_HX_range_of_tactical_trucks) |

**Background reading we drew on**

* Massey, C. et al. (2018). Landslides triggered by the 14 November 2016 Mw 7.8 Kaikōura Earthquake, New Zealand. *Bulletin of the Seismological Society of America*, 108(3B), 1630–1648. [doi:10.1785/0120170305](https://doi.org/10.1785/0120170305)
* Clode, S., Rottensteiner, F., Kootsookos, P. & Zelniker, E. (2007). Detection and vectorization of roads from lidar data. *Photogrammetric Engineering & Remote Sensing*, 73(5), 517–535. [TRID record](https://trid.trb.org/View/808937). An early study showing lidar can map road centrelines *and widths*.
* Willmott, C. J. & Matsuura, K. (2005). Advantages of the mean absolute error (MAE) over the root mean square error (RMSE) in assessing average model performance. *Climate Research*, 30, 79–82. [Abstract](https://int-res.com/abstracts/cr/v30/n1/p79-82). Why we report MAE as our main error measure.
* James, G., Witten, D., Hastie, T. & Tibshirani, R. *An Introduction to Statistical Learning*. [statlearning.com](https://www.statlearning.com/). The "validation set" idea we use: tune on one half, test on the other.
* NZDF (2016). [A community cut off](https://www.nzdf.mil.nz/army/what-we-do/humanitarian-assistance-and-disaster-relief/a-community-cut-off/): the 27-truck Army convoy that drove the inland route from Culverden to Kaikōura on 17–18 November 2016. We use it as a real-world check.
* General road geometry: road cross-slope is typically about 2% on straight sections and up to about 10% on banked (superelevated) curves ([Wikipedia: Cross slope](https://en.wikipedia.org/wiki/Cross_slope); NZ design values follow the Austroads guides).

*LINZ attribution: elevation data sourced from the LINZ Data Service and licensed for re-use under CC BY 4.0 ([LINZ attribution guidance](https://www.linz.govt.nz/products-services/data/licensing-and-using-data/attributing-elevation-or-aerial-imagery-data)).*

## Settings: every choice in one place

| Setting | Value | Why this value | Why we need it |
| --- | --- | --- | --- |
| `STATION_SPACING_M` | 50 m | Gives 2,428 cross-sections for 120 km. The team's U-Net found continuous landslide overlaps on roads of roughly 60–590 m, so most affected stretches are crossed at least twice, which our "two neighbours" pinch rule needs. | We can't measure every centimetre of road, so we sample it at regular intervals. |
| `HALF_LEN_M` | 20 m | The widest real road in the NZTA data here is about 16 m. ±20 m covers that, plus the up-to-4 m map offset, plus room to see the bank beyond the edge, without reaching across to neighbouring roads or the railway. | Each cross-section must be long enough to "see" both road edges. |
| `SAMPLE_STEP_M` | 0.5 m | The lidar has one height per 1 m. Reading every half-metre means we never jump over an edge between two cells; finer readings would add no real information. | Sets how finely we read heights along each cross-section. |
| `SMOOTH_M` | 1 m | One lidar cell. In our tests on made-up roads, a 1 m smoothing window gave almost no bias (−0.04 m), while 1.5 m made roads look 0.24 m too narrow. | Lidar heights wobble by a few centimetres; smoothing stops that noise looking like a road edge. |
| `CENTRE_SEARCH_M` | 4 m | OSM road lines sat a median of 1.25 m from NZTA's points, with 88% within 5 m (Step 7). 4 m covers most cases without latching onto a nearby verge, carpark or railway. | Maps are never perfectly placed, so we let the method look a little to each side for the true road. |
| `MIN_WIDTH_M` | 2.5 m | The narrowest width NZTA records anywhere in our area is 2.5 m. Anything narrower isn't a normal road, so it's worth flagging. | Flags cross-sections that may be blocked by debris. |
| `ROUGH_MAX_M` | 0.15 m | A sealed road surface should be smooth to within a few centimetres in lidar; 15 cm is about kerb or small-rock height. It's a judgement call, used only for confidence labels and to pick clean stretches for calibration. | Separates clean road surfaces from bumpy ones (possible debris or damage). |
| `GROUP_CELL_M`, `WORKERS` | 1 km, 8 | Purely about computing: each 1 km square needs about a million lidar cells (≈4 MB), which Colab handles easily, and 8 downloads at once is fast without overloading the server. | Has no effect on the results, only on speed and memory. |

In [ ]:
# ---------------------------------------------------------------------------------------------
# 1. Load the toolkits (libraries). "import X as Y" just gives X a shorter nickname Y.
# ---------------------------------------------------------------------------------------------
from pathlib import Path                      # file paths (where things are saved)
import hashlib, json, math, time, os, shutil, datetime, urllib.request   # small built-in helpers
from concurrent.futures import ThreadPoolExecutor, as_completed          # do several downloads at once

import numpy as np                            # fast maths on big lists of numbers (all our heights)
import pandas as pd                           # tables, like Excel inside Python
import geopandas as gpd                       # tables where each row also has a shape on a map
import matplotlib.pyplot as plt               # graphs and maps
import requests, pyogrio, rasterio, shapely   # downloads, map-file reading, raster (lidar) reading, geometry
from rasterio.merge import merge              # stitch several lidar tiles together
from rasterio.enums import Resampling
from rasterio.warp import transform_bounds    # convert a box between coordinate systems
from scipy.ndimage import map_coordinates     # read a height *between* lidar cells (interpolation)
from shapely.geometry import box, Point, MultiPoint, LineString   # basic shapes
from shapely.ops import unary_union           # merge several shapes into one
from pyproj import Transformer                # convert coordinates, e.g. NZTM metres <-> latitude/longitude

# ---------------------------------------------------------------------------------------------
# 2. Study area and data (identical to the rest of the team, so our results line up)
# ---------------------------------------------------------------------------------------------
AOI_WGS84 = (173.5, -42.5, 174.0, -42.0)     # area of interest: min longitude, min latitude, max longitude, max latitude
CRS = "EPSG:2193"                             # NZTM, New Zealand's map system in METRES (widths must be in metres)
ROOT = Path("/content/xband_roads"); ROOT.mkdir(exist_ok=True)   # folder where every output is saved

# The team's agreed road file, pinned to one exact version. The "commit" is the version; the SHA-256 is a
# digital fingerprint. If the file ever changed, the fingerprint would not match and the notebook stops,
# so everyone in the team always analyses exactly the same roads (reproducibility).
ROAD_COMMIT = "8d739079424b3b6c7077f470346a8840f20a0e1c"
ROAD_SHA256 = "127841d08d83cfc7a3ec82cf34f850d92d6d66550e650cea4304a2a1aff26b0e"
ROAD_URL = f"https://raw.githubusercontent.com/Loading-my-brain/x-band/{ROAD_COMMIT}/kaikoura_road_route/kai_route.gpkg"

# Web address of NZTA's road-width data, our "answer sheet" for testing accuracy.
NZTA_LAYER_URL = ("https://services.arcgis.com/CXBb7LAjgIIdcsPt/arcgis/rest/services/"
                  "National_Road_Centreline_Road_Controlling_Authority_data/FeatureServer/0")

# Lidar sources, tried in this order. The 2016-17 survey was flown after the quake (what we want); the
# national DEM only fills gaps and mostly shows the rebuilt road (flown 2018 onwards).
DEM_SOURCES = {   # tried in this order
    1: ("Kaikoura 2016-17 lidar (post-quake)",
        "https://nz-elevation.s3.ap-southeast-2.amazonaws.com/canterbury/kaikoura_2016-2017/dem_1m/2193/collection.json"),
    2: ("LINZ national 1 m lidar (mostly 2018+)",
        "https://nz-elevation.s3.ap-southeast-2.amazonaws.com/new-zealand/new-zealand/dem_1m/2193/collection.json"),
}

# OpenStreetMap road types that vehicles can drive on. Anything else (the railway, cycleways, steps) is removed.
DRIVABLE = {"motorway", "motorway_link", "trunk", "trunk_link", "primary", "primary_link", "secondary",
            "secondary_link", "tertiary", "tertiary_link", "unclassified", "residential", "living_street",
            "service", "track", "road"}

# ---------------------------------------------------------------------------------------------
# 3. Cross-sections (see the table above for the reasoning behind each value)
# ---------------------------------------------------------------------------------------------
# 50 m: ~2,400 slices for 120 km. The team's U-Net found landslide overlaps on roads of ~60-590 m, so most
# affected stretches get at least two slices, which the "two neighbours" pinch rule (Step 8) needs.
# Trade-off we accept: a pinch shorter than ~50 m could slip between two slices.
STATION_SPACING_M = 50      # one cross-section every 50 m
# 20 m: widest NZTA road here is ~16 m; +-20 m covers that, the up-to-4 m map offset, and the bank beyond
# the edge, without reaching neighbouring roads or the railway beside SH1.
HALF_LEN_M = 20.0           # each reaches 20 m either side of the centreline
# 0.5 m: half a lidar cell, so no road edge is skipped between two 1 m cells; finer adds nothing real.
SAMPLE_STEP_M = 0.5         # a height reading every 0.5 m
# 1 m: one lidar cell. Removes centimetre-scale noise; in tests on made-up roads it gave ~0 bias, while
# wider smoothing blurred the edges and made roads look narrower.
SMOOTH_M = 1.0              # smoothing before measuring slope
# 4 m: OSM lines sit a median 1.25 m from NZTA's points (88% within 5 m). Bigger values risk grabbing a
# nearby verge or carpark instead of the road.
CENTRE_SEARCH_M = 4.0       # the road may start up to 4 m from the mapped centreline
# 2.5 m: the narrowest width NZTA records in our area. Narrower than this is unusual -> flag it.
MIN_WIDTH_M = 2.5           # narrower than this = 'narrow'
# 15 cm: about kerb / small-rock height. A clean sealed road is smoother than this in the lidar.
# A judgement call; only used for confidence labels and to choose clean stretches for calibration.
ROUGH_MAX_M = 0.15          # smoother than this = clean road surface
# Computing only (no effect on results): 1 km squares keep memory small (~4 MB each); 8 downloads at once.
GROUP_CELL_M, WORKERS = 1000, 8

# ---------------------------------------------------------------------------------------------
# 4. Width rules: what counts as "road"?
# ---------------------------------------------------------------------------------------------
# A rule has two parts:
#   slope_max = the steepest tilt still counted as road (0.15 = a 15% slope, about 8.5 degrees)
#   dev_max   = how far (m) the ground may drop below or rise above the road surface before the road "ends"
#               (None = no height limit)
# Real roads tilt about 2% on straights and up to ~10% on banked curves, so slope limits of 5-15% bracket
# that range; height limits of 10-50 cm bracket kerbs (~15 cm) through to ditches (~0.3-1 m).
STAGE2_RULE = {"slope_max": 0.15, "dev_max": None}            # first version, kept for the before/after comparison
SLOPE_GRID = [0.05, 0.06, 0.08, 0.10, 0.12, 0.15]              # calibration tries every combination
DEV_GRID = [None, 0.50, 0.30, 0.20, 0.15, 0.10]
# Drivable width for passability: a truck drives over bumps and cracks, so only steps over 30 cm stop it.
# Slope 15% sits above the ~10% steepest banked curve. The 30 cm is an assumption, tested in Step 8:
# the answer is the same at 30 cm and 50 cm, so the result doesn't hinge on this exact number.
PASS_RULE = {"slope_max": 0.15, "dev_max": 0.30}               # assumption: a truck drives over steps < 30 cm

# ---------------------------------------------------------------------------------------------
# 5. Matching NZTA records to our roads, and calibration
# ---------------------------------------------------------------------------------------------
# 30 m: an NZTA stretch matches our road if its start AND end points are within 30 m of it. Most genuine
#       matches are within 10 m; 30 m allows for map error without grabbing a parallel road.
# 10 m: skip the first/last 10 m of each stretch, because stretches usually start and end at junctions.
MATCH_TOL_M, END_TRIM_M = 30, 10
# (0.7, 1.4): the length along our road must be 70-140% of NZTA's recorded length, otherwise it's
#             probably the wrong road. (2.5, 20): NZTA widths outside this are treated as typing errors
#             (an earlier check found one record of 44.7 m; the widest real road here is ~16 m).
LEN_RATIO_OK, NZTA_W_OK = (0.7, 1.4), (2.5, 20.0)
# 3: a stretch needs at least 3 clean slices so its median width isn't driven by one odd reading.
# 761: fixes the random 50/50 calibration/validation split so it's identical every run (the same seed
#      the team used for the Random Forest).
MIN_STATIONS, SEED = 3, 761

# ---------------------------------------------------------------------------------------------
# 6. Vehicles (sources in the "Data and sources" section)
# ---------------------------------------------------------------------------------------------
# VEHICLE_WIDTHS: set in the Project inputs cell at the top
# 1 m: room for mirrors, steering and passing debris safely (~0.5 m each side). An assumption; widen it to be
# more cautious. So the MHOV needs 2.6 + 1.0 = 3.6 m of road.
# CLEARANCE_M: set in the Project inputs cell at the top (assumption: about 0.5 m spare each side)
# 0.5: only answer yes/no if at least half of a road's slices could be measured; otherwise say "unknown".
MIN_SHARE = 0.5             # need usable widths on half a road's cross-sections to answer yes/no
# When combining pieces of one road, "no" beats "unknown" beats "yes" (worst case = safest for a convoy).
RANK = {"no": 0, "unknown": 1, "yes": 2}

FORCE_RERUN = False         # True = ignore saved downloads/profiles and fetch again

# ---------------------------------------------------------------------------------------------
# 7. Technical set-up (no decisions here)
# ---------------------------------------------------------------------------------------------
# The 81 positions along every cross-section: -20.0, -19.5, ... 0 ... +19.5, +20.0 m
OFFSETS = np.round(np.arange(-HALF_LEN_M, HALF_LEN_M + 1e-9, SAMPLE_STEP_M), 6)
# Settings that make reading lidar files over the internet faster
os.environ.update({"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.tiff",
                   "GDAL_HTTP_MULTIRANGE": "YES", "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES"})
# The study-area box converted into NZTM metres
aoi_nztm = gpd.GeoSeries([box(*AOI_WGS84)], crs="EPSG:4326").to_crs(CRS).iloc[0]
print("Settings loaded.")

In [ ]:
# ---------------------------------------------------------------------------------------------
# Map furniture: every map/route figure gets a north arrow (top right) and a scalebar (bottom left).
# Matplotlib maps are drawn in NZTM metres, so the scalebar is exact; arrow points to grid north
# (differs from true north by under 1 degree around Kaikoura).
# ---------------------------------------------------------------------------------------------
from matplotlib.offsetbox import AnchoredOffsetbox, AuxTransformBox, DrawingArea, TextArea, VPacker
from matplotlib.patches import Polygon as _MplPolygon, Rectangle as _MplRect


def _nice_length(span_m):
    target = span_m / 5
    mag = 10 ** math.floor(math.log10(target))
    return max(m * mag for m in (1, 2, 5) if m * mag <= target)


def add_map_furniture(ax, arrow_loc="upper right", bar_loc="lower left"):
    """Add a north arrow and a metric scalebar to a matplotlib map axis in metre units (NZTM)."""
    w, h = 22, 34
    da = DrawingArea(w, h)
    da.add_artist(_MplPolygon([(w / 2, h), (w, 0), (w / 2, h * 0.24), (0, 0)], closed=True,
                              facecolor="white", edgecolor="black", lw=2.2, joinstyle="miter"))
    n = TextArea("N", textprops=dict(fontsize=15, fontweight="bold", color="black"))
    arrow = AnchoredOffsetbox(loc=arrow_loc, child=VPacker(children=[n, da], align="center", pad=0, sep=3),
                              pad=0.4, borderpad=0.8, frameon=True)
    arrow.patch.set(facecolor="white", alpha=0.85, edgecolor="none", boxstyle="round,pad=0.2")
    arrow.set_zorder(20)
    ax.add_artist(arrow)

    x0, x1 = ax.get_xlim(); y0, y1 = ax.get_ylim()
    length = _nice_length(abs(x1 - x0))
    nseg = 4 if (length / 4) == int(length / 4) else 2
    seg, bar_h = length / nseg, length * 0.05
    box = AuxTransformBox(ax.transData)
    for i in range(nseg):
        box.add_artist(_MplRect((i * seg, 0), seg, bar_h, facecolor="black" if i % 2 == 0 else "white",
                                edgecolor="black", lw=1))
    km = length >= 1000
    fmt = (lambda v: f"{v / 1000:g}") if km else (lambda v: f"{v:g}")
    ticks = [0, length / 2, length] if nseg == 4 else [0, length]
    labels = AuxTransformBox(ax.transData)
    for t in ticks:
        txt = fmt(t) + (f" {'km' if km else 'm'}" if t == length else "")
        labels.add_artist(plt.Text(t, 0, txt, ha="center" if t < length else "left", va="center", fontsize=9))
    bar = AnchoredOffsetbox(loc=bar_loc, child=VPacker(children=[box, labels], align="left", pad=0, sep=6),
                            pad=0.5, borderpad=0.8, frameon=True)
    bar.patch.set(facecolor="white", alpha=0.85, edgecolor="none", boxstyle="round,pad=0.2")
    bar.set_zorder(20)
    ax.add_artist(bar)
    return ax


_FOLIUM_NORTH_SVG = ("<svg width='34' height='58' viewBox='0 0 34 58' xmlns='http://www.w3.org/2000/svg'>"
                     "<text x='17' y='17' text-anchor='middle' font-family='Arial,sans-serif' "
                     "font-weight='bold' font-size='18'>N</text>"
                     "<polygon points='17,22 30,55 17,46 4,55' fill='white' stroke='black' "
                     "stroke-width='2.6' stroke-linejoin='miter'/></svg>")


def add_folium_furniture(m):
    """Add a north arrow (top right) and a metric scalebar (bottom left) to a folium map.
    Call before folium.LayerControl so the arrow stacks above the layer list."""
    from branca.element import MacroElement, Template
    el = MacroElement()
    el._template = Template("""
    {% macro script(this, kwargs) %}
    (function(){
      var map = {{ this._parent.get_name() }};
      var North = L.Control.extend({options:{position:'topright'}, onAdd:function(){
        var d = L.DomUtil.create('div');
        d.style.cssText = 'background:rgba(255,255,255,.85);padding:4px 6px;border-radius:6px;line-height:0';
        d.title = 'North';
        d.innerHTML = "SVG_HERE";
        return d;}});
      new North().addTo(map);
      L.control.scale({position:'bottomleft', metric:true, imperial:false, maxWidth:200}).addTo(map);
    })();
    {% endmacro %}""".replace("SVG_HERE", _FOLIUM_NORTH_SVG))
    m.add_child(el)
    return m

## The heart of the method: measuring one cross-section

This cell defines the **functions** (reusable recipes) that do the core thinking. Nothing is measured yet; we're only teaching Python *how* to measure.

### How `measure_width` decides where the road is

Imagine walking across the road with a spirit level:

1. **Smooth** the 81 height readings slightly, then work out how steep the ground is at each one.
2. **Start** at the flat reading closest to where the map says the road is.
3. **Fit the road surface**: from the 2 m around that starting point, work out the road's height and tilt.
4. **Walk outwards** left and right, one reading at a time, while the ground is **gentle enough** (`slope_max`) **and** stays **close to the road surface** (`dev_max`). Where each walk stops is a road edge.
5. **Width** = distance between the two edges.

Each measurement also gets a **status**:

| Status | Meaning |
| --- | --- |
| `ok` | both edges found |
| `narrow` | both edges found, but under 2.5 m (possible debris) |
| `edge_not_found` | flat ground runs off the end of the cross-section (e.g. flat farmland), so the width is *at least* this much |
| `no_flat_centre` | nothing flat near the mapped line: debris, a steep cutting, or the map line is in the wrong place |
| `no_dem` | no lidar here |

### Why we needed two rules (a lesson learned)

Our **first version** only checked steepness. When we compared it with NZTA, the roads came out **about 5 m too wide**: gently sloping grass verges and gravel shoulders looked "flat enough", so the walk kept going. Adding the **height limit** (`dev_max`) fixed this: the walk now stops where the ground drops below the road surface.

But we then found the opposite problem. The calibrated rule is so strict (it stops at 10 cm bumps) that it marked the **Inland Road as impassable**, even though 27 Army trucks drove it in November 2016. A truck happily drives over a 10 cm bump. So we now answer two separate questions:

* **Carriageway width** (strict, calibrated): how wide is the sealed road? This is what we test against NZTA.
* **Drivable width** (`PASS_RULE`, lenient): how much room does a truck have? This decides passability.

### The pinch rule

A single cross-section can give a strange reading: a pothole, a parked vehicle in the lidar, or just noise. So a road only counts as narrow if **two neighbouring cross-sections** (50 m apart) are both narrow.

### Self-tests

At the bottom we run the functions on made-up cross-sections where we *know* the right answer (for example a perfect 8 m road cut into a hillside). If any test fails, the notebook stops and tells us, so we can trust the functions before using them on real data.

In [ ]:
def measure_width(z, slope_max, dev_max=None, offsets=OFFSETS, step=SAMPLE_STEP_M, core_m=2.0,
                  smooth_m=SMOOTH_M, centre_search=CENTRE_SEARCH_M, min_width=MIN_WIDTH_M):
    # Measure the road width in ONE cross-section.
    #   z         = the 81 ground heights along the cross-section (metres)
    #   slope_max = steepest tilt still counted as road;  dev_max = max height step from the road surface
    # Returns a small dictionary: width, left/right edge positions, tilt, bumpiness and a status label.
    out = {"width_m": np.nan, "left_m": np.nan, "right_m": np.nan, "centre_offset_m": np.nan,
           "crossfall": np.nan, "roughness_m": np.nan, "status": "no_dem"}
    z = np.asarray(z, dtype=float)
    near_c = np.abs(offsets) <= centre_search                 # readings within 4 m of the mapped line
    if not np.isfinite(z[near_c]).any():                       # no lidar near the road at all
        return out
    # Step 1: smooth slightly, then slope = change in height per metre at every reading
    k = max(1, int(round(smooth_m / step)))
    zs = pd.Series(z).rolling(k, center=True, min_periods=k).mean().to_numpy()
    slope = np.gradient(zs, step)
    flat = np.isfinite(slope) & (np.abs(slope) <= slope_max)  # True where the ground is gentle enough
    # Step 2: start from the flat reading closest to the mapped centreline
    cand = np.where(flat & near_c)[0]
    if not len(cand):
        out["status"] = "no_flat_centre"
        return out
    i0 = cand[np.argmin(np.abs(offsets[cand]))]
    ok = flat.copy()
    # Step 3: fit the road surface (a straight line: height + tilt) from the 2 m around the start, and also
    # require every reading to stay within dev_max of that surface
    if dev_max is not None:
        core = flat & (np.abs(offsets - offsets[i0]) <= core_m) & np.isfinite(zs)
        b, a = np.polyfit(offsets[core], zs[core], 1) if core.sum() >= 3 else (0.0, zs[i0])
        with np.errstate(invalid="ignore"):
            ok &= np.abs(zs - (a + b * offsets)) <= dev_max
        ok[i0] = True
    # Step 4: walk left (i) and right (j) while the ground still counts as road
    i = i0
    while i > 0 and ok[i - 1]:
        i -= 1
    j = i0
    while j < len(z) - 1 and ok[j + 1]:
        j += 1
    # An edge is only "real" if the walk stopped at real ground, not at the end of the data
    left_real = i > 0 and np.isfinite(slope[i - 1])
    right_real = j < len(z) - 1 and np.isfinite(slope[j + 1])
    # Tilt (crossfall) and bumpiness (roughness) of the measured strip
    seg = np.arange(i, j + 1)
    good = np.isfinite(z[seg])
    if good.sum() >= 2:
        p = np.polyfit(offsets[seg][good], z[seg][good], 1)
        out["crossfall"] = float(p[0])
        out["roughness_m"] = float(np.std(z[seg][good] - np.polyval(p, offsets[seg][good])))
    # The slope at the last road reading is computed using the bank's height too, so it already looks
    # steep: the true edge is on average one reading further out on each side.
    out["left_m"], out["right_m"] = float(offsets[i] - step), float(offsets[j] + step)
    # Step 5: width = distance between the edges. centre_offset = how far the real road centre is from the map line
    out["width_m"] = out["right_m"] - out["left_m"]
    out["centre_offset_m"] = (out["left_m"] + out["right_m"]) / 2
    out["status"] = ("edge_not_found" if not (left_real and right_real)
                     else "narrow" if out["width_m"] < min_width else "ok")
    return out


def sustained_pinch(widths):
    # Narrowest width that holds at two neighbouring cross-sections (NaN = not usable).
    # e.g. [8, 8, 1, 8, 8] -> 8 (one odd reading ignored);  [8, 2, 2, 8] -> 2 (a real pinch)
    w = np.asarray(widths, dtype=float)
    if len(w) < 2:
        return np.nan
    both = np.isfinite(w[:-1]) & np.isfinite(w[1:])
    return float(np.min(np.maximum(w[:-1], w[1:])[both])) if both.any() else np.nan


def passability(pinch, share, is_bridge, is_tunnel, need):
    # 'yes' / 'no' / 'unknown' and the reason, for every road at once.
    #   unknown -> bridge/tunnel (lidar can't see the deck) or too little of the road measured
    #   no      -> the pinch is narrower than vehicle width + clearance (`need`)
    #   yes     -> otherwise
    pinch, share = np.asarray(pinch, float), np.asarray(share, float)
    bt = np.asarray(is_bridge, bool) | np.asarray(is_tunnel, bool)
    thin = (share < MIN_SHARE) | ~np.isfinite(pinch)
    conds = [bt, thin, pinch < need]
    return (np.select(conds, ["unknown", "unknown", "no"], default="yes"),
            np.select(conds, ["bridge/tunnel: lidar cannot see the deck", "too little of the road could be measured",
                              "sustained pinch narrower than vehicle + clearance"], default=""))


def worst(s):
    # The worst answer in a list: "no" beats "unknown" beats "yes"
    return min(s, key=RANK.get)


def attach_road_widths(G, by_osm, vehicles=None):
    # For the routing notebook (Craig & Neeravh): copy our widths and passable_<vehicle> answers onto their
    # road network, matching on the OSM way id (their edge attribute 'osmid' = our column 'osm_id').
    # If one of their road links covers several OSM ways, it takes the worst answer.
    vehicles = vehicles or [c[len("passable_"):] for c in by_osm.columns if c.startswith("passable_")]
    lut = by_osm.assign(osm_id=pd.to_numeric(by_osm["osm_id"], errors="coerce")).dropna(subset=["osm_id"])
    lut = lut.set_index(lut["osm_id"].astype("int64"))
    n = 0
    for u, v, k, d in G.edges(keys=True, data=True):
        ids = d.get("osmid")
        ids = ids if isinstance(ids, (list, tuple, set)) else [ids]
        ids = [int(i) for i in pd.to_numeric(pd.Series(list(ids)), errors="coerce").dropna()]
        rows = lut.loc[lut.index.intersection(ids)]
        if not len(rows):
            continue
        for col, how in (("drive_pinch_m", "min"), ("width_median_m", "median")):
            if col in rows and rows[col].notna().any():
                d[col] = float(getattr(rows[col], how)())
        for veh in vehicles:
            d[f"passable_{veh}"] = worst(rows[f"passable_{veh}"])
        n += 1
    return n


def _self_tests():
    # Made-up cross-sections with known answers. If any check fails, Python stops with an error.
    import networkx as nx
    s = OFFSETS
    old_rule = {"slope_max": 0.15, "dev_max": None}
    # 1. 8 m road benched into a hillside -> should measure ~8 m
    z = np.where(s < -3, 10 + (-3 - s) * 0.7, np.where(s <= 5, 10 + 0.03 * s, 10 + 0.15 - (s - 5) * 0.9))
    r = measure_width(z, **old_rule)
    assert r["status"] == "ok" and abs(r["width_m"] - 8) <= 0.75, r
    # 2. flat plain -> no edges found;  3. no lidar -> 'no_dem'
    assert measure_width(np.full_like(s, 50.0), **old_rule)["status"] == "edge_not_found"
    assert measure_width(np.full_like(s, np.nan), **old_rule)["status"] == "no_dem"
    # 4. 7 m road with gently sloping grass verges: the old slope-only rule over-measures, the height limit fixes it
    a = np.abs(s)
    zv = np.where(a <= 3.5, 10 - 0.03 * a, np.where(a <= 7.5, 9.895 - 0.12 * (a - 3.5),
                  np.where(a <= 16, 9.415, 9.415 + (a - 16) * 0.8)))
    w_old, w_new = measure_width(zv, **old_rule)["width_m"], measure_width(zv, 0.15, 0.25)["width_m"]
    assert w_old > 25 and 7 <= w_new <= 11, (w_old, w_new)
    # 5. pinch rule: one noisy reading doesn't count, two neighbours do; yes/no/unknown logic
    assert sustained_pinch([8, 8, 1, 8, 8]) == 8 and sustained_pinch([8, 2, 2, 8]) == 2
    ans, _ = passability([2.5, 9, 9, 9], [1, 1, 0.2, 1], [0, 0, 0, 1], [0, 0, 0, 0], need=3.0)
    assert list(ans) == ["no", "yes", "unknown", "unknown"], ans
    # 6. joining onto a tiny routing network (one link merges two OSM ways -> takes the worst answer)
    T = nx.MultiDiGraph()
    T.add_edge(0, 1, osmid=111); T.add_edge(1, 2, osmid=[222, 333]); T.add_edge(2, 3, osmid=999)
    tab = pd.DataFrame({"osm_id": [111, 222, 333], "width_median_m": [8.0, 7.0, 3.0],
                        "drive_pinch_m": [7.5, 6.5, 2.0], "passable_mhov": ["yes", "yes", "no"]})
    assert attach_road_widths(T, tab) == 2 and T[1][2][0]["passable_mhov"] == "no" and "passable_mhov" not in T[2][3][0]
    print(f"Self-tests passed: 8 m bench -> {r['width_m']:.1f} m; sloping verges -> old rule {w_old:.1f} m, "
          f"height limit {w_new:.1f} m (true 7 m); single noisy readings don't close roads; routing join works.")

_self_tests()

## Helper functions: downloading data and placing the cross-sections

More recipes, this time for the GIS "plumbing". Again nothing runs yet. In plain words, these functions:

* **`fetch_arcgis_points`** asks NZTA's web service for every road record inside our study area. The service only hands out 2,000 at a time, so it keeps asking until it has them all.
* **`stac_tiles`** reads LINZ's catalogue (called a *STAC catalogue*) to find which lidar tiles exist and where each one is. LINZ stores the lidar as many square files ("tiles") in a cloud-friendly format, so we only download the bits we need.
* **`make_stations`** walks along every road and drops a cross-section every 50 m, pointing straight across the road. It also flags cross-sections that touch a *different* road (a junction), because the flat area there is the intersection, not the road.
* **`read_dem_window`** and **`sample_dem`** download the lidar for a small area and read the ground height at each of the 81 points on a cross-section. Points fall *between* lidar cells, so the height is blended from the four nearest cells (bilinear interpolation).
* **`heights_for_group`** does that for all cross-sections in one 1 km square, trying the post-quake lidar first and filling any gaps from the national lidar.
* **`measure_all`** runs `measure_width` on every cross-section.
* **`pinch_per_piece`** works out the pinch width for each piece of road.

In [ ]:
def fetch_arcgis_points(layer_url, bbox_wgs84, page=2000):
    # Download every NZTA record inside a longitude/latitude box, returned in NZTM metres.
    feats, offset = [], 0
    while True:                                   # keep asking until the server says there are no more
        params = {"where": "1=1", "outFields": "*", "returnGeometry": "true",
                  "geometry": ",".join(map(str, bbox_wgs84)), "geometryType": "esriGeometryEnvelope",
                  "inSR": 4326, "spatialRel": "esriSpatialRelIntersects", "outSR": 2193,
                  "resultOffset": offset, "resultRecordCount": page, "f": "json"}
        for attempt in range(3):                  # internet hiccup? try up to 3 times
            try:
                r = requests.get(layer_url + "/query", params=params, timeout=180)
                r.raise_for_status(); js = r.json(); break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(3 * (attempt + 1))
        if "error" in js:
            raise RuntimeError(f"NZTA service error: {js['error']}")
        batch = js.get("features", [])
        feats += batch
        if not batch or not js.get("exceededTransferLimit"):
            break
        offset += len(batch)
    # Turn the server's records into a map table (one point per record)
    rows = []
    for f in feats:
        a, g = dict(f["attributes"]), f.get("geometry") or {}
        a["geometry"] = (Point(g["x"], g["y"]) if "x" in g else
                         MultiPoint([tuple(p[:2]) for p in g["points"]]) if g.get("points") else None)
        rows.append(a)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=CRS)


def stac_tiles(collection_url, cache_path):
    # List every lidar tile in a LINZ catalogue with its map extent; saved to disk so it's only done once.
    if cache_path.exists() and not FORCE_RERUN:
        return [(u, tuple(b)) for u, b in json.loads(cache_path.read_text())]
    base = collection_url.rsplit("/", 1)[0] + "/"
    coll = json.load(urllib.request.urlopen(collection_url, timeout=120))
    hrefs = [l["href"] for l in coll["links"] if l["rel"] == "item"]
    def one(h):                                   # read one tile's description
        for attempt in range(3):
            try:
                item = json.load(urllib.request.urlopen(base + h.lstrip("./"), timeout=60)); break
            except Exception:
                if attempt == 2:
                    raise
                time.sleep(2 * (attempt + 1))
        href = next(a["href"] for a in item["assets"].values() if a["href"].endswith((".tif", ".tiff")))
        return (href if href.startswith("http") else base + href.lstrip("./"),
                transform_bounds("EPSG:4326", CRS, *item["bbox"]))
    with ThreadPoolExecutor(32) as ex:            # read many tile descriptions at once
        tiles = list(ex.map(one, hrefs))
    cache_path.write_text(json.dumps(tiles))
    return tiles


def make_stations(lines, spacing, half_len):
    # Drop a cross-section every `spacing` m along every road.
    # Each one stores its centre (cx, cy), the direction straight across the road (nx, ny) and the line itself.
    rows = []
    cols = ["road_id", "parent_id", "highway", "name", "is_bridge", "is_tunnel"]
    for rec, g in zip(lines[cols].to_dict("records"), lines.geometry):
        L = g.length
        if L < 2:
            continue
        # short roads get one cross-section in the middle; longer ones one every `spacing` m
        for d in (np.arange(spacing / 2, L, spacing) if L >= spacing else np.array([L / 2])):
            # direction of the road here (from 2 m behind to 2 m ahead), then turn it 90 degrees
            p, a, b = g.interpolate(d), g.interpolate(max(d - 2, 0)), g.interpolate(min(d + 2, L))
            n = math.hypot(b.x - a.x, b.y - a.y)
            if n == 0:
                continue
            nx_, ny_ = -(b.y - a.y) / n, (b.x - a.x) / n
            rows.append({**rec, "station_id": f"{rec['road_id']}@{int(round(d))}", "chainage_m": float(d),
                         "cx": p.x, "cy": p.y, "nx": nx_, "ny": ny_, "geometry": p,
                         "transect": LineString([(p.x - nx_ * half_len, p.y - ny_ * half_len),
                                                 (p.x + nx_ * half_len, p.y + ny_ * half_len)])})
    st = gpd.GeoDataFrame(rows, geometry="geometry", crs=lines.crs)
    st["station_id"] += st.groupby("station_id").cumcount().map(lambda k: "" if k == 0 else f"_{k}")
    # Junction: the cross-section touches a different OSM road
    i_st, i_rd = lines.sindex.query(gpd.GeoSeries(st["transect"], crs=st.crs), predicate="intersects")
    other = lines["parent_id"].to_numpy()[i_rd] != st["parent_id"].to_numpy()[i_st]
    st["near_junction"] = False
    st.loc[np.unique(i_st[other]), "near_junction"] = True
    return st.reset_index(drop=True)


def read_dem_window(tiles, bounds):
    # Download and stitch together the lidar covering one small box (NZTM). Returns heights + their position.
    xmin, ymin, xmax, ymax = bounds
    hits = [u for u, b in tiles if b[0] < xmax and b[2] > xmin and b[1] < ymax and b[3] > ymin]
    if not hits:
        return None
    srcs = [rasterio.open(u) for u in hits]
    try:
        arr, tr = merge(srcs, bounds=bounds, res=1.0, nodata=-9999, resampling=Resampling.nearest)
    finally:
        for s in srcs:
            s.close()
    a = arr[0].astype("float32"); a[a <= -9998] = np.nan     # -9999 means "no data" -> NaN (not a number)
    return a, tr


def sample_dem(arr, tr, xs, ys):
    # Height at any map position, blended from the 4 nearest lidar cells; NaN next to missing data.
    cols, rows = ~tr * (xs, ys)                               # map position -> grid position
    rows, cols = np.asarray(rows) - 0.5, np.asarray(cols) - 0.5
    z = map_coordinates(np.nan_to_num(arr, nan=0.0), [rows, cols], order=1, mode="constant", cval=np.nan)
    ok = map_coordinates(np.isfinite(arr).astype("float32"), [rows, cols], order=1, mode="constant", cval=0.0)
    z[ok < 0.999] = np.nan                                    # don't trust a blend that touched missing data
    return z


def heights_for_group(st_group, tiles_by_source):
    # Heights along every cross-section in one 1 km square: post-quake lidar first, national lidar fills gaps.
    # Also records which source each height came from (1 = post-quake, 2 = national, 0 = none).
    xs = st_group["cx"].to_numpy()[:, None] + st_group["nx"].to_numpy()[:, None] * OFFSETS[None, :]
    ys = st_group["cy"].to_numpy()[:, None] + st_group["ny"].to_numpy()[:, None] * OFFSETS[None, :]
    bounds = (math.floor(xs.min()) - 2, math.floor(ys.min()) - 2, math.ceil(xs.max()) + 2, math.ceil(ys.max()) + 2)
    Z = np.full(xs.shape, np.nan, dtype="float32"); SRC = np.zeros(xs.shape, dtype="uint8")
    for code_ in sorted(tiles_by_source):
        need = ~np.isfinite(Z)
        if not need.any():
            break
        w = read_dem_window(tiles_by_source[code_], bounds)
        if w is None:
            continue
        z = sample_dem(*w, xs[need], ys[need]); got = np.isfinite(z)
        Z[need] = np.where(got, z, np.nan); SRC[need] = np.where(got, code_, 0)
    return Z, SRC


def measure_all(rule, idx=None):
    # Run the width rule on every saved cross-section (or only the ones listed in idx).
    idx = range(len(Z_ALL)) if idx is None else idx
    return pd.DataFrame([measure_width(Z_ALL[i], **rule) for i in idx], index=list(idx))


def pinch_per_piece(widths, ok):
    # For every piece of road: its pinch width (two-neighbour rule) and the share of cross-sections usable.
    o = stations.sort_values(["road_id", "chainage_m"]).index          # cross-sections in order along each road
    w = pd.Series(np.where(ok, widths, np.nan), index=stations.index).loc[o]
    rid = stations.loc[o, "road_id"]
    return pd.DataFrame({"pinch": w.groupby(rid).apply(lambda s: sustained_pinch(s.to_numpy())),
                         "share": w.notna().groupby(rid).mean()})

print("Data functions ready.")

## Step 1 – Load the team's roads (and keep only the drivable ones)

**What happens:** we download the team's road file, check its fingerprint, convert it to NZTM metres, cut it to our study area and split multi-part lines into single pieces. Each piece gets an ID (`road_id`), while we keep the original OpenStreetMap ID (`full_id` / `osm_id`), because that's what the routing team joins on.

**Something that surprised us:** the "road" file isn't only roads. It also contains about **75 km of the Main North Line railway** (lines with no road tag at all), **19 km of cycleways** and a few steps. None of these carry trucks, and a cycleway running beside SH1 could even steal measurements from the highway. So we keep only drivable road types. Reassuringly, what's left adds up to **120.2 km**, exactly the total the team's U-Net notebook uses, so we know we're working on the same road network as everyone else.

We also note which pieces are **bridges** or **tunnels**. Lidar ground models remove bridge decks (you see the river underneath), so we can never measure a bridge's width this way and always label them "unknown".

In [ ]:
# Download the team's road file (only the first time) and check it's the agreed version
road_path = ROOT / "kai_route.gpkg"
if not road_path.exists():
    r = requests.get(ROAD_URL, timeout=120); r.raise_for_status(); road_path.write_bytes(r.content)
assert hashlib.sha256(road_path.read_bytes()).hexdigest() == ROAD_SHA256, "Road file is not the team's agreed version."

# Read it, convert to NZTM metres, cut to the study area, and split multi-part lines into single lines
roads = gpd.read_file(road_path, layer=pyogrio.list_layers(road_path)[0][0]).to_crs(CRS)
roads = gpd.clip(roads[roads.geometry.notna() & ~roads.geometry.is_empty], aoi_nztm)
roads = roads.explode(index_parts=False).reset_index(drop=True)
roads = roads[roads.geom_type == "LineString"].reset_index(drop=True)
# IDs: parent_id = the original OSM road; road_id = our numbered piece of it
roads["parent_id"] = roads["full_id"].astype(str)
roads["road_id"] = roads["parent_id"] + ":piece" + roads.groupby("parent_id").cumcount().astype(str)
roads["length_m"] = roads.length

# Keep only drivable road types (the railway, cycleways and steps are removed)
hw = roads["highway"].astype("string")
keep = hw.isin(DRIVABLE).fillna(False).astype(bool)
print("Removed (km):", (roads[~keep].groupby(hw[~keep].fillna("(no highway tag - railway)")).length_m.sum() / 1000)
      .round(1).to_dict())
drive = roads[keep].copy().reset_index(drop=True)

def has_tag(df, col):
    # True where an OSM tag such as bridge=yes or tunnel=yes is set
    if col not in df.columns:
        return pd.Series(False, index=df.index)
    v = df[col].astype("string").str.lower().fillna("no")
    return (~v.isin(["no", "0", "false", "none", ""])).fillna(False).astype(bool)

drive["is_bridge"], drive["is_tunnel"] = has_tag(drive, "bridge"), has_tag(drive, "tunnel")
if "name" not in drive.columns:
    drive["name"] = pd.NA
print(f"Drivable roads: {len(drive)} pieces, {drive.length_m.sum()/1000:.1f} km "
      f"| bridges: {drive.is_bridge.sum()} pieces, tunnels: {drive.is_tunnel.sum()}")

## Step 2 – Load the NZTA road widths (our "answer sheet")

**What happens:** we download NZTA's road records for our study area (only the first time; after that the saved copy is used), drop old superseded records, and pull out the **carriageway width** in metres.

**Why NZTA?** It's the official record kept by the agencies that look after the roads, so it's a defensible thing to test against. It has a few catches we have to keep in mind:

* It describes **today's roads**, and much of SH1 was rebuilt after the quake.
* It gives **one typical width per stretch of road**, not the width at every point.
* Its geometry is just the **start and end points** of each stretch, so in Step 4 we have to work out which of our cross-sections fall between them.

The printed table shows how many records have a width and what typical widths look like for each road class (for example, *National* = SH1).

In [ ]:
nzta_path = ROOT / "nzta_rca_aoi.gpkg"
if nzta_path.exists() and not FORCE_RERUN:
    nzta = gpd.read_file(nzta_path)                     # use the saved copy
else:
    nzta = fetch_arcgis_points(NZTA_LAYER_URL, AOI_WGS84)
    for c in [c for c in nzta.columns if c.lower().endswith("date")]:     # dates arrive as milliseconds since 1970
        nzta[c] = pd.to_datetime(nzta[c], unit="ms", errors="coerce")
    nzta.to_file(nzta_path, driver="GPKG")
if "retiredDate" in nzta.columns:                       # retired = an old, superseded version of the record
    nzta = nzta[nzta["retiredDate"].isna()].copy()
nzta = nzta.reset_index(drop=True)
nzta["width_m"] = pd.to_numeric(nzta[next(c for c in nzta.columns if c.lower() == "width")], errors="coerce")
nzta.loc[nzta.width_m <= 0, "width_m"] = np.nan         # 0 or negative = not recorded

print(f"NZTA records in the AOI: {len(nzta)}, with a width: {nzta.width_m.notna().mean():.0%}")
print(nzta.groupby("ONRCClass").width_m.agg(["count", "mean", "min", "max"]).round(1).to_string())

## Step 3 – Place the cross-sections and read the lidar

**What happens:**

1. Look up which lidar tiles exist near our roads.
2. Drop a cross-section every 50 m on every drivable road (about 2,400 of them).
3. For each one, read the ground height every 0.5 m along its 40 m length.

This is the slowest step the first time (a few minutes of downloading), so the heights are saved to `lidar_profiles.npz` and simply reloaded on later runs.

**What to look for in the output:** the last line says where the heights came from. In our run **all 2,428 cross-sections used the post-quake 2016–17 lidar**, which is the best case: every width describes the road as it was a few weeks after the earthquake, not after the rebuild.

In [ ]:
# 1. Which lidar tiles are near our roads?
road_box = drive.total_bounds + np.array([-50, -50, 50, 50])
TILES = {}
for code_, (label, url) in DEM_SOURCES.items():
    allt = stac_tiles(url, ROOT / f"stac_tiles_{code_}.json")
    TILES[code_] = [(u, b) for u, b in allt if b[0] < road_box[2] and b[2] > road_box[0]
                    and b[1] < road_box[3] and b[3] > road_box[1]]
    print(f"{label}: {len(TILES[code_])} tiles near the roads")

# 2. Cross-sections every 50 m
stations = make_stations(drive, STATION_SPACING_M, HALF_LEN_M)
print(f"Cross-sections: {len(stations)} | at junctions: {stations.near_junction.sum()} | "
      f"on bridges: {stations.is_bridge.sum()} | in tunnels: {stations.is_tunnel.sum()}")

# 3. Heights along every cross-section (reuse the saved file if it matches)
prof_path = ROOT / "lidar_profiles.npz"
saved = np.load(prof_path) if prof_path.exists() and not FORCE_RERUN else None
if saved is not None and list(saved["station_id"]) == stations.station_id.tolist() and np.allclose(saved["offsets"], OFFSETS):
    Z_ALL, SRC_ALL = saved["Z"], saved["SRC"]
    print("Loaded saved lidar heights.")
else:
    Z_ALL = np.full((len(stations), len(OFFSETS)), np.nan, dtype="float32")   # one row per cross-section
    SRC_ALL = np.zeros(Z_ALL.shape, dtype="uint8")
    # group cross-sections into 1 km squares so each lidar download stays small
    cell = (np.floor(stations["cx"] / GROUP_CELL_M).astype(int).astype(str) + "_" +
            np.floor(stations["cy"] / GROUP_CELL_M).astype(int).astype(str))
    groups = list(stations.groupby(cell).groups.values())
    t0 = time.time()
    with ThreadPoolExecutor(WORKERS) as ex:
        futs = {ex.submit(heights_for_group, stations.loc[idx], TILES): idx for idx in groups}
        for n, f in enumerate(as_completed(futs), 1):
            Z_ALL[futs[f]], SRC_ALL[futs[f]] = f.result()
            if n % 25 == 0 or n == len(groups):
                print(f"  lidar read for {n}/{len(groups)} squares ({time.time()-t0:.0f} s)", flush=True)
    np.savez_compressed(prof_path, Z=Z_ALL, SRC=SRC_ALL, offsets=OFFSETS,
                        station_id=stations.station_id.to_numpy().astype(str))
stations["dem_src"] = SRC_ALL[:, len(OFFSETS) // 2]      # which lidar the centre point came from
print("Height source at the centreline:",
      {DEM_SOURCES.get(k, ("none",))[0]: int(v) for k, v in stations.dem_src.value_counts().items()})

## Step 4 – Line up NZTA's road stretches with our cross-sections

To compare widths fairly, we need to know **which of our cross-sections lie on the same stretch of road as each NZTA record**.

**How:**

1. Join our road pieces that share a name into continuous lines (e.g. all of *State Highway 1*).
2. An NZTA record matches a line when **both** its start and end points are within 30 m of it.
3. Measure how far along the line the start and end fall, and take the cross-sections in between, skipping 10 m at each end (usually junctions).
4. **Sanity check:** if the length along our line is very different from NZTA's recorded length, it's probably the wrong road, so we drop it.

**What to expect:** most NZTA records are labelled *"road not in team file"*. That's not an error. NZTA covers every road in the area, while the team's file only covers the main route corridor. The remaining *matched* records (86 in our run) are what we test against.

In [ ]:
# 1. Continuous lines per road name (unnamed pieces stay on their own)
drive["name_key"] = np.where(drive["name"].notna() & (drive["name"].astype(str) != ""),
                             drive["name"].astype(str), "~" + drive["road_id"])
comps = []
for key, grp in drive.groupby("name_key"):
    merged = shapely.line_merge(unary_union(list(grp.geometry)))
    for k, g in enumerate([merged] if merged.geom_type == "LineString" else list(merged.geoms)):
        comps.append({"comp_id": f"{key}#{k}", "name_key": key, "geometry": g})
comps = gpd.GeoDataFrame(comps, geometry="geometry", crs=CRS)

# Each cross-section: which line it is on, and how far along it (chainage)
stations["name_key"] = stations["road_id"].map(drive.set_index("road_id")["name_key"])
stations["comp_id"], stations["chain_m"] = None, np.nan
for key, idx in stations.groupby("name_key").groups.items():
    cands = comps[comps.name_key == key]
    d = np.column_stack([stations.loc[idx, "geometry"].distance(g).to_numpy() for g in cands.geometry])
    for row, b in zip(idx, d.argmin(axis=1)):
        stations.at[row, "comp_id"] = cands.comp_id.iloc[b]
        stations.at[row, "chain_m"] = cands.geometry.iloc[b].project(stations.at[row, "geometry"])

# NZTA start/end coordinates and recorded lengths
num = lambda c: pd.to_numeric(nzta.get(c), errors="coerce").to_numpy()
sx, sy, ex, ey = (num(c) for c in ("assetCwayStartNZTMX", "assetCwayStartNZTMY", "assetCwayEndNZTMX", "assetCwayEndNZTMY"))
rec_len = num("carriagewayEnd") - num("carriagewayStart")
st_comp, st_chain = stations.comp_id.to_numpy(), stations.chain_m.to_numpy()

# 2-4. Match every NZTA record (or record why it couldn't be matched)
rec_rows, pairs = [], []
for r in range(len(nzta)):
    base = {"rec": r, "nzta_width_m": nzta.width_m.iat[r], "onrc": nzta.ONRCClass.iat[r], "road": nzta.fullRoadName.iat[r]}
    if not np.isfinite([sx[r], sy[r], ex[r], ey[r]]).all():
        rec_rows.append({**base, "status": "no coordinates"}); continue
    if not (NZTA_W_OK[0] <= base["nzta_width_m"] <= NZTA_W_OK[1]):
        rec_rows.append({**base, "status": "implausible width"}); continue
    ps, pe = Point(sx[r], sy[r]), Point(ex[r], ey[r])
    both = list(set(comps.sindex.query(ps.buffer(MATCH_TOL_M), predicate="intersects")) &
                set(comps.sindex.query(pe.buffer(MATCH_TOL_M), predicate="intersects")))
    if not both:
        rec_rows.append({**base, "status": "road not in team file"}); continue
    c = both[int(np.argmin([max(comps.geometry.iloc[i].distance(ps), comps.geometry.iloc[i].distance(pe)) for i in both]))]
    g = comps.geometry.iloc[c]
    c0, c1 = sorted((g.project(ps), g.project(pe)))
    if c1 - c0 < 2 * END_TRIM_M + STATION_SPACING_M:
        rec_rows.append({**base, "status": "too short"}); continue
    if np.isfinite(rec_len[r]) and rec_len[r] > 0 and not (LEN_RATIO_OK[0] <= (c1 - c0) / rec_len[r] <= LEN_RATIO_OK[1]):
        rec_rows.append({**base, "status": "length mismatch"}); continue
    sel = np.where((st_comp == comps.comp_id.iloc[c]) & (st_chain >= c0 + END_TRIM_M) & (st_chain <= c1 - END_TRIM_M))[0]
    rec_rows.append({**base, "status": "matched", "gap_start_m": g.distance(ps), "gap_end_m": g.distance(pe)})
    pairs += [(r, s) for s in sel]

recs, pairs = pd.DataFrame(rec_rows), pd.DataFrame(pairs, columns=["rec", "st"])
print(recs.status.value_counts().to_string())

## Step 5 – Calibrate the width rule, then test it honestly

This is where we find out **how accurate our widths really are**.

**Calibration:** we try all 36 combinations of `slope_max` × `dev_max` from the settings. For each NZTA stretch we take the **median** width of its clean cross-sections (status `ok`, smooth surface, not at a junction, bridge or tunnel) and compare it with NZTA's width.

**Keeping ourselves honest:** if we picked the best setting *and* reported its accuracy on the same records, we'd be marking our own homework. So we randomly split the matched NZTA stretches 50/50 ([James et al.'s "validation set" approach](https://www.statlearning.com/)):

* the **calibration half** is used to choose the best setting (lowest average error), and
* the **validation half** is kept aside and only used at the end to report accuracy, because the choice never saw it.

**The error measures:**

* **Bias** = average of (lidar − NZTA). Positive means our widths are too wide.
* **MAE** (mean absolute error) = the average size of the error, ignoring whether it's too wide or too narrow. Our main measure, because it's the easiest to interpret ([Willmott & Matsuura, 2005](https://int-res.com/abstracts/cr/v30/n1/p79-82)).
* **RMSE** = like MAE, but big misses count extra.
* **r** = correlation: do wider NZTA roads also come out wider in the lidar?

**What we found in our run:** the first version was **5.5 m too wide** on average; the calibrated rule is within **0.4 m on average (MAE 1.7 m)**, and best on SH1 (MAE 1.4 m). The correlation dropped a little (0.74 → 0.65), which tells us the calibrated widths are pulled towards the average: narrow roads read a bit wide and wide roads a bit narrow. It's an honest limitation worth reporting, as is the small sample (about 40 validation stretches).

In [ ]:
# Random 50/50 split of the matched NZTA stretches (fixed by SEED, so it's the same every run)
matched = recs[recs.status == "matched"].copy()
matched["half"] = np.where(np.random.default_rng(SEED).random(len(matched)) < 0.5, "calibration", "validation")
nzta_w = matched.set_index("rec")["nzta_width_m"]
pairs = pairs[pairs.rec.isin(matched.rec)]
exclude = (stations.is_bridge | stations.is_tunnel | stations.near_junction).to_numpy()


def record_widths(rule):
    # Median width of the clean cross-sections on each NZTA stretch, using a given rule
    res = measure_all(rule, sorted(pairs.st.unique()))
    clean = (res.status == "ok") & (res.roughness_m <= ROUGH_MAX_M) & ~exclude[res.index]
    p = pairs[pairs.st.map(clean)]
    agg = p.assign(w=p.st.map(res.width_m)).groupby("rec").w.agg(["median", "size"])
    return agg.loc[agg["size"] >= MIN_STATIONS, "median"]


def scores(lidar, keep):
    # Bias, MAE, RMSE and correlation between our widths and NZTA's, for the records in `keep`
    e = (lidar - nzta_w.reindex(lidar.index))[lambda s: s.index.isin(keep)]
    if len(e) < 2:
        return {"n": len(e), "bias_m": np.nan, "mae_m": np.nan, "rmse_m": np.nan, "r": np.nan}
    return {"n": len(e), "bias_m": e.mean(), "mae_m": e.abs().mean(), "rmse_m": np.sqrt((e ** 2).mean()),
            "r": np.corrcoef(lidar.loc[e.index], nzta_w.loc[e.index])[0, 1]}


# Try all 36 settings on the calibration half and choose the one with the lowest MAE
# (among settings that still measure at least half as many stretches as the best-covered one)
cal_recs, val_recs = matched.rec[matched.half == "calibration"], matched.rec[matched.half == "validation"]
grid = pd.DataFrame([{"slope_max": sm, "dev_max": "none" if dm is None else dm,
                      **scores(record_widths({"slope_max": sm, "dev_max": dm}), cal_recs)}
                     for sm in SLOPE_GRID for dm in DEV_GRID])
ok_rows = grid[(grid.n >= 0.5 * grid.n.max()) & grid.mae_m.notna()]
if len(ok_rows):
    b = ok_rows.loc[ok_rows.mae_m.idxmin()]
    BEST = {"slope_max": float(b.slope_max), "dev_max": None if b.dev_max == "none" else float(b.dev_max)}
else:
    BEST = dict(STAGE2_RULE)
    print("WARNING: too few matched stretches to calibrate - using the first-version rule.")
print(f"Calibrated rule: slope_max = {BEST['slope_max']}, dev_max = {BEST['dev_max']}  "
      f"(best of {len(grid)} settings on {int(grid.n.max())} calibration stretches)\n")

# Headline accuracy: first version vs calibrated, on the validation half (never used for choosing)
lw_old, lw_cal = record_widths(STAGE2_RULE), record_widths(BEST)
acc = pd.DataFrame([{"rule": "First version (slope only)", "records": "validation", **scores(lw_old, val_recs)},
                    {"rule": "Calibrated",                 "records": "validation", **scores(lw_cal, val_recs)},
                    {"rule": "Calibrated",                 "records": "all matched", **scores(lw_cal, matched.rec)}])
print(acc.round(2).to_string(index=False))

cmp = pd.DataFrame({"nzta_width_m": nzta_w, "lidar_first_m": lw_old, "lidar_cal_m": lw_cal,
                    "half": matched.set_index("rec").half, "onrc": matched.set_index("rec").onrc}).dropna(subset=["lidar_cal_m"])
cmp["error_m"] = cmp.lidar_cal_m - cmp.nzta_width_m
print("\nCalibrated error by road class (validation half):")
print(cmp[cmp.half == "validation"].groupby("onrc").error_m
      .agg(n="size", bias_m="mean", mae_m=lambda e: e.abs().mean()).round(2).to_string())

# Scatter plot: each dot is one NZTA stretch; dots on the black line = perfect agreement
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharex=True, sharey=True)
lim = [0, max(25, np.nanmax(cmp[["nzta_width_m", "lidar_first_m", "lidar_cal_m"]].to_numpy()) + 1)]
for ax, col, title in ((axes[0], "lidar_first_m", "First version (slope only)"), (axes[1], "lidar_cal_m", "Calibrated rule")):
    for h, mk in (("calibration", "o"), ("validation", "s")):
        d = cmp[cmp.half == h]
        ax.scatter(d.nzta_width_m, d[col], s=18, marker=mk, alpha=0.7, label=f"{h} half")
    ax.plot(lim, lim, color="black", lw=1, label="perfect agreement")
    ax.set(xlim=lim, ylim=lim, aspect="equal", title=title,
           xlabel="NZTA carriageway width (m)", ylabel="lidar width, median of stretch (m)")
    ax.legend(fontsize=8)
plt.tight_layout(); fig.savefig(ROOT / "fig_lidar_vs_nzta.png", dpi=150); plt.show()

## Step 6 – Measure every cross-section: carriageway width and drivable width

Now we apply both rules to all ~2,400 cross-sections:

* **Carriageway width** (the calibrated rule from Step 5): the sealed road, comparable with NZTA.
* **Drivable width** (`PASS_RULE`): room for a vehicle, where small bumps and cracks don't stop the measurement. Flat ground running off the end of a cross-section (`edge_not_found`) still counts here, because the road is *at least* that wide.

Cross-sections on bridges, in tunnels or at junctions are skipped. We then summarise each **piece of road**: its typical width (median), how much of it we could measure, and its **pinch width** (the narrowest drivable width holding across two neighbouring cross-sections).

**Confidence labels:** `high` = measured on the post-quake lidar with a smooth surface; `medium` = measured, but rougher or from the national lidar; `low` = not measured.

In [ ]:
skip = stations.is_bridge | stations.is_tunnel | stations.near_junction

# Carriageway width (calibrated rule)
cal = measure_all(BEST)
for c in cal.columns:
    stations[c] = cal[c].to_numpy()
stations.loc[stations.is_bridge, "status"] = "bridge"
stations.loc[stations.is_tunnel, "status"] = "tunnel"
stations["usable"] = stations.status.isin(["ok", "narrow"]) & ~skip
stations["confidence"] = np.select(
    [stations.usable & (stations.dem_src == 1) & (stations.roughness_m <= ROUGH_MAX_M), stations.usable],
    ["high", "medium"], default="low")

# Drivable width (passability rule)
drv = measure_all(PASS_RULE)
stations["drive_width_m"], stations["drive_status"] = drv.width_m.to_numpy(), drv.status.to_numpy()
stations["drive_usable"] = stations.drive_status.isin(["ok", "narrow", "edge_not_found"]) & ~skip

print("Carriageway rule status:", stations.status.value_counts().to_dict())
print("Drivable rule status:   ", stations.drive_status[~skip].value_counts().to_dict())
u = stations[stations.usable]
print(f"\nCarriageway width by OSM class ({len(u)} usable cross-sections):")
print(u.groupby("highway").width_m.describe()[["count", "mean", "25%", "50%", "75%"]].round(1).to_string())

# One row per road piece
agg = u.groupby("road_id").width_m.agg(n_usable="size", width_median_m="median")
pp = pinch_per_piece(stations.drive_width_m, stations.drive_usable)
base_cols = ["road_id", "parent_id", "osm_id", "name", "highway", "is_bridge", "is_tunnel", "length_m", "geometry"]
pieces = drive[[c for c in base_cols if c in drive.columns]].merge(stations.groupby("road_id").agg(
            n_stations=("station_id", "size"),
            share_high=("confidence", lambda c: (c == "high").mean()),
            n_flagged=("status", lambda s: s.isin(["narrow", "no_flat_centre"]).sum())),
         on="road_id", how="left").merge(agg, on="road_id", how="left")
pieces["n_usable"] = pieces.n_usable.fillna(0).astype(int)
pieces["share_measured"] = (pieces.n_usable / pieces.n_stations).fillna(0)
pieces["drive_pinch_m"] = pieces.road_id.map(pp.pinch)
pieces["drive_share"] = pieces.road_id.map(pp.share).fillna(0)
print(f"\nRoad pieces with a carriageway width: {(pieces.n_usable > 0).sum()} of {len(pieces)}")

## Step 7 – Are the roads where the map says they are?

Part of our job was to check **where** the roads are, not only how wide. We do that two independent ways:

1. **NZTA points → team road line:** for stretches that exist in both datasets, how far NZTA's start and end points sit from our road line. **This is our main position result.**
2. **Lidar road centre → team road line:** at every cross-section where both edges were found, the middle of the measured road is compared with the mapped line.

*An honest caveat about check 2:* the method only looks for a road within 4 m of the mapped line (`CENTRE_SEARCH_M`), so it physically can't report a bigger offset. Its numbers are therefore optimistic. One of our example cross-sections in Step 9 shows SH1 that appears to be mapped about 10 m off, which check 2 would miss.

**In our run:** median 1.25 m, 88% within 5 m and 95% within 10 m. The team's road lines are generally well placed.

In [ ]:
def summary(name, g):
    # Typical distance, RMSE, 90th percentile, and share within 5 m / 10 m
    g = pd.Series(g).dropna()
    return {"check": name, "n": len(g), "median_m": g.median(), "rmse_m": np.sqrt((g ** 2).mean()),
            "p90_m": g.quantile(0.9), "within_5m": (g <= 5).mean(), "within_10m": (g <= 10).mean()}

pos = pd.DataFrame([summary("NZTA points to team line", pd.concat([matched.gap_start_m, matched.gap_end_m])),
                    summary("Lidar road centre to team line",
                            stations.loc[stations.usable & (stations.status == "ok"), "centre_offset_m"].abs())])
print(pos.round(2).to_string(index=False))

## Step 8 – Can each vehicle get through?

**The rule for each road:**

* **no**: the pinch width is narrower than the vehicle's width plus 1 m clearance (MHOV: 3.6 m; Pinzgauer: 2.8 m),
* **unknown**: a bridge or tunnel, or less than half the road could be measured,
* **yes**: otherwise.

If a road is split into several pieces, we take the **worst** answer (`no` beats `unknown` beats `yes`), because a convoy is only as safe as the narrowest point.

> **Important:** "yes" means *wide enough in the lidar*, not *safe*. The NZDF account of the 2016 convoy describes boulders perched above the road; loose rock, cracked surfaces and damaged bridges still need checking on the ground. And "unknown" must never be treated as "yes".

### A reality check against 2016

In November 2016, **27 Army trucks drove the ~100 km inland route from Culverden to Kaikōura** ([NZDF](https://www.nzdf.mil.nz/army/what-we-do/humanitarian-assistance-and-disaster-relief/a-community-cut-off/)). If our method is sensible, it shouldn't mark the Inland Road as impassable. The lidar was flown a few weeks later, so this is a useful guide rather than proof.

### How much does the 30 cm assumption matter?

The **sensitivity table** reruns the passability check with different height steps (10, 20, 30 and 50 cm). If the answer barely changes, our conclusion doesn't hinge on that one assumption. In our run, 30 cm and 50 cm gave exactly the same result (6.8 km not passable for the MHOV), which gave us confidence in the 30 cm choice.

In [ ]:
# Plain whole-number OSM ids (the routing team's join key), and a yes/no/unknown per vehicle for every piece
pieces["osm_id"] = pd.to_numeric(pieces.get("osm_id"), errors="coerce").fillna(
    pd.to_numeric(pieces.parent_id.str.extract(r"(\d+)")[0], errors="coerce"))
pieces = pieces[pieces.osm_id.notna()].copy()
pieces["osm_id"] = pieces.osm_id.astype("int64")
pieces = pieces.rename(columns={"parent_id": "full_id"})
for v, d in VEHICLE_WIDTHS.items():
    pieces[f"passable_{v}"], pieces[f"reason_{v}"] = passability(
        pieces.drive_pinch_m, pieces.drive_share, pieces.is_bridge, pieces.is_tunnel, d["width_m"] + CLEARANCE_M)


def per_osm(p):
    # Combine pieces into one row per OSM road (worst-case answers, length-weighted shares)
    p = p.assign(_ws=p.share_measured * p.length_m, _wd=p.drive_share * p.length_m)
    g = p.groupby("osm_id")
    out = g.agg(full_id=("full_id", "first"), name=("name", "first"), highway=("highway", "first"),
                length_m=("length_m", "sum"), is_bridge=("is_bridge", "any"), is_tunnel=("is_tunnel", "any"),
                n_flagged=("n_flagged", "sum"), width_median_m=("width_median_m", "median"),
                drive_pinch_m=("drive_pinch_m", "min"), _ws=("_ws", "sum"), _wd=("_wd", "sum"))
    out["share_measured"], out["drive_share"] = out._ws / out.length_m, out._wd / out.length_m
    out = out.drop(columns=["_ws", "_wd"])
    for v in VEHICLE_WIDTHS:
        out[f"passable_{v}"] = g[f"passable_{v}"].agg(worst)
        out[f"reason_{v}"] = g.apply(lambda d, v=v: d.loc[d[f"passable_{v}"] == worst(d[f"passable_{v}"]), f"reason_{v}"].iloc[0])
    geom = g.geometry.apply(lambda s: unary_union(list(s)))
    return gpd.GeoDataFrame(out.reset_index(), geometry=geom.reindex(out.index).to_numpy(), crs=CRS)

by_osm = per_osm(pieces)
for v, d in VEHICLE_WIDTHS.items():
    km = by_osm.groupby(f"passable_{v}").length_m.sum() / 1000
    print(f"{d['label']} (needs {d['width_m'] + CLEARANCE_M:.2f} m): " +
          ", ".join(f"{k} {km.get(k, 0):.1f} km" for k in ("yes", "no", "unknown")))

# 2016 reality check: what do we say about the Inland Road?
inland = by_osm["name"].astype(str).str.contains("Inland", case=False)
print("\n2016 reality check - Inland Road (convoy route), answer for the MHOV:",
      by_osm.loc[inland, "passable_mhov"].value_counts().to_dict())

# Sensitivity: how the answer depends on the height-step assumption
need_mhov = VEHICLE_WIDTHS["mhov"]["width_m"] + CLEARANCE_M
sens = []
for dm in [0.10, 0.20, 0.30, 0.50]:
    r_ = measure_all({"slope_max": PASS_RULE["slope_max"], "dev_max": dm})
    ok_ = r_.status.isin(["ok", "narrow", "edge_not_found"]).to_numpy() & ~skip.to_numpy()
    pp_ = pinch_per_piece(r_.width_m.to_numpy(), ok_)
    ans, _ = passability(pieces.road_id.map(pp_.pinch), pieces.road_id.map(pp_.share).fillna(0),
                         pieces.is_bridge, pieces.is_tunnel, need_mhov)
    t = pieces.assign(a=ans)
    is_inland = t["name"].astype(str).str.contains("Inland", case=False)
    sens.append({"dev_max_m": dm, "MHOV no (km)": t.loc[t.a == "no", "length_m"].sum() / 1000,
                 "MHOV unknown (km)": t.loc[t.a == "unknown", "length_m"].sum() / 1000,
                 "Inland Road answer": worst(t.loc[is_inland, "a"]) if is_inland.any() else "n/a"})
print("\nSensitivity to the PASS_RULE height step (MHOV):")
print(pd.DataFrame(sens).round(1).to_string(index=False))

# Which roads are too narrow?
cols = ["osm_id", "name", "highway", "length_m", "width_median_m", "drive_pinch_m"] + [f"passable_{v}" for v in VEHICLE_WIDTHS]
nope = by_osm[(by_osm[[f"passable_{v}" for v in VEHICLE_WIDTHS]] == "no").any(axis=1)]
print("\nRoads NOT passable for at least one vehicle:")
print(nope.sort_values("drive_pinch_m")[cols].round(1).to_string(index=False) if len(nope) else "  none")

**Reading the results (from our run).** Three of the SH1 sections marked "no" (osm\_id 797156556, 453284424 and 32257464) are also on the team's U-Net list of road sections overlapping predicted landslides. Two completely different methods, lidar shape and a neural network on satellite images, point at the same stretches of road, which makes us more confident both are picking up real debris. **Kowhai Ford Road** is probably a false alarm: the road dips through a stream crossing (a ford), which looks like a narrow, non-flat road in the lidar. It's a good reminder that every "no" deserves a look before acting on it.

## Step 9 – Figures for the report

1. **Four example cross-sections.** The black line is the ground; the green band is the measured carriageway; the blue outline is the drivable width; the dotted line is where the map puts the road centre. These are the best way to *see* what the method is doing, and the best way to spot when it goes wrong.
2. **Passability maps** for each vehicle: green = yes, red = no, grey = unknown.

(The lidar-vs-NZTA scatter plot was drawn in Step 5.) All figures are saved as PNG files in `/content/xband_roads`: open the folder icon on the left of Colab to download them.

In [ ]:
def pick(mask, by=None, target=None):
    # Choose one example cross-section that matches a condition
    d = stations[mask]
    if not len(d):
        return None
    if by is None:
        return d.index[0]
    return (d[by] - target).abs().idxmin() if target is not None else d[by].idxmin()

u = stations[stations.usable]
examples = [("Typical road", pick(stations.usable, "width_m", u.width_m.median() if len(u) else 0)),
            ("Narrowest drivable", pick(stations.drive_usable & stations.drive_status.isin(["ok", "narrow"]), "drive_width_m")),
            ("Flat ground (edge not found)", pick(stations.drive_status.eq("edge_not_found") & ~skip)),
            ("Nothing flat at the centreline", pick(stations.status.eq("no_flat_centre")))]
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
for ax, (title, i) in zip(axes.ravel(), examples):
    if i is None:
        ax.set_axis_off(); ax.set_title(f"{title}: none found"); continue
    r = stations.loc[i]
    ax.plot(OFFSETS, Z_ALL[i], color="black", lw=1.2)                       # the ground
    if np.isfinite(r.left_m):
        ax.axvspan(r.left_m, r.right_m, color="tab:green", alpha=0.25, label=f"carriageway {r.width_m:.1f} m")
    dr = measure_width(Z_ALL[i], **PASS_RULE)
    if np.isfinite(dr["left_m"]):
        ax.axvspan(dr["left_m"], dr["right_m"], fill=False, edgecolor="tab:blue", lw=1.5,
                   label=f"drivable {dr['width_m']:.1f} m")
    ax.axvline(0, color="grey", ls=":", lw=1, label="mapped centreline")
    nm = r["name"] if isinstance(r["name"], str) else r.highway
    ax.set(title=f"{title}\n{nm} ({r.status})", xlabel="distance across road (m)", ylabel="height (m)")
    ax.title.set_fontsize(9); ax.legend(fontsize=7)
plt.tight_layout(); fig.savefig(ROOT / "fig_example_cross_sections.png", dpi=150); plt.show()

# Passability maps
colours = {"yes": "tab:green", "no": "tab:red", "unknown": "#bbbbbb"}
fig, axes = plt.subplots(1, len(VEHICLE_WIDTHS), figsize=(8 * len(VEHICLE_WIDTHS), 9))
for ax, (v, d) in zip(np.atleast_1d(axes), VEHICLE_WIDTHS.items()):
    for ans in ("unknown", "yes", "no"):
        sub = by_osm[by_osm[f"passable_{v}"] == ans]
        if len(sub):
            sub.plot(ax=ax, color=colours[ans], lw=3 if ans == "no" else 1.5, label=f"{ans} ({sub.length_m.sum()/1000:.1f} km)")
    ax.set(title=f"{d['label']}: needs {d['width_m'] + CLEARANCE_M:.2f} m", aspect="equal")
    ax.legend(loc="lower right")
    add_map_furniture(ax)
plt.tight_layout(); fig.savefig(ROOT / "fig_passability_map.png", dpi=150, bbox_inches="tight"); plt.show()

## Step 10 – Package everything for the routing team

This creates a `handoff` folder, zips it and downloads it automatically:

| File | What it is |
| --- | --- |
| `xband_road_widths.gpkg`, layer **`by_osm_id`** | **The one routing joins on**: one row per OpenStreetMap road (`osm_id` = their network's `osmid`) |
| same file, layer `road_pieces` | the same information per piece of road (more detail, for maps) |
| same file, layer `cross_sections` | every 50 m measurement, for checking in QGIS |
| `by_osm_id.csv` | the main table without map shapes (opens in Excel) |
| `README.md` | what each column means, in plain English |
| `handoff_metadata.json` | every setting, our accuracy results, data sources and limitations, so the work can be repeated |

**For Craig and Neeravh:** copy the `attach_road_widths` function (in the "heart of the method" cell) into the routing notebook, then:

```
by_osm = gpd.read_file("xband_road_widths.gpkg", layer="by_osm_id")
attach_road_widths(G_inc, by_osm)
for u, v, k, d in G_inc.edges(keys=True, data=True):
    if d.get("passable_mhov") == "no":
        d["p_block"] = 1.0      # closed to the MHOV, like their bridge weight-limit rule
```

In [ ]:
OUT = ROOT / "handoff"
if OUT.exists():
    shutil.rmtree(OUT)                     # start with an empty folder each time
OUT.mkdir()
gpkg = OUT / "xband_road_widths.gpkg"

# Round numbers to 2 decimal places for tidy tables
r2 = lambda df, cols: df.assign(**{c: df[c].round(2) for c in cols if c in df.columns})
num_cols = ["length_m", "width_median_m", "drive_pinch_m", "share_measured", "drive_share", "share_high"]
# Layer 1: one row per OSM road (what routing joins on)
r2(by_osm, num_cols).to_file(gpkg, layer="by_osm_id", driver="GPKG")
# Layer 2: one row per road piece
piece_cols = ["road_id", "osm_id", "full_id", "name", "highway", "is_bridge", "is_tunnel", "length_m", "n_stations",
              "n_usable", "n_flagged", "share_measured", "share_high", "width_median_m", "drive_share", "drive_pinch_m"] + \
             [f"{p}_{v}" for v in VEHICLE_WIDTHS for p in ("passable", "reason")] + ["geometry"]
r2(gpd.GeoDataFrame(pieces[piece_cols], geometry="geometry", crs=CRS), num_cols).to_file(gpkg, layer="road_pieces", driver="GPKG")
# Layer 3: every cross-section
st_cols = ["station_id", "road_id", "parent_id", "name", "highway", "chainage_m", "dem_src", "near_junction", "is_bridge",
           "is_tunnel", "status", "usable", "confidence", "width_m", "centre_offset_m", "roughness_m",
           "drive_status", "drive_width_m", "geometry"]
stations[st_cols].to_file(gpkg, layer="cross_sections", driver="GPKG")
r2(by_osm, num_cols).drop(columns="geometry").to_csv(OUT / "by_osm_id.csv", index=False)

# Metadata: everything someone would need to understand or repeat this work
meta = {
    "created": datetime.date.today().isoformat(), "authors": "X-Band road width (Hinami & Evan)",
    "join_key": "osm_id (integer OSM way id) = osmnx edge attribute 'osmid'", "crs": CRS, "aoi_wgs84": AOI_WGS84,
    "roads": {"repository": "https://github.com/Loading-my-brain/x-band", "commit": ROAD_COMMIT, "sha256": ROAD_SHA256,
              "filter": "drivable OSM highway classes (railway, cycleways, steps removed)", "attribution": "OpenStreetMap contributors"},
    "elevation": {str(k): {"label": l, "collection": u} for k, (l, u) in DEM_SOURCES.items()},
    "width_reference": {"name": "NZTA National Road Centreline - RCA data", "licence": "CC BY 4.0",
                        "url": "https://opendata-nzta.opendata.arcgis.com/datasets/NZTA::national-road-centreline-road-controlling-authority-data"},
    "carriageway_rule": BEST, "drivable_rule": PASS_RULE, "first_version_rule": STAGE2_RULE,
    "cross_sections": {"spacing_m": STATION_SPACING_M, "half_length_m": HALF_LEN_M, "sample_step_m": SAMPLE_STEP_M},
    "vehicles": {v: {**d, "clearance_m": CLEARANCE_M, "needs_m": d["width_m"] + CLEARANCE_M} for v, d in VEHICLE_WIDTHS.items()},
    "accuracy_width_vs_nzta": acc.round(3).to_dict("records"),
    "accuracy_position": pos.round(3).to_dict("records"),
    "sensitivity_pass_rule": pd.DataFrame(sens).round(2).to_dict("records"),
    "limitations": [
        "'yes' = wide enough in the Dec 2016 - Jan 2017 lidar, not confirmed safe (loose debris above the road, surface damage and bridge condition are not assessed).",
        "Bridges and tunnels are always 'unknown'.",
        "Carriageway accuracy was validated on ~40 NZTA stretches; widths are compressed toward the average.",
        "NZTA widths describe today's rebuilt roads.",
        "The drivable-width height step and the clearance margin are assumptions (see sensitivity table).",
        "Vehicle widths: Pinzgauer 1.8 m (NightHawk; Pinzgauer brochure), MHOV 2.6 m (Deagel; Wikipedia gives 2.5 m for the HX77). The clearance margin is an assumption.",
        "Roads outside the team road file or the AOI have no values: treat as unknown.",
    ],
}
(OUT / "handoff_metadata.json").write_text(json.dumps(meta, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
(OUT / "README.md").write_text(f'''# X-Band road widths - handoff to routing

Join layer `by_osm_id` to the routing graph on **osm_id = edge attribute osmid** (function `attach_road_widths` in the notebook).

| Column | Meaning |
|---|---|
| osm_id / full_id | OSM way id as a number / with the 'w' prefix |
| width_median_m | typical carriageway width (calibrated against NZTA) |
| drive_pinch_m | narrowest drivable width holding across two neighbouring cross-sections - use for "does it fit?" |
| share_measured / drive_share | fraction of the road's 50 m cross-sections with a usable carriageway / drivable width |
| n_flagged | cross-sections that looked narrow or had nothing flat at the centre (possible debris) |
| passable_<vehicle> | yes / no / unknown (vehicle width + {CLEARANCE_M} m clearance) |
| reason_<vehicle> | why it is no / unknown |

'unknown' must never be treated as 'yes'. Settings, accuracy, sources and limitations: handoff_metadata.json.
''')

# Zip the folder and download it
bundle = shutil.make_archive(str(ROOT / "xband_road_widths_handoff"), "zip", OUT)
print("Handoff files:", sorted(p.name for p in OUT.iterdir()))
try:
    from google.colab import files
    files.download(bundle)
except Exception:
    print("Download the zip from the Files panel:", bundle)

## Optional – check our widths against your own measurements

NZTA is one reference. A small set of widths you measure yourself is a second, completely independent one, and a nice addition to a report.

This cell writes `hand_check_sample.csv`: 40 random cross-sections spread across road types, each with a link to LINZ Basemaps aerial imagery. Open each link, measure the road width with the ruler tool, type it into the `width_hand_m` column, save the file as `hand_check_filled.csv` in `/content/xband_roads`, and run this cell again. It will then print how far our lidar widths are from yours.

*Keep in mind:* current imagery shows the rebuilt road, so compare against cross-sections that weren't damaged.

In [ ]:
filled, sample = ROOT / "hand_check_filled.csv", ROOT / "hand_check_sample.csv"
if filled.exists():
    # You've filled it in: compare
    hc = pd.read_csv(filled).dropna(subset=["width_hand_m"])
    e = hc.width_lidar_m - hc.width_hand_m
    print(f"Hand check: {len(e)} cross-sections | bias {e.mean():+.2f} m | MAE {e.abs().mean():.2f} m | RMSE {np.sqrt((e**2).mean()):.2f} m")
else:
    # Not yet: pick 40 random measured cross-sections, spread across road types
    pool = stations[stations.usable & (stations.status == "ok")]
    per = max(1, 40 // max(1, pool.highway.nunique()))
    picked = [i for _, d in pool.groupby("highway") for i in d.sample(min(len(d), per), random_state=SEED).index]
    smp = pool.loc[picked]
    if len(smp) < 40:
        smp = pd.concat([smp, pool.drop(smp.index).sample(min(40 - len(smp), len(pool) - len(smp)), random_state=SEED)])
    lon, lat = Transformer.from_crs(CRS, "EPSG:4326", always_xy=True).transform(smp["cx"].to_numpy(), smp["cy"].to_numpy())
    out = pd.DataFrame({"station_id": smp.station_id.to_numpy(), "road": smp["name"].to_numpy(), "highway": smp.highway.to_numpy(),
                        "lat": np.round(lat, 6), "lon": np.round(lon, 6), "width_lidar_m": smp.width_m.round(1).to_numpy(),
                        "width_hand_m": ""})
    out["linz_basemaps"] = [f"https://basemaps.linz.govt.nz/@{a},{b},z20" for a, b in zip(out.lat, out.lon)]
    out.to_csv(sample, index=False)
    print(f"Wrote {sample.name} ({len(out)} rows). Fill in width_hand_m and save as {filled.name}.")

## Looking back: what we learned, and what we'd do next

**What worked well**

* Lidar cross-sections measured road width to within about **1.7 m on average** (MAE) on stretches the method had never seen, and within **1.4 m on SH1**, the road that matters most for convoys.
* The team's road lines sit close to where NZTA says the roads are (median 1.25 m).
* Splitting the problem into *carriageway* width and *drivable* width fixed an obvious error (the Inland Road being marked impassable) and matched what really happened in 2016.
* Our narrowest SH1 sections overlap the team's U-Net landslide predictions: two independent methods agreeing.

**Honest limitations**

* Only ~40 NZTA stretches were available to validate against, and the calibrated widths are pulled toward the average.
* NZTA widths describe today's rebuilt roads, not the 2016 roads.
* Bridges and tunnels can't be measured with lidar ground models.
* The 2016–17 lidar was flown weeks after the quake, too late for the first 72 hours of a response.
* "Wide enough" isn't the same as "safe": rockfall risk, surface damage and bridge strength need checking on the ground.

**If we kept going**

* **Use faster 3D data.** Drone lidar or drone photogrammetry over specific roads could produce the same kind of cross-sections within hours of an event. The method doesn't care where the 3D ground model comes from.
* **Compare before and after.** With free pre-quake LINZ lidar as a baseline, subtracting "before" from "after" would show exactly where debris landed on the road.
* **Let satellites choose where to look.** The team's radar and U-Net results could tell drones which roads to fly first, with this notebook measuring whether trucks fit.
* **Shorter spacing** (e.g. 25 m) on flagged roads, so short pinches can't slip between cross-sections.

---
# Part 2 – Landslide risk from Sentinel-2 (Random Forest)

From `Kaikoura_UNet_Road_Integration`, with the U-Net removed. Output used by routing:
`/content/kaikoura_unet_roads/outputs/landslide_by_osm_id_rf.csv`.

# Kaikōura: Random Forest and GNS road comparison

Run in Colab to apply the saved Random Forest (150 trees, threshold 0.28) to the road-selected imagery and compare it with the GNS V3 mapped landslide reference. No new training or threshold tuning is performed.

The Random Forest road result is exported for routing first. The final comparison includes RF/GNS pixel metrics, road-overlap lengths and RF/GNS maps. GNS is a landslide inventory, **not confirmed road-blockage truth**. The road-selected tiles may overlap training/validation: these are integration reference scores, not a new independent test.

Use your own registered Earth Engine compute project. The reference comparison requires read access to the shared GNS assets. The saved RF is downloaded from GitHub; if that fails, ask ncha509 to send you `rf_150_trees.joblib.gz` and upload it to `/content/kaikoura_unet_roads/`.

Unobserved road sections remain **unknown**. A predicted intersection indicates a potential impact; the 10 m imagery does not establish full blockage, remaining vehicle clearance, or safe passage. The interactive Esri basemap is illustrative and is not verified 2016 imagery.

In [ ]:
from pathlib import Path
import json, hashlib, math, time
import ee

# EE_PROJECT: set in the Project inputs cell at the top
ROOT = Path('/content/kaikoura_unet_roads')
ROOT.mkdir(exist_ok=True)
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate(auth_mode='colab')
    ee.Initialize(project=EE_PROJECT)
assert ee.Number(1).getInfo() == 1
print('Earth Engine connected:', EE_PROJECT)

## Shared feature preprocessing

The U-Net checkpoint has been removed. This cell keeps the fixed, label-independent feature normalisation that the saved Random Forest was built with.

In [ ]:
# @title Shared feature normalisation (no edits needed)
import numpy as np

FEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]


def normalize_features(arr):
    """Same fixed, label-independent transform for both models."""
    x = np.nan_to_num(arr[:8].astype("float32"), nan=0, posinf=0, neginf=0)
    x[:6] = np.clip(x[:6], 0, 1)
    x[6] = np.clip(x[6], 0, 3000) / 3000.0
    x[7] = np.clip(x[7], 0, 90) / 90.0
    return x


print('Features:', FEATURE_NAMES)

In [ ]:
# @title Load road-analysis tools (no edits needed)
# Embedded reviewed integration helpers; no external repository code is executed.
import sys
MODULE_SOURCES = {'road_impact.py': '"""Component 2: road-centreline overlap, not confirmed blockage or passability."""\nfrom pathlib import Path\nimport argparse, json\nimport geopandas as gpd\nimport pandas as pd\nfrom shapely import make_valid, union_all\nfrom shapely.geometry import box, LineString\nfrom shapely.ops import linemerge\n\nAOI = (173.5, -42.5, 174.0, -42.0)\nCRS = \'EPSG:2193\'\nVEHICLE_ROADS = {\'motorway\',\'motorway_link\',\'trunk\',\'trunk_link\',\'primary\',\'primary_link\',\n \'secondary\',\'secondary_link\',\'tertiary\',\'tertiary_link\',\'unclassified\',\'residential\',\n \'living_street\',\'service\',\'track\',\'road\'}\n\ndef line_parts(geom):\n    if geom.is_empty: return []\n    if geom.geom_type == \'LineString\': return [geom] if geom.length > 0 else []\n    if hasattr(geom, \'geoms\'):\n        return [p for child in geom.geoms for p in line_parts(child)]\n    return []\n\ndef lengths(line, hazard):\n    pieces = line_parts(line.intersection(hazard))\n    if not pieces: return 0.0, 0.0, []\n    merged = union_all(pieces)\n    if merged.geom_type == \'MultiLineString\': merged = linemerge(merged)\n    runs = line_parts(merged)\n    return sum(p.length for p in runs), max(p.length for p in runs), runs\n\ndef read_hazards(path, roi):\n    data = gpd.read_file(path)\n    if data.crs is None: raise ValueError(f\'Missing CRS: {path}\')\n    data = data.to_crs(CRS)\n    data = data.loc[data.geometry.intersects(roi)].copy()\n    data.geometry = data.geometry.map(make_valid).intersection(roi)\n    return data\n\ndef run(roads_path, source_path, debris_path, out_dir):\n    out = Path(out_dir);out.mkdir(parents=True,exist_ok=True)\n    roi = gpd.GeoSeries([box(*AOI)],crs=4326).to_crs(CRS).iloc[0]\n    roads = gpd.read_file(roads_path)\n    if roads.crs is None: raise ValueError(\'Road input has no CRS\')\n    if not roads.geom_type.isin([\'LineString\',\'MultiLineString\']).all():\n        raise ValueError(\'Road input must contain lines: Kaikoura.shp is an administrative polygon. Use kai_route.gpkg.\')\n    if \'full_id\' not in roads or roads.full_id.isna().any() or roads.full_id.duplicated().any():\n        raise ValueError(\'Expected unique full_id from agreed team road file.\')\n    raw_count = len(roads)\n    # Exclude railways, buildings and pedestrian-only paths. Access restrictions retained as attributes.\n    roads = roads.loc[roads.highway.isin(VEHICLE_ROADS)].to_crs(CRS).copy()\n    roads = roads.loc[roads.geometry.intersects(roi)].copy()\n    roads.geometry = roads.geometry.intersection(roi)\n    rows=[]\n    for _,row in roads.iterrows():\n        for i,part in enumerate(line_parts(row.geometry)):\n            attrs={k:row.get(k) for k in [\'full_id\',\'osm_id\',\'name\',\'highway\',\'surface\',\'access\',\'motor_vehicle\',\'bridge\',\'tunnel\',\'width\']}\n            rows.append({**attrs,\'road_id\':f\'{row.full_id}:part{i}\', \'part_index\':i,\'geometry\':part})\n    if not rows: raise ValueError(\'No road lines in AOI\')\n    segments = gpd.GeoDataFrame(rows,crs=CRS)\n    segments[\'road_length_m\'] = segments.length\n    source = read_hazards(source_path,roi);debris = read_hazards(debris_path,roi)\n    source_union = union_all(source.geometry.values)\n    debris_union = union_all(debris.geometry.values)\n    hazard = union_all([source_union,debris_union])  # Prevent double counting.\n    source_index = source.sindex\n    debris_index = debris.sindex\n    results=[];affected_rows=[]\n    for row in segments.itertuples():\n        near = row.geometry.buffer(30)\n        local_source = union_all(source.geometry.iloc[source_index.query(near, predicate=\'intersects\')].values)\n        local_debris = union_all(debris.geometry.iloc[debris_index.query(near, predicate=\'intersects\')].values)\n        local_hazard = union_all([local_source,local_debris])\n        total, longest, pieces = lengths(row.geometry,local_hazard)\n        record={\'road_id\':row.road_id,\'affected_length_m\':total,\n            \'max_affected_run_m\':longest,\'affected_fraction\':total/row.road_length_m,\n            \'source_overlap_m\':lengths(row.geometry,local_source)[0],\n            \'debris_overlap_m\':lengths(row.geometry,local_debris)[0],\n            \'impact_status\':\'potentially_affected\' if total>0 else \'no_mapped_overlap\',\n            \'hazard_source\':\'GNS V3 mapped landslide inventory\'}\n        # Sensitivity to positional error, not measured road width or proven damage.\n        for distance in [10,30]:\n            record[f\'near_hazard_{distance}m_overlap_m\']=lengths(row.geometry,local_hazard.buffer(distance))[0]\n        results.append(record)\n        for i,piece in enumerate(pieces):\n            affected_rows.append({\'road_id\':row.road_id,\'run_id\':i,\'length_m\':piece.length,\'geometry\':piece})\n    result = segments.merge(pd.DataFrame(results),on=\'road_id\',validate=\'one_to_one\')\n    assert (result.affected_length_m <= result.road_length_m + 1e-6).all()\n    assert result.road_id.is_unique\n    result.drop(columns=\'geometry\').to_csv(out/\'road_impact.csv\',index=False)\n    result.to_file(out/\'road_impact.gpkg\',layer=\'road_segments\',driver=\'GPKG\')\n    result.to_crs(4326).to_file(out/\'road_impact.geojson\',driver=\'GeoJSON\')\n    if affected_rows:\n        gpd.GeoDataFrame(affected_rows,crs=CRS).to_file(out/\'affected_runs.gpkg\',driver=\'GPKG\')\n    import folium\n    m=folium.Map(location=[-42.25,173.75],zoom_start=10,tiles=\'OpenStreetMap\')\n    folium.GeoJson(gpd.GeoSeries([hazard],crs=CRS).to_crs(4326).__geo_interface__,\n        name=\'GNS mapped landslides\',style_function=lambda f:{\'color\':\'orange\',\'weight\':1,\'fillOpacity\':0.25}).add_to(m)\n    folium.GeoJson(result.to_crs(4326).to_json(),name=\'Road segments\',\n        style_function=lambda f:{\'color\':\'red\' if f[\'properties\'][\'affected_length_m\']>0 else \'blue\',\'weight\':3},\n        tooltip=folium.GeoJsonTooltip(fields=[\'road_id\',\'name\',\'affected_length_m\',\'max_affected_run_m\',\'impact_status\'])).add_to(m)\n    folium.LayerControl().add_to(m);m.save(out/\'road_impact_map.html\')\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    fig,ax=plt.subplots(figsize=(9,9))\n    gpd.GeoSeries([hazard],crs=CRS).plot(ax=ax,color=\'#e8c685\',alpha=0.6)\n    result.plot(ax=ax,color=\'#64748b\',linewidth=0.8)\n    if affected_rows: gpd.GeoDataFrame(affected_rows,crs=CRS).plot(ax=ax,color=\'red\',linewidth=2)\n    ax.set_title(\'Kaikōura: mapped landslides and potential road impacts\\nRed = centreline overlap; not confirmed road closure\')\n    ax.set_xlabel(\'NZTM easting (m)\');ax.set_ylabel(\'NZTM northing (m)\')\n    fig.tight_layout();fig.savefig(out/\'road_impact_map.png\',dpi=180);plt.close(fig)\n    summary={\'input_features\':raw_count,\'road_parts_in_aoi\':len(result),\n        \'potentially_affected_parts\':int((result.affected_length_m>0).sum()),\n        \'total_overlap_m\':float(result.affected_length_m.sum()),\n        \'max_continuous_overlap_m\':float(result.max_affected_run_m.max()),\n        \'aoi_wgs84\':AOI,\'metric_crs\':CRS,\'source_polygons_in_aoi\':len(source),\'debris_polygons_in_aoi\':len(debris),\n        \'limitations\':[\'OSM snapshot date unknown; may reflect post-2016 road realignment.\',\n        \'Road file is a subset, not verified complete regional routable graph.\',\n        \'No overlap is not evidence of safe passage. Overlap is not confirmed closure.\',\n        \'Lengths measured on road centrelines; no vehicle clearance or road width inference.\',\n        \'part IDs identify clipped line components, not routing graph edges split at junctions.\',\n        \'10/30 m buffers are positional sensitivity checks, not extra confirmed impacts.\']}\n    (out/\'summary.json\').write_text(json.dumps(summary,indent=2))\n    print(json.dumps(summary,indent=2));return result\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'--roads\',required=True);p.add_argument(\'--source\',required=True)\n    p.add_argument(\'--debris\',required=True);p.add_argument(\'--out\',default=\'outputs/component2\')\n    args=p.parse_args();run(args.roads,args.source,args.debris,args.out)\n', 'landslide_benchmark.py': 'import numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset\nFEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]\n\n\ndef normalize_features(arr):\n    """Same fixed, label-independent transform for both models."""\n    x = np.nan_to_num(arr[:8].astype("float32"), nan=0, posinf=0, neginf=0)\n    x[:6] = np.clip(x[:6], 0, 1)\n    x[6] = np.clip(x[6], 0, 3000) / 3000.0\n    x[7] = np.clip(x[7], 0, 90) / 90.0\n    return x\n\n\ndef read_tile(record):\n    with rasterio.open(record["path"]) as ds:\n        arr = ds.read()\n    return normalize_features(arr), (arr[8] > 0.5), (arr[9] > 0.5)\n\n\nclass TileDataset(Dataset):\n    def __init__(self, records, augment=False):\n        self.records, self.augment = records, augment\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, i):\n        x, y, valid = read_tile(self.records[i])\n        y, valid = y[None].astype("float32"), valid[None].astype("float32")\n        if self.augment:\n            if random.random() < 0.5:\n                x, y, valid = x[:, :, ::-1], y[:, :, ::-1], valid[:, :, ::-1]\n            if random.random() < 0.5:\n                x, y, valid = x[:, ::-1, :], y[:, ::-1, :], valid[:, ::-1, :]\n            k = random.randrange(4)\n            x, y, valid = [np.rot90(a, k, axes=(1, 2)) for a in (x, y, valid)]\n        return tuple(torch.from_numpy(a.copy()) for a in (x, y, valid))\n\n\nclass ConvBlock(nn.Module):\n    def __init__(self, in_channels, out_channels):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Conv2d(in_channels, out_channels, 3, padding=1),\n            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),\n            nn.Conv2d(out_channels, out_channels, 3, padding=1),\n            nn.GroupNorm(4, out_channels), nn.ReLU(inplace=True),\n        )\n\n    def forward(self, x):\n        return self.net(x)\n\n\nclass PostEventUNet(nn.Module):\n    def __init__(self, in_channels=8, base=8):\n        super().__init__()\n        self.enc = nn.ModuleList([\n            ConvBlock(in_channels, base), ConvBlock(base, base * 2),\n            ConvBlock(base * 2, base * 4), ConvBlock(base * 4, base * 8),\n        ])\n        self.pool = nn.MaxPool2d(2)\n        self.dec = nn.ModuleList([\n            ConvBlock(base * 8 + base * 4, base * 4),\n            ConvBlock(base * 4 + base * 2, base * 2),\n            ConvBlock(base * 2 + base, base),\n        ])\n        self.head = nn.Conv2d(base, 1, 1)\n\n    def forward(self, x):\n        skips = []\n        for i, block in enumerate(self.enc):\n            x = block(self.pool(x) if i else x)\n            skips.append(x)\n        for block, skip in zip(self.dec, reversed(skips[:-1])):\n            x = nn.functional.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)\n            x = block(torch.cat([x, skip], dim=1))\n        return self.head(x)\n\n', 'unet_road_integration.py': '"""Inference-only U-Net / road integration with explicit observation coverage.\n\nNo training is performed. Neural imports and forward passes occur only when\ninfer_road_tiles is explicitly called in Colab. Geometry utilities are independent.\n"""\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport hashlib\nimport json\nimport math\nimport time\n\nimport geopandas as gpd\nimport numpy as np\nimport pandas as pd\nimport rasterio\nfrom rasterio.transform import Affine\nfrom rasterio.features import shapes\nfrom rasterio.windows import Window\nimport requests\nfrom shapely import make_valid, union_all\nfrom shapely.geometry import box, shape, GeometryCollection\n\ntry:\n    from .road_impact import VEHICLE_ROADS, line_parts, lengths\nexcept ImportError:\n    from road_impact import VEHICLE_ROADS, line_parts, lengths\n\nAOI_WGS84 = (173.5, -42.5, 174.0, -42.0)\nMETRIC_CRS = "EPSG:2193"\nGRID_XMIN, GRID_YTOP = 1641080, 5350110\nSCALE, TILE_SIZE, BLOCK_TILES = 10, 128, 4\nBLOCK_SIZE = TILE_SIZE * BLOCK_TILES\nFEATURE_NAMES = ["B2", "B3", "B4", "B8", "B11", "B12", "elevation", "slope"]\nDEFAULT_THRESHOLD = 0.31\n\n\ndef aoi_metric():\n    return gpd.GeoSeries([box(*AOI_WGS84)], crs=4326).to_crs(METRIC_CRS).iloc[0]\n\n\ndef polygon_parts(geometry):\n    if geometry is None or geometry.is_empty:\n        return []\n    if geometry.geom_type == "Polygon":\n        return [geometry] if geometry.area > 0 else []\n    if hasattr(geometry, "geoms"):\n        return [part for child in geometry.geoms for part in polygon_parts(child)]\n    return []\n\n\ndef empty_geodataframe(columns, crs=METRIC_CRS):\n    data = {column: pd.Series(dtype="object") for column in columns if column != "geometry"}\n    return gpd.GeoDataFrame(data, geometry=gpd.GeoSeries([], crs=crs), crs=crs)\n\n\ndef _json_atomic(path, value):\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(json.dumps(value, indent=2))\n    temporary.replace(path)\n\n\ndef _file_hash(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef read_road_segments(road_path):\n    """Retain road attributes and IDs using the agreed component-2 ID convention."""\n    roads = gpd.read_file(road_path)\n    if roads.crs is None:\n        raise ValueError("Road input has no CRS.")\n    if not roads.geom_type.isin(["LineString", "MultiLineString"]).all():\n        raise ValueError("Use the kai_route.gpkg line layer; Kaikoura.shp is an administrative polygon.")\n    if "full_id" not in roads or roads.full_id.isna().any() or roads.full_id.duplicated().any():\n        raise ValueError("Road file needs unique non-null full_id values.")\n    if "highway" not in roads:\n        raise ValueError("Road file has no highway classification.")\n    raw_count = len(roads)\n    roads = roads.loc[roads.highway.isin(VEHICLE_ROADS)].to_crs(METRIC_CRS).copy()\n    roi = aoi_metric()\n    roads = roads.loc[roads.geometry.intersects(roi)].copy()\n    roads.geometry = roads.geometry.map(make_valid).intersection(roi)\n    rows = []\n    geometry_column = roads.geometry.name\n    for _, record in roads.iterrows():\n        attributes = {key: value for key, value in record.items() if key != geometry_column}\n        for index, part in enumerate(line_parts(record.geometry)):\n            rows.append(dict(attributes, road_id=f"{record.full_id}:part{index}",\n                             part_index=index, geometry=part))\n    if not rows:\n        raise ValueError("No vehicle-road line components intersect the agreed AOI.")\n    result = gpd.GeoDataFrame(rows, crs=METRIC_CRS)\n    result["road_length_m"] = result.length\n    if not result.road_id.is_unique:\n        raise ValueError("Exploded road IDs are not unique.")\n    result.attrs["input_feature_count"] = raw_count\n    result.attrs["road_file"] = str(road_path)\n    return result\n\n\ndef plan_road_tiles(segments):\n    """Select original 128-pixel tiles solely by clipped road geometry.\n\n    Include every intersected original tile, including mostly ocean/invalid tiles.\n    Group downloads in 4×4 blocks while retaining 128×128 inference windows.\n    """\n    if segments.crs is None:\n        raise ValueError("Road segments need a CRS.")\n    segments = segments.to_crs(METRIC_CRS)\n    roi = aoi_metric()\n    minx, miny, maxx, maxy = roi.bounds\n    nx = math.ceil((maxx - GRID_XMIN) / (TILE_SIZE * SCALE))\n    ny = math.ceil((GRID_YTOP - miny) / (TILE_SIZE * SCALE))\n    roads_union = union_all(segments.geometry.values).intersection(roi)\n    if roads_union.is_empty:\n        raise ValueError("No roads in AOI for inference planning.")\n    width = TILE_SIZE * SCALE\n    road_minx, road_miny, road_maxx, road_maxy = roads_union.bounds\n    cmin = max(0, math.floor((road_minx - GRID_XMIN) / width) - 1)\n    cmax = min(nx - 1, math.floor((road_maxx - GRID_XMIN) / width))\n    rmin = max(0, math.floor((GRID_YTOP - road_maxy) / width) - 1)\n    rmax = min(ny - 1, math.floor((GRID_YTOP - road_miny) / width))\n    tiles, blocks = [], {}\n    for row in range(rmin, rmax + 1):\n        for col in range(cmin, cmax + 1):\n            x, y = GRID_XMIN + col * width, GRID_YTOP - row * width\n            extent = box(x, y - width, x + width, y)\n            if not extent.intersects(roads_union):\n                continue\n            block_row, block_col = row // BLOCK_TILES, col // BLOCK_TILES\n            block_id = f"br{block_row:03d}_bc{block_col:03d}"\n            tile = {"tile": f"r{row:03d}_c{col:03d}", "row": row, "col": col,\n                    "block_id": block_id, "local_row": row % BLOCK_TILES,\n                    "local_col": col % BLOCK_TILES,\n                    "transform": [SCALE, 0, x, 0, -SCALE, y]}\n            tiles.append(tile)\n            if block_id not in blocks:\n                bx = GRID_XMIN + block_col * BLOCK_SIZE * SCALE\n                by = GRID_YTOP - block_row * BLOCK_SIZE * SCALE\n                blocks[block_id] = {"block_id": block_id, "block_row": block_row,\n                    "block_col": block_col, "transform": [SCALE, 0, bx, 0, -SCALE, by], "tiles": []}\n            blocks[block_id]["tiles"].append(tile["tile"])\n    return {"version": 1, "aoi_wgs84": list(AOI_WGS84), "crs": METRIC_CRS,\n        "grid_origin": [GRID_XMIN, GRID_YTOP], "scale": SCALE, "tile_size": TILE_SIZE,\n        "block_size": BLOCK_SIZE, "grid_columns": nx, "grid_rows": ny,\n        "selection": "all original-grid tiles intersecting agreed road centrelines; no label or valid-fraction selection",\n        "tiles": tiles, "blocks": sorted(blocks.values(), key=lambda item: (item["block_row"], item["block_col"]))}\n\n\ndef download_inference_blocks(stack, plan, cache_dir, *, max_workers=4, timeout=180,\n                               retries=3, data_signature=None):\n    """Download nine-band feature+valid stacks; fail explicitly on any failed block."""\n    if not 1 <= max_workers <= 4:\n        raise ValueError("Use one to four concurrent Earth Engine downloads.")\n    cache_dir = Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    serialized = stack.serialize()\n    identity = {"stack_sha256": hashlib.sha256(serialized.encode()).hexdigest(),\n        "data_signature": data_signature, "features": FEATURE_NAMES, "extra_band": "valid",\n        "crs": METRIC_CRS, "origin": [GRID_XMIN, GRID_YTOP],\n        "scale": SCALE, "dimensions": [BLOCK_SIZE, BLOCK_SIZE]}\n    cache_key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()[:16]\n    directory = cache_dir / cache_key\n    directory.mkdir(exist_ok=True)\n    _json_atomic(directory / "cache_config.json", identity)\n    _json_atomic(directory / "inference_plan.json", plan)\n\n    def fetch(block):\n        path = directory / (block["block_id"] + ".tif")\n        meta_path = path.with_suffix(".json")\n        if path.exists() and meta_path.exists():\n            meta = json.loads(meta_path.read_text())\n            if meta.get("sha256") != _file_hash(path):\n                raise ValueError(f"Cached block hash mismatch: {path.name}")\n        else:\n            error = None\n            for attempt in range(retries):\n                try:\n                    url = stack.getDownloadURL({"crs": METRIC_CRS,\n                        "crs_transform": block["transform"],\n                        "dimensions": [BLOCK_SIZE, BLOCK_SIZE],\n                        "format": "GEO_TIFF", "filePerBand": False})\n                    response = requests.get(url, timeout=timeout)\n                    response.raise_for_status()\n                    temporary = path.with_suffix(".tif.tmp")\n                    temporary.write_bytes(response.content)\n                    with rasterio.open(temporary) as dataset:\n                        if dataset.count != 9 or dataset.width != BLOCK_SIZE or dataset.height != BLOCK_SIZE:\n                            raise ValueError("Expected a nine-band 512×512 input block.")\n                        if dataset.crs.to_epsg() != 2193 or not dataset.transform.almost_equals(Affine(*block["transform"])):\n                            raise ValueError("Downloaded block CRS/grid mismatch.")\n                    temporary.replace(path)\n                    meta = {**block, "sha256": _file_hash(path), "path": str(path), "cache_key": cache_key}\n                    _json_atomic(meta_path, meta)\n                    error = None\n                    break\n                except Exception as exception:\n                    error = exception\n                    if attempt + 1 < retries:\n                        time.sleep(2 * (attempt + 1))\n            if error is not None:\n                raise RuntimeError(f"{block[\'block_id\']} failed after {retries} attempts: {error}") from error\n        with rasterio.open(path) as dataset:\n            if dataset.count != 9 or dataset.width != BLOCK_SIZE or dataset.height != BLOCK_SIZE:\n                raise ValueError(f"Cached block shape mismatch: {path.name}")\n            if dataset.crs.to_epsg() != 2193 or not dataset.transform.almost_equals(Affine(*block["transform"])):\n                raise ValueError(f"Cached block CRS/grid mismatch: {path.name}")\n        return {**block, "path": str(path), "sha256": _file_hash(path), "cache_key": cache_key}\n\n    records, errors = [], []\n    with ThreadPoolExecutor(max_workers=max_workers) as executor:\n        futures = {executor.submit(fetch, block): block for block in plan["blocks"]}\n        for future in as_completed(futures):\n            block = futures[future]\n            try:\n                records.append(future.result())\n                print(f"Input block ready {len(records)}/{len(plan[\'blocks\'])}: {block[\'block_id\']}", flush=True)\n            except Exception as exception:\n                errors.append({"block_id": block["block_id"], "error": str(exception)})\n    _json_atomic(directory / "download_errors.json", errors)\n    if errors:\n        raise RuntimeError(f"{len(errors)} inference blocks failed; see {directory/\'download_errors.json\'}. No unaffected-road result was produced.")\n    records.sort(key=lambda item: (item["block_row"], item["block_col"]))\n    _json_atomic(directory / "downloaded_blocks.json", records)\n    return records\n\n\ndef infer_road_tiles(model, plan, block_records, out_dir, *, threshold=DEFAULT_THRESHOLD,\n                     device=None, batch_size=16, zero_invalid_features=False,\n                     checkpoint_metadata=None):\n    """Run the frozen model only on selected original 128×128 windows in Colab.\n\n    Probability, binary mask, and polygon outputs exclude invalid pixels. Default\n    input handling reproduces the existing benchmark normalize_features exactly;\n    zero_invalid_features can be enabled only if used by checkpoint training.\n    """\n    import torch\n    try:\n        from .landslide_benchmark import normalize_features\n    except ImportError:\n        from landslide_benchmark import normalize_features\n    if not 1 <= batch_size <= 16 or not 0 < threshold < 1:\n        raise ValueError("Use batch_size 1–16 and a probability threshold between zero and one.")\n    out_dir = Path(out_dir)\n    tile_dir = out_dir / "inference_tiles"\n    tile_dir.mkdir(parents=True, exist_ok=True)\n    blocks = {item["block_id"]: item for item in block_records}\n    missing = sorted({item["block_id"] for item in plan["tiles"]} - blocks.keys())\n    if missing:\n        raise RuntimeError(f"Missing input blocks {missing}; inference cannot mark roads unaffected.")\n    device = torch.device(device) if device is not None else next(model.parameters()).device\n    model = model.to(device)\n    model.eval()\n    roi = aoi_metric()\n    hazard_rows, coverage_rows, tile_statistics = [], [], []\n    tiles = plan["tiles"]\n    with torch.inference_mode():\n        for start in range(0, len(tiles), batch_size):\n            batch = tiles[start:start + batch_size]\n            features, raw_masks = [], []\n            for tile in batch:\n                window = Window(tile["local_col"] * TILE_SIZE, tile["local_row"] * TILE_SIZE,\n                                TILE_SIZE, TILE_SIZE)\n                with rasterio.open(blocks[tile["block_id"]]["path"]) as dataset:\n                    array = dataset.read(window=window)\n                    transform = dataset.window_transform(window)\n                if array.shape != (9, TILE_SIZE, TILE_SIZE) or not transform.almost_equals(Affine(*tile["transform"])):\n                    raise ValueError(f"Original tile window/grid mismatch: {tile[\'tile\']}")\n                valid = (array[8] > 0.5) & np.isfinite(array[:8]).all(axis=0)\n                x = normalize_features(array[:8])\n                if zero_invalid_features:\n                    x[:, ~valid] = 0\n                features.append(x)\n                raw_masks.append(valid)\n            inputs = torch.from_numpy(np.stack(features)).to(device)\n            probabilities = torch.sigmoid(model(inputs))[:, 0].cpu().numpy()\n            if probabilities.shape != (len(batch), TILE_SIZE, TILE_SIZE) or not np.isfinite(probabilities).all():\n                raise RuntimeError("Invalid neural probability output; no road-impact result produced.")\n            for tile, probability, valid in zip(batch, probabilities, raw_masks):\n                transform = Affine(*tile["transform"])\n                mask = (probability >= threshold).astype("uint8")\n                profile = {"driver": "GTiff", "width": TILE_SIZE, "height": TILE_SIZE,\n                    "count": 1, "crs": METRIC_CRS, "transform": transform,\n                    "compress": "deflate", "dtype": "float32", "nodata": -9999}\n                with rasterio.open(tile_dir / (tile["tile"] + "_probability.tif"), "w", **profile) as dataset:\n                    dataset.write(np.where(valid, probability, -9999).astype("float32"), 1)\n                profile.update(dtype="uint8", nodata=255)\n                with rasterio.open(tile_dir / (tile["tile"] + "_mask.tif"), "w", **profile) as dataset:\n                    dataset.write(np.where(valid, mask, 255).astype("uint8"), 1)\n                for geometry, value in shapes(valid.astype("uint8"), mask=valid, transform=transform):\n                    if value == 1:\n                        for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                            coverage_rows.append({"tile": tile["tile"], "geometry": part})\n                for geometry, value in shapes(mask, mask=valid & (mask == 1), transform=transform):\n                    if value == 1:\n                        for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                            hazard_rows.append({"tile": tile["tile"], "geometry": part})\n                tile_statistics.append({"tile": tile["tile"], "valid_pixels": int(valid.sum()),\n                    "positive_pixels": int((valid & (mask == 1)).sum()),\n                    "valid_fraction": float(valid.mean())})\n            print(f"Frozen U-Net inference: {min(start+batch_size,len(tiles))}/{len(tiles)} original tiles", flush=True)\n    hazards = gpd.GeoDataFrame(hazard_rows, crs=METRIC_CRS) if hazard_rows else empty_geodataframe(["tile", "geometry"])\n    coverage = gpd.GeoDataFrame(coverage_rows, crs=METRIC_CRS) if coverage_rows else empty_geodataframe(["tile", "geometry"])\n    _json_atomic(out_dir / "inference_manifest.json", tile_statistics)\n    _json_atomic(out_dir / "inference_config.json", {"threshold": float(threshold),\n        "feature_order": FEATURE_NAMES, "tile_size": TILE_SIZE, "scale": SCALE,\n        "origin": [GRID_XMIN, GRID_YTOP], "crs": METRIC_CRS,\n        "zero_invalid_features": bool(zero_invalid_features), "batch_size": batch_size,\n        "checkpoint_metadata": checkpoint_metadata,\n        "selection": plan["selection"], "training": "none; frozen inference only"})\n    return {"hazards": hazards, "coverage": coverage, "tiles": tile_statistics,\n            "threshold": float(threshold), "tile_directory": str(tile_dir)}\n\n\ndef _polygon_layer(layer, roi):\n    if layer.crs is None:\n        raise ValueError("Hazard/coverage polygon input needs an explicit CRS.")\n    layer = layer.to_crs(METRIC_CRS)\n    rows = []\n    for geometry in layer.geometry:\n        if geometry is None or geometry.is_empty:\n            continue\n        for part in polygon_parts(make_valid(geometry).intersection(roi)):\n            rows.append({"geometry": part})\n    return gpd.GeoDataFrame(rows, crs=METRIC_CRS) if rows else empty_geodataframe(["geometry"])\n\n\ndef assess_road_impacts(segments, predicted_hazards, observed_coverage):\n    """Generic polygon-to-road adapter retaining unknown road length explicitly."""\n    if segments.crs is None or not segments.road_id.is_unique:\n        raise ValueError("Road components need a CRS and unique road_id values.")\n    result = segments.to_crs(METRIC_CRS).copy()\n    roi = aoi_metric()\n    result = result.loc[result.geometry.intersects(roi)].copy()\n    result.geometry = result.geometry.map(make_valid).intersection(roi)\n    result = result.loc[result.length > 0].copy()\n    if result.empty:\n        raise ValueError("No nonzero road line components remain inside the agreed AOI.")\n    hazard_layer = _polygon_layer(predicted_hazards, roi)\n    coverage_layer = _polygon_layer(observed_coverage, roi)\n    coverage_union = union_all(coverage_layer.geometry.values)\n    # Restrict hazards to observed valid coverage even for externally supplied polygons.\n    hazard_union = union_all(hazard_layer.geometry.values).intersection(coverage_union)\n    impact_rows, affected_rows, unknown_rows, observed_rows = [], [], [], []\n    tolerance = 1e-6\n    for record in result.itertuples():\n        line = record.geometry.intersection(roi)\n        road_length = float(line.length)\n        observed_length, _, observed = lengths(line, coverage_union)\n        affected_length, longest, affected = lengths(line, hazard_union)\n        unknown = line_parts(line.difference(coverage_union))\n        unknown_length = sum(piece.length for piece in unknown)\n        if not affected_length <= observed_length + tolerance or not observed_length <= road_length + tolerance:\n            raise AssertionError(f"Length hierarchy failed for {record.road_id}.")\n        if abs(observed_length + unknown_length - road_length) > max(tolerance, road_length * 1e-9):\n            raise AssertionError(f"Observed/unknown lengths do not partition road {record.road_id}.")\n        if affected_length > tolerance:\n            status = "potentially_affected"\n        elif observed_length <= tolerance:\n            status = "not_assessed"\n        elif unknown_length > tolerance:\n            status = "partially_observed"\n        else:\n            status = "no_predicted_overlap"\n        impact_rows.append({"road_id": record.road_id, "road_length_m": road_length,\n            "observed_length_m": float(observed_length), "unknown_length_m": float(unknown_length),\n            "coverage_fraction": float(observed_length / road_length) if road_length else 0.0,\n            "affected_length_m": float(affected_length), "max_affected_run_m": float(longest),\n            "affected_fraction": float(affected_length / road_length) if road_length else 0.0,\n            "impact_status": status, "has_unknown_length": bool(unknown_length > tolerance),\n            "hazard_source": "frozen U-Net predicted mapped-landslide extent"})\n        for index, piece in enumerate(affected):\n            affected_rows.append({"road_id": record.road_id, "run_id": index,\n                                  "length_m": float(piece.length), "geometry": piece})\n        for index, piece in enumerate(unknown):\n            unknown_rows.append({"road_id": record.road_id, "run_id": index,\n                                 "length_m": float(piece.length), "geometry": piece})\n        for index, piece in enumerate(observed):\n            observed_rows.append({"road_id": record.road_id, "run_id": index,\n                                  "length_m": float(piece.length), "geometry": piece})\n    result = result.drop(columns=["road_length_m"], errors="ignore").merge(\n        pd.DataFrame(impact_rows), on="road_id", validate="one_to_one")\n    def runs(rows):\n        return gpd.GeoDataFrame(rows, crs=METRIC_CRS) if rows else empty_geodataframe(["road_id", "run_id", "length_m", "geometry"])\n    clean_hazards = gpd.GeoDataFrame({"geometry": polygon_parts(hazard_union)}, crs=METRIC_CRS)\n    clean_coverage = gpd.GeoDataFrame({"geometry": polygon_parts(coverage_union)}, crs=METRIC_CRS)\n    return {"roads": result, "affected_runs": runs(affected_rows), "unknown_runs": runs(unknown_rows),\n            "observed_runs": runs(observed_rows), "hazards": clean_hazards, "coverage": clean_coverage}\n\n\ndef _write_geojson(layer, path):\n    if layer.empty:\n        path.write_text(json.dumps({"type": "FeatureCollection", "features": []}))\n    else:\n        layer.to_crs(4326).to_file(path, driver="GeoJSON")\n\n\ndef export_integration_outputs(assessment, out_dir, *, inference_result=None,\n                                plan=None, checkpoint_metadata=None):\n    """Export complete road/coverage results and an illustrative Esri map."""\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    roads = assessment["roads"]\n    roads.drop(columns="geometry").to_csv(out_dir / "road_impact.csv", index=False)\n    roads.to_file(out_dir / "road_impact.gpkg", layer="road_segments", driver="GPKG")\n    _write_geojson(roads, out_dir / "road_impact.geojson")\n    for key in ("affected_runs", "unknown_runs", "observed_runs"):\n        layer = assessment[key]\n        _write_geojson(layer, out_dir / (key + ".geojson"))\n        # GeoPackage supports empty layers using modern GeoPandas/pyogrio.\n        layer.to_file(out_dir / (key + ".gpkg"), layer=key, driver="GPKG")\n    _write_geojson(assessment["hazards"], out_dir / "unet_landslides.geojson")\n    _write_geojson(assessment["coverage"], out_dir / "inference_coverage.geojson")\n    status_counts = roads.impact_status.value_counts().to_dict()\n    summary = {"road_parts": len(roads), "status_counts": status_counts,\n        "total_road_length_m": float(roads.road_length_m.sum()),\n        "total_observed_length_m": float(roads.observed_length_m.sum()),\n        "total_unknown_length_m": float(roads.unknown_length_m.sum()),\n        "total_affected_length_m": float(roads.affected_length_m.sum()),\n        "max_continuous_affected_run_m": float(roads.max_affected_run_m.max()) if len(roads) else 0.0,\n        "aoi_wgs84": list(AOI_WGS84), "metric_crs": METRIC_CRS,\n        "threshold": inference_result["threshold"] if inference_result else None,\n        "selected_tile_count": len(plan["tiles"]) if plan else None,\n        "download_block_count": len(plan["blocks"]) if plan else None,\n        "checkpoint_metadata": checkpoint_metadata, "training": "none; frozen model inference",\n        "limitations": ["Potential hazard overlap is not confirmed blockage or passability.",\n            "No predicted overlap never establishes an open or safe road.",\n            "Road sections without valid observations remain explicitly unknown.",\n            "OSM road snapshot date is unverified; road file is a subset, not a complete routable graph.",\n            "Esri imagery is an illustrative current mosaic with unknown acquisition dates, not verified 2016 disaster imagery.",\n            "Road IDs identify clipped line components, not junction-split routing graph edges.",\n            "No new segmentation accuracy is claimed for road-selected inference tiles."]}\n    _json_atomic(out_dir / "summary.json", summary)\n    if plan is not None:\n        _json_atomic(out_dir / "inference_plan.json", plan)\n    import folium\n    map_view = folium.Map(location=[-42.25, 173.75], zoom_start=10, tiles=None)\n    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",\n        attr="Esri, Maxar, Earthstar Geographics and the GIS User Community",\n        name="Esri imagery — illustrative; acquisition dates unverified", overlay=False, control=True).add_to(map_view)\n    layers = [(assessment["hazards"], "U-Net predicted landslides", "#f0a020", 1, 0.20, True),\n        (assessment["coverage"], "Valid inference coverage", "#4c9d69", 1, 0.10, False),\n        (roads, "Agreed road components", "#00c9ee", 3, 0, True),\n        (assessment["unknown_runs"], "Unknown / unobserved road sections", "#808080", 5, 0, True),\n        (assessment["affected_runs"], "Potentially affected observed runs", "#ff3b30", 5, 0, True)]\n    tooltip = folium.GeoJsonTooltip(fields=["road_id", "impact_status", "road_length_m", "observed_length_m",\n        "unknown_length_m", "affected_length_m", "max_affected_run_m"])\n    for layer, name, color, weight, opacity, show in layers:\n        if layer.empty:\n            continue\n        style = {"color": color, "weight": weight, "fillOpacity": opacity}\n        folium.GeoJson(layer.to_crs(4326).to_json(), name=name, show=show,\n            style_function=lambda feature, style=style: style,\n            tooltip=tooltip if name == "Agreed road components" else None).add_to(map_view)\n    bounds = roads.to_crs(4326).total_bounds\n    map_view.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])\n    folium.LayerControl(collapsed=False).add_to(map_view)\n    notice = "<div style=\'position:fixed;bottom:25px;left:15px;z-index:9999;background:white;padding:8px;font-size:12px\'>" \\\n        "Amber: prediction · Cyan: roads · Red: potential overlap · Gray: unknown<br>" \\\n        "Illustrative imagery; not verified 2016 imagery. No overlap ≠ safe passage.</div>"\n    map_view.get_root().html.add_child(folium.Element(notice))\n    map_view.save(out_dir / "road_impact_map.html")\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots(figsize=(9, 9))\n    if not assessment["hazards"].empty:\n        assessment["hazards"].plot(ax=ax, color="#f0a020", alpha=0.4)\n    roads.plot(ax=ax, color="#00a5ce", linewidth=1)\n    if not assessment["unknown_runs"].empty:\n        assessment["unknown_runs"].plot(ax=ax, color="#808080", linewidth=2)\n    if not assessment["affected_runs"].empty:\n        assessment["affected_runs"].plot(ax=ax, color="#ff3b30", linewidth=2)\n    ax.set_title("Frozen U-Net predictions and potential road impacts\\nGray = unobserved; red = predicted overlap; no confirmed closure")\n    ax.set_xlabel("NZTM easting (m)")\n    ax.set_ylabel("NZTM northing (m)")\n    fig.tight_layout()\n    fig.savefig(out_dir / "road_impact_map.png", dpi=180)\n    plt.close(fig)\n    print(json.dumps(summary, indent=2), flush=True)\n    return {"summary": summary, "map": map_view, "output_directory": str(out_dir)}\n'}
module_dir = ROOT/'modules'
module_dir.mkdir(exist_ok=True)
for name, source_text in MODULE_SOURCES.items():
    (module_dir/name).write_text(source_text)
sys.path.insert(0, str(module_dir))
from unet_road_integration import (read_road_segments, plan_road_tiles,
    download_inference_blocks, infer_road_tiles, assess_road_impacts, export_integration_outputs)
print('Road-integration helpers ready.')

In [ ]:
# Identical inputs and preprocessing to the existing saved model.
from datetime import datetime, timezone

AOI = [173.5, -42.5, 174.0, -42.0]
POST_START, POST_END = '2016-11-14', '2017-02-01'
BANDS = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
CRS, GRID = 'EPSG:2193', [10, 0, 1641080, 0, -10, 5350110]
roi = ee.Geometry.Rectangle(AOI, geodesic=False)

def cloud_mask(image):
    qa = image.select('QA60')
    clear = qa.bitwiseAnd(1 << 10).eq(0).And(qa.bitwiseAnd(1 << 11).eq(0))
    return image.select(BANDS).multiply(0.0001).updateMask(clear)

collection = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED').filterBounds(roi)
              .filterDate(POST_START, POST_END)
              .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 50)))
scene_count = collection.size().getInfo()
if scene_count == 0:
    raise RuntimeError('No imagery in the original post-event window.')
optical_projection = ee.Image(collection.first()).select('B2').projection()
post = (collection.map(cloud_mask).median().setDefaultProjection(optical_projection)
        .resample('bilinear').clip(roi))
dem_native = ee.Image('USGS/SRTMGL1_003').select('elevation')
slope_native = ee.Terrain.slope(dem_native)
terrain = (dem_native.resample('bilinear').rename('elevation')
           .addBands(slope_native.resample('bilinear').rename('slope')))
features = post.addBands(terrain).select(FEATURE_NAMES).clip(roi)
# Keep the same land mask as checkpoint training for reproducibility.
land = ee.Image('UMD/hansen/global_forest_change_2022_v1_10').select('datamask').eq(1)
valid = (features.mask().reduce(ee.Reducer.min()).And(land)
         .clip(roi).unmask(0, False).rename('valid'))
# Exactly nine bands: eight predictors + availability, without GNS labels.
stack = features.addBands(valid).unmask(0, False).toFloat()
assert stack.bandNames().getInfo() == FEATURE_NAMES + ['valid']
imagery_config = {
    'collection': 'COPERNICUS/S2_HARMONIZED',
    'post_start': POST_START, 'post_end_exclusive': POST_END,
    'scene_count': scene_count, 'scene_ids': collection.aggregate_array('system:index').getInfo(),
    'bands': FEATURE_NAMES, 'crs': CRS, 'grid': GRID,
    'aoi': AOI, 'cloud_mask': 'QA60 cloud/cirrus, scene cloud percentage <50, median',
    'terrain': 'USGS/SRTMGL1_003 elevation and slope, bilinear',
    'land_mask': 'UMD/hansen/global_forest_change_2022_v1_10 datamask ==1',
    'normalization': 'original fixed normalization; no explicit zeroing by validity mask',
}
(ROOT/'imagery_config.json').write_text(json.dumps(imagery_config, indent=2))
print('Original post-event scenes:', scene_count)
print('Pure inference inputs:', stack.bandNames().getInfo())

In [ ]:
# @title Download the fixed probe tile (no edits needed)
# One earlier validation tile, used below to check the saved Random Forest reproduces its saved probabilities.
# (The U-Net probability check that used to run here has been removed.)
import io, requests, rasterio
from rasterio.io import MemoryFile
PROBE = {'split': 'val', 'tile': 'r031_c007', 'sha256': '506d29acd56d37410ad0e8f6c46f9ca2b0378d84955148ed2ee018ee136371fc', 'valid_pixels': 16384, 'positive_pixels': 0}
probe_row, probe_col = [int(v[1:]) for v in PROBE['tile'].split('_')]
probe_transform = [10,0,1641080+probe_col*1280,0,-10,5350110-probe_row*1280]
probe_url = stack.getDownloadURL({'crs':CRS, 'crs_transform':probe_transform,
    'dimensions':[128,128], 'format':'GEO_TIFF', 'filePerBand':False})
response = requests.get(probe_url, timeout=180)
response.raise_for_status()
with MemoryFile(response.content) as memory:
    with memory.open() as dataset:
        probe_array = dataset.read()
assert probe_array.shape == (9,128,128)
probe_valid = probe_array[8] > 0.5
assert int(probe_valid.sum()) == PROBE['valid_pixels']
parity = {'tile':PROBE['tile'], 'valid_pixels':int(probe_valid.sum()),
          'optical_valid_percentiles':np.percentile(probe_array[:6,probe_valid],[2,50,98]).tolist()}
(ROOT/'parity_check.json').write_text(json.dumps(parity,indent=2))
print('Probe tile ready:', parity)

In [ ]:
# The current agreed road file is pinned for reproducible joins and geometry.
import requests
import geopandas as gpd

ROAD_COMMIT = '8d739079424b3b6c7077f470346a8840f20a0e1c'
ROAD_SHA256 = '127841d08d83cfc7a3ec82cf34f850d92d6d66550e650cea4304a2a1aff26b0e'
ROAD_URL = ('https://raw.githubusercontent.com/Loading-my-brain/x-band/'
            + ROAD_COMMIT + '/kaikoura_road_route/kai_route.gpkg')
road_path = ROOT/'kai_route.gpkg'
if not road_path.exists():
    response = requests.get(ROAD_URL, timeout=120)
    response.raise_for_status()
    temporary = road_path.with_suffix('.tmp')
    temporary.write_bytes(response.content)
    temporary.replace(road_path)
assert hashlib.sha256(road_path.read_bytes()).hexdigest() == ROAD_SHA256, 'Road file hash mismatch.'

roads = read_road_segments(road_path)
plan = plan_road_tiles(roads)
print(f'Agreed road components: {len(roads)} | {roads.length.sum()/1000:.2f} km')
print(f'All road-touching original tiles: {len(plan["tiles"])} | download blocks: {len(plan["blocks"])}')
print('No training/validation/test tile budget or minimum valid-fraction filter is used.')
data_signature = hashlib.sha256(json.dumps(imagery_config, sort_keys=True).encode()).hexdigest()
blocks = download_inference_blocks(stack, plan, ROOT/'input_cache', data_signature=data_signature)
print('All road input blocks ready.')

In [ ]:
# Check a 512-block window against a direct original-128 download.
from rasterio.windows import Window
first_tile = plan['tiles'][0]
first_block = next(b for b in blocks if b['block_id']==first_tile['block_id'])
with rasterio.open(first_block['path']) as dataset:
    window_array = dataset.read(window=Window(first_tile['local_col']*128,first_tile['local_row']*128,128,128))
direct_url = stack.getDownloadURL({'crs':CRS,'crs_transform':first_tile['transform'],
    'dimensions':[128,128],'format':'GEO_TIFF','filePerBand':False})
response = requests.get(direct_url,timeout=180)
response.raise_for_status()
with MemoryFile(response.content) as memory:
    with memory.open() as dataset:
        direct_array = dataset.read()
assert np.array_equal(window_array[8],direct_array[8]), 'Block/window validity masks differ.'
block_difference = float(np.max(np.abs(window_array[:8]-direct_array[:8])))
assert np.allclose(window_array,direct_array,atol=1e-5,rtol=1e-6), 'Download block changes original inputs.'
parity['block_window_tile'] = first_tile['tile']
parity['block_window_max_absolute_difference'] = block_difference
(ROOT/'parity_check.json').write_text(json.dumps(parity,indent=2))
print('512-download / 128-inference window check passed. Max input difference:',block_difference)

In [ ]:
# Load the previously trained 150-tree RF; no training and no encoded weights in this cell.
import gzip, io, joblib, sklearn
import requests
from pathlib import Path

RF_METADATA = {'model': 'Random Forest', 'trees': 150, 'threshold': 0.28, 'threshold_selection': 'existing validation F1 on 0.01–0.99 grid; frozen', 'training_pixels': 150000, 'seed': 761, 'features': ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'elevation', 'slope'], 'file_sha256': 'db43c611a13967c447049e6934e617a4b4111cfaf9fc8fc8671c6e0c2cd1f5ad', 'original_joblib_sha256': 'f19a1d236938093a6b905aedf80cb8893c362e356ec9495de8eef3b73c8baad6', 'training': 'previous cloud-trained baseline; no new training', 'drive_file_id': '136t-zBYg7bNVK3ye5UEI3YReoeqdpXiM'}
RF_THRESHOLD = 0.28
# The saved RF is accepted either compressed (rf_150_trees.joblib.gz) or extracted (rf_150_trees.joblib);
# both are checked against the saved checksums. Where it is looked for, in order:
#   1. already in place from an earlier run (rf_checkpoint)
#   2. GitHub (RF_GITHUB_URL, set in the Project inputs cell at the top)
#   3. a copy you uploaded to Colab yourself (folder icon on the left), in /content or in ROOT
#   4. none of these -> stop and ask ncha509 for the file
rf_checkpoint = ROOT/'rf_150_trees.joblib'
RF_UPLOAD_PATHS = [folder/name for folder in (Path('/content'), ROOT, Path.cwd())
                   for name in ('rf_150_trees.joblib.gz', 'rf_150_trees.joblib')]

def _rf_bytes(content):
    # Raw joblib bytes if content is the saved RF (.joblib.gz or extracted .joblib), else None
    digest = hashlib.sha256(content).hexdigest()
    if digest == RF_METADATA['file_sha256']:
        return gzip.decompress(content)
    if digest == RF_METADATA['original_joblib_sha256']:
        return content
    return None

original_bytes = _rf_bytes(rf_checkpoint.read_bytes()) if rf_checkpoint.exists() else None
if original_bytes is None:
    try:
        response = requests.get(RF_GITHUB_URL, timeout=300)
        response.raise_for_status()
        original_bytes = _rf_bytes(response.content)
        if original_bytes is None:
            raise ValueError('downloaded file does not match the saved Random Forest checksum')
        print('Loaded saved RF from GitHub:', RF_GITHUB_URL)
    except Exception as error:
        for path in RF_UPLOAD_PATHS:
            if path.exists():
                original_bytes = _rf_bytes(path.read_bytes())
                if original_bytes is not None:
                    print('Not on GitHub; using the copy uploaded to Colab:', path)
                    break
        if original_bytes is None:
            raise RuntimeError(
                f'The saved Random Forest is not on GitHub ({error}) and no uploaded copy was found in Colab.\n'
                'Ask ncha509 to send you the downloadable rf_150_trees.joblib.gz (or extracted rf_150_trees.joblib) file, '
                'upload it in Colab to /content (folder icon on the left), then re-run this cell.') from error
    rf_checkpoint.write_bytes(original_bytes)
assert hashlib.sha256(original_bytes).hexdigest()==RF_METADATA['original_joblib_sha256']
assert sklearn.__version__=='1.6.1', 'Use the pinned scikit-learn 1.6.1 setup cell, then restart runtime.'
rf_model = joblib.load(io.BytesIO(original_bytes))
del original_bytes
assert rf_model.n_estimators==150 and len(rf_model.estimators_)==150
assert rf_model.n_features_in_==8 and list(rf_model.classes_)==[0,1]

# Compare a few fixed saved-validation probabilities as a reproducibility check.
rf_probe_indices = np.array([0, 528, 1057, 1585, 2114, 2642, 3171, 3699, 4228, 4756, 5285, 5813, 6342, 6870, 7399, 7927, 8456, 8984, 9513, 10041, 10570, 11098, 11627, 12155, 12684, 13212, 13741, 14269, 14798, 15326, 15855, 16383],dtype=int)
rf_probe_expected = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.004790991747656947, 0.005573168852091931, 0.0, 0.0, 0.06755642658632621, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.006165902029909369, 0.0, 0.0, 0.006051049533961453, 0.0, 0.0062099074954612265, 0.024638293955685876, 0.02891863884114688, 0.0, 0.0, 0.0, 0.0, 0.06510894697032106, 0.017732386642129453, 0.028161678835819513],dtype='float32')
rf_probe_features = normalize_features(probe_array[:8]).reshape(8,-1).T[rf_probe_indices]
rf_probe_actual = rf_model.predict_proba(rf_probe_features)[:,1]
rf_parity_error = float(np.max(np.abs(rf_probe_actual-rf_probe_expected)))
assert rf_parity_error<1e-6, f'Saved RF feature/parity mismatch: {rf_parity_error}'
print('Saved RF ready: 150 trees | frozen threshold 0.28 | parity max error',rf_parity_error)

In [ ]:
# @title Load RF / GNS comparison tools (no edits needed)
# Embedded reviewed helpers; run model computations in Colab only.
comparison_source = '"""Inference-only RF / U-Net / GNS integration comparison.\n\nNo training or model prediction runs on import. RF inference is an explicit Colab\noperation. Reference scores on road-selected tiles are integration demonstrations,\nnot new independent held-out accuracy or confirmed road-closure accuracy.\n"""\nfrom pathlib import Path\nimport json\n\nimport geopandas as gpd\nimport numpy as np\nimport pandas as pd\nimport rasterio\nfrom rasterio.features import shapes\nfrom rasterio.transform import Affine\nfrom rasterio.windows import Window\nfrom shapely import make_valid, union_all\nfrom shapely.geometry import shape\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\ntry:\n    from .unet_road_integration import (\n        METRIC_CRS, TILE_SIZE, FEATURE_NAMES, aoi_metric, polygon_parts,\n        empty_geodataframe, assess_road_impacts, _json_atomic)\nexcept ImportError:\n    from unet_road_integration import (\n        METRIC_CRS, TILE_SIZE, FEATURE_NAMES, aoi_metric, polygon_parts,\n        empty_geodataframe, assess_road_impacts, _json_atomic)\n\n\ndef _write_comparison_geojson(layer, path):\n    """Write full-precision WGS84 GeoJSON with valid polygon geometry.\n\n    Repair after projection, retain polygon components for polygon inputs, and\n    avoid the OGR GeoJSON driver\'s default coordinate rounding. Validate a\n    read-back before replacing the destination file. Metrics are not recomputed.\n    """\n    path = Path(path)\n    if layer.empty:\n        path.write_text(json.dumps({"type": "FeatureCollection", "features": []}))\n        return\n    if layer.crs is None:\n        raise ValueError("GeoJSON export requires an explicit input CRS.")\n    geographic = layer.to_crs(4326).copy()\n    repaired = []\n    for original in geographic.geometry:\n        if original is None or original.is_empty:\n            repaired.append(None)\n            continue\n        fixed = make_valid(original)\n        if original.geom_type in {"Polygon", "MultiPolygon"}:\n            parts = polygon_parts(fixed)\n            fixed = union_all(parts) if parts else None\n        repaired.append(fixed)\n    geographic.geometry = repaired\n    geographic = geographic.loc[geographic.geometry.notna() & ~geographic.geometry.is_empty].copy()\n    if not geographic.geometry.is_valid.all():\n        raise ValueError(f"Invalid geometry remains after WGS84 repair: {path.name}")\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    temporary.write_text(geographic.to_json(drop_id=True))\n    if not geographic.empty:\n        roundtrip = gpd.read_file(temporary)\n        if len(roundtrip) != len(geographic) or not roundtrip.geometry.notna().all() or not roundtrip.geometry.is_valid.all():\n            raise ValueError(f"GeoJSON validity/read-back check failed: {path.name}")\n    temporary.replace(path)\n\n\ndef infer_rf_road_tiles(model, plan, block_records, out_dir, *, threshold=0.28,\n                        pixel_chunk_size=65536, expected_n_estimators=150,\n                        checkpoint_metadata=None):\n    """Apply the saved RF to valid pixels in every road-selected original tile."""\n    try:\n        from .landslide_benchmark import normalize_features\n    except ImportError:\n        from landslide_benchmark import normalize_features\n    if not 0 < threshold < 1 or not 1 <= pixel_chunk_size <= 65536:\n        raise ValueError("Use a frozen probability threshold and chunks of 1–65,536 pixels.")\n    if getattr(model, "n_features_in_", None) != 8:\n        raise ValueError("Expected the saved eight-feature Random Forest.")\n    if getattr(model, "n_estimators", None) != expected_n_estimators:\n        raise ValueError(f"Expected the frozen {expected_n_estimators}-tree RF checkpoint.")\n    estimators = getattr(model, "estimators_", None)\n    if estimators is None or len(estimators) != expected_n_estimators:\n        raise ValueError(f"Saved RF must contain exactly {expected_n_estimators} fitted estimators.")\n    classes = list(model.classes_)\n    if len(classes) != 2 or 0 not in classes or 1 not in classes:\n        raise ValueError("RF classes must be background=0 and mapped landslide=1.")\n    positive_index = classes.index(1)\n    blocks = {item["block_id"]: item for item in block_records}\n    missing = sorted({tile["block_id"] for tile in plan["tiles"]} - blocks.keys())\n    if missing:\n        raise RuntimeError(f"Missing feature blocks {missing}; RF inference cannot mark roads unaffected.")\n    out_dir = Path(out_dir)\n    tile_dir = out_dir / "inference_tiles"\n    tile_dir.mkdir(parents=True, exist_ok=True)\n    roi = aoi_metric()\n    hazard_rows, coverage_rows, manifest = [], [], []\n    for index, tile in enumerate(plan["tiles"]):\n        window = Window(tile["local_col"] * TILE_SIZE, tile["local_row"] * TILE_SIZE,\n                        TILE_SIZE, TILE_SIZE)\n        with rasterio.open(blocks[tile["block_id"]]["path"]) as dataset:\n            array = dataset.read(window=window)\n            transform = dataset.window_transform(window)\n        if array.shape != (9, TILE_SIZE, TILE_SIZE) or not transform.almost_equals(Affine(*tile["transform"])):\n            raise ValueError(f"Original feature tile window/grid mismatch: {tile[\'tile\']}")\n        valid = (array[8] > 0.5) & np.isfinite(array[:8]).all(axis=0)\n        features = normalize_features(array[:8])\n        selected = features.reshape(8, -1).T[valid.ravel()]\n        valid_probability = np.empty(len(selected), dtype="float32")\n        for start in range(0, len(selected), pixel_chunk_size):\n            probability = model.predict_proba(selected[start:start + pixel_chunk_size])\n            valid_probability[start:start + len(probability)] = probability[:, positive_index]\n        if not np.isfinite(valid_probability).all() or np.any((valid_probability < 0) | (valid_probability > 1)):\n            raise RuntimeError("Saved RF returned invalid probabilities; no road-impact result produced.")\n        probability = np.full(valid.shape, -9999, dtype="float32")\n        probability[valid] = valid_probability\n        binary = np.full(valid.shape, 255, dtype="uint8")\n        binary[valid] = (valid_probability >= threshold).astype("uint8")\n        profile = {"driver": "GTiff", "width": TILE_SIZE, "height": TILE_SIZE,\n            "count": 1, "crs": METRIC_CRS, "transform": transform, "compress": "deflate",\n            "dtype": "float32", "nodata": -9999}\n        with rasterio.open(tile_dir / (tile["tile"] + "_probability.tif"), "w", **profile) as dataset:\n            dataset.write(probability, 1)\n        profile.update(dtype="uint8", nodata=255)\n        with rasterio.open(tile_dir / (tile["tile"] + "_mask.tif"), "w", **profile) as dataset:\n            dataset.write(binary, 1)\n        for geometry, value in shapes(valid.astype("uint8"), mask=valid, transform=transform):\n            if value == 1:\n                for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                    coverage_rows.append({"tile": tile["tile"], "geometry": part})\n        for geometry, value in shapes(binary, mask=valid & (binary == 1), transform=transform):\n            if value == 1:\n                for part in polygon_parts(make_valid(shape(geometry)).intersection(roi)):\n                    hazard_rows.append({"tile": tile["tile"], "geometry": part})\n        manifest.append({"tile": tile["tile"], "valid_pixels": int(valid.sum()),\n            "positive_pixels": int((binary == 1).sum()), "valid_fraction": float(valid.mean())})\n        if (index + 1) % 10 == 0 or index + 1 == len(plan["tiles"]):\n            print(f"Frozen RF inference: {index + 1}/{len(plan[\'tiles\'])} original tiles", flush=True)\n    hazards = gpd.GeoDataFrame(hazard_rows, crs=METRIC_CRS) if hazard_rows else empty_geodataframe(["tile", "geometry"])\n    coverage = gpd.GeoDataFrame(coverage_rows, crs=METRIC_CRS) if coverage_rows else empty_geodataframe(["tile", "geometry"])\n    _json_atomic(out_dir / "inference_manifest.json", manifest)\n    config = {"model": "saved Random Forest", "n_estimators": int(model.n_estimators),\n        "threshold": float(threshold), "feature_order": FEATURE_NAMES,\n        "normalization": "unchanged benchmark normalize_features", "selected_tile_count": len(plan["tiles"]),\n        "training": "none; saved checkpoint inference only", "checkpoint_metadata": checkpoint_metadata}\n    _json_atomic(out_dir / "inference_config.json", config)\n    return {"hazards": hazards, "coverage": coverage, "tiles": manifest,\n            "threshold": float(threshold), "tile_directory": str(tile_dir), "config": config}\n\n\ndef _prediction_directory(value):\n    if isinstance(value, dict):\n        value = value["tile_directory"]\n    directory = Path(value)\n    return directory if directory.name == "inference_tiles" else directory / "inference_tiles"\n\n\ndef _read_prediction(tile, directory, threshold):\n    with rasterio.open(directory / (tile + "_probability.tif")) as dataset:\n        probability, transform, crs = dataset.read(1), dataset.transform, dataset.crs\n        nodata = dataset.nodata\n    with rasterio.open(directory / (tile + "_mask.tif")) as dataset:\n        mask = dataset.read(1)\n        if dataset.crs != crs or not dataset.transform.almost_equals(transform):\n            raise ValueError(f"Mask/probability grid mismatch: {tile}")\n    if probability.shape != (TILE_SIZE, TILE_SIZE) or crs.to_epsg() != 2193:\n        raise ValueError(f"Prediction is not an original NZTM 128×128 tile: {tile}")\n    valid = np.isfinite(probability) & (mask != 255)\n    if nodata is not None:\n        valid &= probability != nodata\n    if np.any(valid & ~np.isin(mask, [0, 1])) or np.any(valid & ((probability < 0) | (probability > 1))):\n        raise ValueError(f"Invalid probability/mask values: {tile}")\n    if np.any((mask[valid] == 1) != (probability[valid] >= threshold)):\n        raise ValueError(f"Saved binary mask does not match its frozen threshold: {tile}")\n    return probability, mask == 1, valid, transform, crs\n\n\ndef _read_label(record):\n    """Read a one-band 128 label tile, or a 128 window of a larger label block.\n\n    Optional band 2 is an explicit label-validity mask. A nodata value of zero\n    does not discard legitimate background labels; inference coverage masks AOI.\n    """\n    with rasterio.open(record["path"]) as dataset:\n        if dataset.width == TILE_SIZE and dataset.height == TILE_SIZE:\n            window = None\n            transform = dataset.transform\n        else:\n            if "local_row" not in record or "local_col" not in record:\n                raise ValueError("Larger reference blocks require local_row/local_col metadata.")\n            window = Window(record["local_col"] * TILE_SIZE, record["local_row"] * TILE_SIZE,\n                            TILE_SIZE, TILE_SIZE)\n            transform = dataset.window_transform(window)\n        labels = dataset.read(1, window=window)\n        valid = np.isfinite(labels) & np.isin(labels, [0, 1])\n        if dataset.count > 1:\n            valid &= dataset.read(2, window=window) > 0.5\n        crs = dataset.crs\n        allowed = [0, 1, 255, -9999]\n        if dataset.nodata is not None:\n            allowed.append(dataset.nodata)\n        if np.any(np.isfinite(labels) & ~np.isin(labels, allowed)):\n            raise ValueError(f"Reference is not a binary GNS target: {record[\'tile\']}")\n    if labels.shape != (TILE_SIZE, TILE_SIZE) or crs.to_epsg() != 2193:\n        raise ValueError("Reference label grid must be an original NZTM 128×128 tile.")\n    return labels > 0.5, valid, transform, crs\n\n\ndef _counts(truth, predicted):\n    return {"tn": int((~truth & ~predicted).sum()), "fp": int((~truth & predicted).sum()),\n            "fn": int((truth & ~predicted).sum()), "tp": int((truth & predicted).sum())}\n\n\ndef _count_metrics(counts):\n    tn, fp, fn, tp = (counts[key] for key in ["tn", "fp", "fn", "tp"])\n    return {**counts, "precision": tp / max(tp + fp, 1), "recall": tp / max(tp + fn, 1),\n        "f1": 2 * tp / max(2 * tp + fp + fn, 1), "iou": tp / max(tp + fp + fn, 1),\n        "accuracy": (tp + tn) / max(tp + tn + fp + fn, 1),\n        "balanced_accuracy": (tp / max(tp + fn, 1) + tn / max(tn + fp, 1)) / 2}\n\n\ndef score_prediction_tiles(label_records, predictions_by_model, out_dir, *,\n                            thresholds=None, evaluation_scope="road-selected integration demonstration; may overlap model training/validation"):\n    """Compare frozen predictions against aligned GNS raster labels, never tune."""\n    thresholds = thresholds or {"U-Net": 0.31, "Random Forest": 0.28}\n    if set(predictions_by_model) != set(thresholds):\n        raise ValueError("Every model needs an explicit frozen threshold.")\n    if len({record["tile"] for record in label_records}) != len(label_records):\n        raise ValueError("Duplicate reference tile IDs.")\n    directories = {name: _prediction_directory(value) for name, value in predictions_by_model.items()}\n    counts = {name: dict(tn=0, fp=0, fn=0, tp=0) for name in directories}\n    probabilities = {name: [] for name in directories}\n    truth_parts, tile_rows = [], []\n    for record in label_records:\n        truth, label_valid, transform, crs = _read_label(record)\n        items = {name: _read_prediction(record["tile"], directory, thresholds[name])\n                 for name, directory in directories.items()}\n        shared = label_valid.copy()\n        for probability, predicted, valid, model_transform, model_crs in items.values():\n            if model_crs != crs or not model_transform.almost_equals(transform):\n                raise ValueError(f"Reference/prediction grid mismatch: {record[\'tile\']}")\n            shared &= valid\n        truth_parts.append(truth[shared])\n        for name, (probability, predicted, valid, _, _) in items.items():\n            item_counts = _counts(truth[shared], predicted[shared])\n            for key in counts[name]:\n                counts[name][key] += item_counts[key]\n            probabilities[name].append(probability[shared])\n            tile_rows.append({"tile": record["tile"], "model": name,\n                "shared_valid_pixels": int(shared.sum()), "gns_positive_pixels": int((truth & shared).sum()),\n                "threshold": float(thresholds[name]), **_count_metrics(item_counts)})\n    truth = np.concatenate(truth_parts) if truth_parts else np.array([], dtype=bool)\n    if not len(truth):\n        raise ValueError("No shared valid reference/prediction pixels are available.")\n    metrics = {}\n    for name in directories:\n        values = np.concatenate(probabilities[name])\n        result = _count_metrics(counts[name])\n        result.update(threshold=float(thresholds[name]), valid_pixels=int(len(truth)),\n                      positive_fraction=float(truth.mean()))\n        if len(np.unique(truth)) == 2:\n            result["average_precision"] = float(average_precision_score(truth, values))\n            result["roc_auc"] = float(roc_auc_score(truth, values))\n        metrics[name] = result\n    metadata = {"evaluation_scope": evaluation_scope, "threshold_tuning": "none; frozen model thresholds",\n        "reference": "GNS V3 source/debris union at original 10 m grid; not road-closure truth",\n        "tile_count": len(label_records), "shared_valid_pixels": int(len(truth)),\n        "new_held_out_accuracy_claim": False}\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    _json_atomic(out_dir / "pixel_reference_metrics.json", {"metadata": metadata, "models": metrics})\n    pd.DataFrame(metrics).T.to_csv(out_dir / "pixel_reference_metrics.csv")\n    pd.DataFrame(tile_rows).to_csv(out_dir / "pixel_reference_metrics_by_tile.csv", index=False)\n    return {"metadata": metadata, "models": metrics, "per_tile": tile_rows}\n\n\ndef compare_road_models(roads, unet_result, rf_result, gns_polygons, out_dir):\n    """Compare three hazards on exactly the same observed road-centreline portions."""\n    roi = aoi_metric()\n    unet_coverage = unet_result["coverage"].to_crs(METRIC_CRS)\n    rf_coverage = rf_result["coverage"].to_crs(METRIC_CRS)\n    shared_geometry = union_all(unet_coverage.geometry.values).intersection(\n        union_all(rf_coverage.geometry.values)).intersection(roi)\n    shared = gpd.GeoDataFrame({"geometry": polygon_parts(shared_geometry)}, crs=METRIC_CRS)\n    assessments = {\n        "U-Net": assess_road_impacts(roads, unet_result["hazards"], shared),\n        "Random Forest": assess_road_impacts(roads, rf_result["hazards"], shared),\n        "GNS reference": assess_road_impacts(roads, gns_polygons, shared),\n    }\n    provenance = {\n        "U-Net": "frozen U-Net predicted mapped-landslide extent",\n        "Random Forest": "saved 150-tree Random Forest predicted mapped-landslide extent",\n        "GNS reference": "GNS V3 mapped source/debris inventory reference; not confirmed road blockage",\n    }\n    for name, assessment in assessments.items():\n        assessment["roads"]["hazard_source"] = provenance[name]\n    first = assessments["U-Net"]["roads"]\n    table = first[["road_id", "full_id", "osm_id", "road_length_m", "observed_length_m",\n                   "unknown_length_m", "coverage_fraction", "geometry"]].copy()\n    for name, prefix in [("U-Net", "unet"), ("Random Forest", "rf"), ("GNS reference", "gns")]:\n        layer = assessments[name]["roads"]\n        selected = layer[["road_id", "affected_length_m", "max_affected_run_m", "impact_status"]].rename(\n            columns={"affected_length_m": prefix + "_affected_length_m",\n                     "max_affected_run_m": prefix + "_max_affected_run_m", "impact_status": prefix + "_status"})\n        table = table.merge(selected, on="road_id", validate="one_to_one")\n    u, r = table.unet_affected_length_m > 1e-6, table.rf_affected_length_m > 1e-6\n    table["model_agreement"] = np.select([u & r, u & ~r, ~u & r],\n        ["both", "unet_only", "rf_only"], default="neither")\n    table["has_shared_observation"] = table.observed_length_m > 1e-6\n    table["has_unknown_length"] = table.unknown_length_m > 1e-6\n    observed = table.loc[table.has_shared_observation]\n    summary = {"scope": "reference overlap on shared observed centreline portions; not confirmed road blockage",\n        "new_held_out_accuracy_claim": False, "road_parts": len(table),\n        "observed_road_parts": len(observed), "not_assessed_parts": int((~table.has_shared_observation).sum()),\n        "agreement_counts_observed_parts_only": observed.model_agreement.value_counts().to_dict(),\n        "total_observed_length_m": float(table.observed_length_m.sum()),\n        "total_unknown_length_m": float(table.unknown_length_m.sum()), "models": {}}\n    reference = observed.gns_affected_length_m.to_numpy() > 1e-6\n    for name, prefix in [("U-Net", "unet"), ("Random Forest", "rf"), ("GNS reference", "gns")]:\n        length = table[prefix + "_affected_length_m"]\n        item = {"affected_road_parts": int((length > 1e-6).sum()),\n            "affected_length_m": float(length.sum()),\n            "max_continuous_affected_run_m": float(table[prefix + "_max_affected_run_m"].max())}\n        if name != "GNS reference":\n            predicted = observed[prefix + "_affected_length_m"].to_numpy() > 1e-6\n            item["observed_roadpart_reference_metrics"] = _count_metrics(_counts(reference, predicted))\n            differences = observed[prefix + "_affected_length_m"] - observed.gns_affected_length_m\n            item["affected_length_mae_per_observed_roadpart_m"] = float(differences.abs().mean()) if len(differences) else None\n            table[prefix + "_minus_gns_length_m"] = table[prefix + "_affected_length_m"] - table.gns_affected_length_m\n        summary["models"][name] = item\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    table.drop(columns="geometry").to_csv(out_dir / "road_model_agreement.csv", index=False)\n    _write_comparison_geojson(table, out_dir / "road_model_agreement.geojson")\n    table.to_file(out_dir / "road_model_agreement.gpkg", layer="road_comparison", driver="GPKG")\n    _write_comparison_geojson(shared, out_dir / "shared_observed_coverage.geojson")\n    for name, stem in [("U-Net", "unet"), ("Random Forest", "rf"), ("GNS reference", "gns_reference")]:\n        _write_comparison_geojson(assessments[name]["hazards"], out_dir / (stem + "_hazards.geojson"))\n        _write_comparison_geojson(assessments[name]["affected_runs"], out_dir / (stem + "_affected_runs.geojson"))\n    _json_atomic(out_dir / "road_reference_summary.json", summary)\n    return {"roads": table, "assessments": assessments, "shared_coverage": shared, "summary": summary}\n\n\ndef export_comparison_visuals(comparison, out_dir, *, pixel_scores=None,\n                               plan=None, block_records=None, label_records=None,\n                               predictions_by_model=None, thresholds=None):\n    """Matched maps and label-independent qualitative samples, no model calls."""\n    import folium\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import ListedColormap\n    out_dir = Path(out_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    roads, assessments = comparison["roads"], comparison["assessments"]\n    colors = {"U-Net": "#ed4337", "Random Forest": "#8c43d1", "GNS reference": "#31a354"}\n    map_view = folium.Map(location=[-42.25, 173.75], zoom_start=10, tiles=None)\n    folium.TileLayer("https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",\n        attr="Esri, Maxar, Earthstar Geographics and GIS User Community",\n        name="Illustrative imagery — dates unverified; not confirmed 2016", overlay=False).add_to(map_view)\n    folium.GeoJson(roads.to_crs(4326).to_json(), name="Shared road components",\n        style_function=lambda feature: {"color": "#65c7d0", "weight": 2},\n        tooltip=folium.GeoJsonTooltip(fields=["road_id", "model_agreement", "observed_length_m", "unknown_length_m",\n            "unet_affected_length_m", "rf_affected_length_m", "gns_affected_length_m"])).add_to(map_view)\n    unknown = assessments["U-Net"]["unknown_runs"]\n    if not unknown.empty:\n        folium.GeoJson(unknown.to_crs(4326).to_json(), name="Unknown / unobserved sections",\n            style_function=lambda feature: {"color": "#808080", "weight": 4}).add_to(map_view)\n    for name in ["U-Net", "Random Forest", "GNS reference"]:\n        group = folium.FeatureGroup(name=name + " hazards + affected runs", show=name == "U-Net")\n        for key, weight, opacity in [("hazards", 1, 0.18), ("affected_runs", 5, 0)]:\n            layer = assessments[name][key]\n            if not layer.empty:\n                style = {"color": colors[name], "weight": weight, "fillOpacity": opacity}\n                folium.GeoJson(layer.to_crs(4326).to_json(), style_function=lambda feature, style=style: style).add_to(group)\n        group.add_to(map_view)\n    bounds = roads.to_crs(4326).total_bounds\n    map_view.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])\n    folium.LayerControl(collapsed=False).add_to(map_view)\n    notice = "<div style=\'position:fixed;bottom:25px;left:15px;z-index:9999;background:white;padding:8px;font-size:12px\'>" \\\n        "Toggle model/reference layers: red U-Net · purple RF · green GNS<br>" \\\n        "Gray: unknown. GNS overlap is not confirmed closure. Imagery dates unverified.</div>"\n    map_view.get_root().html.add_child(folium.Element(notice))\n    map_view.save(out_dir / "three_way_comparison.html")\n    metric_bounds = roads.total_bounds\n    padding = max(metric_bounds[2] - metric_bounds[0], metric_bounds[3] - metric_bounds[1]) * 0.02\n    fig, axes = plt.subplots(1, 3, figsize=(15, 7), sharex=True, sharey=True)\n    for ax, name in zip(axes, ["U-Net", "Random Forest", "GNS reference"]):\n        layer = assessments[name]\n        if not layer["hazards"].empty:\n            layer["hazards"].plot(ax=ax, color=colors[name], alpha=0.25)\n        roads.plot(ax=ax, color="#84bdc3", linewidth=0.8)\n        if not unknown.empty:\n            unknown.plot(ax=ax, color="#808080", linewidth=1.5)\n        if not layer["affected_runs"].empty:\n            layer["affected_runs"].plot(ax=ax, color=colors[name], linewidth=2)\n        ax.set_xlim(metric_bounds[0] - padding, metric_bounds[2] + padding)\n        ax.set_ylim(metric_bounds[1] - padding, metric_bounds[3] + padding)\n        ax.set_title(name)\n        ax.set_xlabel("NZTM easting (m)")\n    axes[0].set_ylabel("NZTM northing (m)")\n    fig.suptitle("Same roads and observed coverage: model predictions versus GNS mapped reference\\nIntegration demonstration; no confirmed road closure or new held-out accuracy")\n    fig.tight_layout()\n    fig.savefig(out_dir / "three_way_maps.png", dpi=180)\n    plt.close(fig)\n    fig, axes = plt.subplots(1, 2 if pixel_scores else 1, figsize=(12 if pixel_scores else 7, 4), squeeze=False)\n    offset = 0\n    if pixel_scores:\n        score_table = pd.DataFrame(pixel_scores["models"]).T[["f1", "iou"]]\n        score_table.plot.bar(ax=axes[0, 0], color=["#327cb8", "#f2a535"], rot=0)\n        axes[0, 0].set_ylim(0, 1)\n        axes[0, 0].set_ylabel("Pixel score against GNS")\n        axes[0, 0].set_title("Road-selected integration pixels; not held-out")\n        offset = 1\n    names = ["U-Net", "Random Forest", "GNS reference"]\n    values = [comparison["summary"]["models"][name]["affected_length_m"] / 1000 for name in names]\n    axes[0, offset].bar(names, values, color=[colors[name] for name in names])\n    axes[0, offset].set_ylabel("Observed road-centreline overlap (km)")\n    axes[0, offset].set_title("Potential impact length; not blockage accuracy")\n    fig.tight_layout()\n    fig.savefig(out_dir / "reference_metrics_and_road_lengths.png", dpi=180)\n    plt.close(fig)\n    if all(value is not None for value in [plan, block_records, label_records, predictions_by_model]):\n        thresholds = thresholds or {"U-Net": 0.31, "Random Forest": 0.28}\n        directories = {name: _prediction_directory(value) for name, value in predictions_by_model.items()}\n        label_map = {record["tile"]: record for record in label_records}\n        blocks = {record["block_id"]: record for record in block_records}\n        ordered = sorted(plan["tiles"], key=lambda record: (record["row"], record["col"]))\n        indices = np.unique(np.linspace(0, len(ordered) - 1, min(6, len(ordered))).round().astype(int))\n        selected = [ordered[index] for index in indices]\n        fig, axes = plt.subplots(len(selected), 6, figsize=(18, 3 * len(selected)), squeeze=False)\n        error_cmap = ListedColormap(["#eeeeee", "#35a657", "#dc4437", "#3367d6"])\n        error_cmap.set_bad("#808080")\n        binary_cmap = plt.get_cmap("viridis").copy()\n        binary_cmap.set_bad("#808080")\n        for row, tile in enumerate(selected):\n            window = Window(tile["local_col"] * TILE_SIZE, tile["local_row"] * TILE_SIZE, TILE_SIZE, TILE_SIZE)\n            with rasterio.open(blocks[tile["block_id"]]["path"]) as dataset:\n                array = dataset.read(window=window)\n            truth, label_valid, _, _ = _read_label(label_map[tile["tile"]])\n            items = {name: _read_prediction(tile["tile"], directory, thresholds[name]) for name, directory in directories.items()}\n            shared = label_valid.copy()\n            for value in items.values():\n                shared &= value[2]\n            rgb = np.clip(array[[2, 1, 0]].transpose(1, 2, 0) / 0.30, 0, 1)\n            rgb[~shared] = 0.5\n            images = [rgb, np.ma.masked_where(~shared, truth)]\n            for name in ["U-Net", "Random Forest"]:\n                images.append(np.ma.masked_where(~shared, items[name][1]))\n            for name in ["U-Net", "Random Forest"]:\n                predicted = items[name][1]\n                error = np.zeros(truth.shape, dtype="uint8")\n                error[truth & predicted] = 1\n                error[~truth & predicted] = 2\n                error[truth & ~predicted] = 3\n                images.append(np.ma.masked_where(~shared, error))\n            titles = ["Post-event RGB", "GNS reference", "U-Net mask", "RF mask", "U-Net errors", "RF errors"]\n            for col, (image, title) in enumerate(zip(images, titles)):\n                if col == 0:\n                    axes[row, col].imshow(image)\n                else:\n                    axes[row, col].imshow(image, cmap=error_cmap if col >= 4 else binary_cmap,\n                                         vmin=0, vmax=3 if col >= 4 else 1, interpolation="nearest")\n                axes[row, col].axis("off")\n                if row == 0:\n                    axes[row, col].set_title(title, pad=24)\n            axes[row, 0].text(0, 1.03, tile["tile"], transform=axes[row, 0].transAxes)\n        fig.suptitle("Fixed road-tile quantiles; no selection by model scores\\nMasks: yellow landslide · purple background · gray invalid\\nErrors against GNS: green TP · red FP · blue FN")\n        fig.tight_layout(rect=[0, 0, 1, 0.97])\n        fig.savefig(out_dir / "deterministic_tile_examples.png", dpi=160)\n        plt.close(fig)\n        _json_atomic(out_dir / "qualitative_example_selection.json", {"rule": "even index quantiles of road-selected tiles ordered north–south, then west–east; no metric selection",\n            "tiles": [tile["tile"] for tile in selected]})\n    return {"map": map_view, "output_directory": str(out_dir)}\n'
(module_dir/'road_model_comparison.py').write_text(comparison_source)
import importlib
importlib.invalidate_caches()
if "road_model_comparison" in sys.modules:
    importlib.reload(sys.modules["road_model_comparison"])
from road_model_comparison import (infer_rf_road_tiles, score_prediction_tiles,
    compare_road_models, export_comparison_visuals)
print('RF / U-Net / GNS comparison tools ready.')

In [ ]:
# =============================================================================
# Builds landslide_by_osm_id.csv in the same layout as by_osm_id.csv (road widths),
# so routing reads it with its existing load_passability(). Called in the next cell.
# =============================================================================
# @title Routing CSV builder: landslide_by_osm_id (same layout as by_osm_id.csv)
"""
One row per OSM way, using the same column names as the road-width file (by_osm_id.csv),
so the routing notebook can read it with its existing load_passability().

Mapping from the U-Net road verdict to passable_<vehicle> (same for both vehicles):
  potentially_affected, longest run >= SLIDE_RUN_BLOCK_M -> unknown, "landslide: predicted run >= 50 m"
  potentially_affected, longest run <  SLIDE_RUN_BLOCK_M -> unknown, "landslide: predicted overlap < 50 m"
  not_assessed / partially_observed                     -> unknown, "landslide: road not fully observed"
  no_predicted_overlap                                  -> yes
Set SLIDE_BLOCK_STATUS = "no" to make long runs a hard block in A* (PEN_NO_S) instead of a delay.
"""
import pandas as pd

SLIDE_RUN_BLOCK_M = 50
SLIDE_BLOCK_STATUS = "unknown"     # "unknown" = big delay in A*, "no" = avoid if at all possible
VEHICLES = tuple(VEHICLE_WIDTHS)          # from Project inputs
REASON_BLOCK = f"landslide: predicted run >= {SLIDE_RUN_BLOCK_M} m"
REASON_DELAY = f"landslide: predicted overlap < {SLIDE_RUN_BLOCK_M} m"
REASON_UNSEEN = "landslide: road not fully observed"
IMPACT_RANK = {"no_predicted_overlap": 0, "partially_observed": 1, "not_assessed": 2, "potentially_affected": 3}


def _flag(series):
    """OSM bridge/tunnel tags ('yes', 'viaduct', ...) -> True/False."""
    return series.fillna("").astype(str).str.lower().isin(["yes", "true", "1", "viaduct", "culvert", "building_passage"])


def landslide_by_osm_id(road_result):
    """Collapse clipped road parts (w123:part0, w123:part1) into one row per OSM way."""
    df = road_result.drop(columns="geometry", errors="ignore").copy()
    df["full_id"] = df["road_id"].str.split(":").str[0]
    df["osm_id"] = df["full_id"].str.lstrip("w").astype("int64")
    df["rank"] = df["impact_status"].map(IMPACT_RANK)
    df["is_bridge"] = _flag(df["bridge"]) if "bridge" in df else False
    df["is_tunnel"] = _flag(df["tunnel"]) if "tunnel" in df else False

    out = df.groupby("osm_id").agg(
        full_id=("full_id", "first"), name=("name", "first"), highway=("highway", "first"),
        length_m=("road_length_m", "sum"), is_bridge=("is_bridge", "any"), is_tunnel=("is_tunnel", "any"),
        n_parts=("road_id", "size"),
        affected_length_m=("affected_length_m", "sum"), max_affected_run_m=("max_affected_run_m", "max"),
        observed_length_m=("observed_length_m", "sum"), unknown_length_m=("unknown_length_m", "sum"),
        rank=("rank", "max"),
    ).reset_index()
    out["coverage_fraction"] = out.observed_length_m / out.length_m
    out["impact_status"] = out["rank"].map({v: k for k, v in IMPACT_RANK.items()})
    out["width_median_m"] = out["drive_pinch_m"] = float("nan")   # width columns kept empty so route_blockages() works

    def verdict(r):
        if r.impact_status == "potentially_affected":
            if r.max_affected_run_m >= SLIDE_RUN_BLOCK_M:
                return SLIDE_BLOCK_STATUS, REASON_BLOCK
            return "unknown", REASON_DELAY
        if r.impact_status in ("not_assessed", "partially_observed"):
            return "unknown", REASON_UNSEEN
        return "yes", None

    status_reason = out.apply(verdict, axis=1, result_type="expand")
    for v in VEHICLES:
        out[f"passable_{v}"], out[f"reason_{v}"] = status_reason[0], status_reason[1]

    cols = ["osm_id", "full_id", "name", "highway", "length_m", "is_bridge", "is_tunnel", "n_parts",
            "affected_length_m", "max_affected_run_m", "observed_length_m", "unknown_length_m",
            "coverage_fraction", "impact_status", "width_median_m", "drive_pinch_m"]
    cols += [c for v in VEHICLES for c in (f"passable_{v}", f"reason_{v}")]
    return out[cols].round(2).sort_values("osm_id")

In [ ]:
# @title Random Forest road inference -> landslide_by_osm_id_rf.csv (input for routing)
# Frozen RF inference in Colab only. No retraining, threshold tuning or ground-truth input.
import shutil
import logging
import pandas as pd
from IPython.display import Image, Markdown, display
from google.colab import files

class _TiffColorMetadataFilter(logging.Filter):
    def filter(self, record):
        return 'Sum of Photometric type-related color channels and ExtraSamples' not in record.getMessage()

logging.getLogger('rasterio._env').addFilter(_TiffColorMetadataFilter())

RF_OUT = ROOT/'rf_outputs'
OUT = ROOT/'outputs'
OUT.mkdir(parents=True, exist_ok=True)
rf_result = infer_rf_road_tiles(rf_model, plan, blocks, RF_OUT,
    threshold=RF_THRESHOLD, checkpoint_metadata=RF_METADATA)
assert rf_result['threshold']==RF_THRESHOLD
assert {t['tile'] for t in rf_result['tiles']}=={t['tile'] for t in plan['tiles']}

# Random Forest road table -> landslide_by_osm_id_rf.csv (no GNS needed)
rf_assessment = assess_road_impacts(roads, rf_result['hazards'], rf_result['coverage'])
rf_roads = rf_assessment['roads']
rf_roads['hazard_source'] = 'saved 150-tree Random Forest predicted mapped-landslide extent'
assert rf_roads.road_id.is_unique
assert (rf_roads.affected_length_m <= rf_roads.observed_length_m + 1e-6).all()

rf_csv = landslide_by_osm_id(rf_roads)          # same function as Cell A
assert rf_csv.osm_id.is_unique
rf_path = OUT / 'landslide_by_osm_id_rf.csv'
rf_csv.to_csv(rf_path, index=False)
rf_roads.drop(columns='geometry').to_csv(OUT/'road_impact_rf.csv', index=False)
print(f'Saved {rf_path.name}: {len(rf_csv)} OSM ways')
print(rf_csv.passable_mhov.value_counts().to_string())
display(Markdown('### Random Forest + roads result'))
display(rf_roads[['road_id', 'name', 'highway', 'impact_status', 'affected_length_m',
                  'max_affected_run_m', 'coverage_fraction', 'unknown_length_m']]
        .sort_values('affected_length_m', ascending=False).head(20).round(2))

## Interpretation and handoff

* `landslide_by_osm_id_rf.csv`: one row per OSM way, same columns as the road-width `by_osm_id.csv`; this is what routing reads.
* `road_impact_rf.csv`: one row per clipped road piece (parent `full_id`/`osm_id`, clipped `road_id`, affected/observed/unknown lengths).
* Join to routing with the parent IDs and agreed graph segmentation; these clipped components are not a complete routing graph.
* The 10 m output cannot establish remaining vehicle clearance, complete blockage or current passability. No new road-blockage accuracy is claimed.

Sources: [team road repository](https://github.com/Loading-my-brain/x-band), [OpenStreetMap attribution](https://www.openstreetmap.org/copyright), [Sentinel-2 catalog](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_HARMONIZED), [SRTM catalog](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003). The model was trained using [GNS V3 Kaikōura source/debris labels](https://doi.org/10.17603/ds2-rpyg-0h84).

In [ ]:
# GNS reference labels: complete inventory in the agreed AOI, not only road-touching polygons.
from concurrent.futures import ThreadPoolExecutor, as_completed
import geopandas as gpd
import pandas as pd
import json, time
import requests, rasterio
from rasterio.transform import Affine

GNS_ROOT = ROOT/'gns_reference'
GNS_ROOT.mkdir(exist_ok=True)
# GNS_SOURCE_ASSET / GNS_DEBRIS_ASSET: set in the Project inputs cell at the top
gns_source = ee.FeatureCollection(GNS_SOURCE_ASSET).filterBounds(roi)
gns_debris = ee.FeatureCollection(GNS_DEBRIS_ASSET).filterBounds(roi)
print('Checking GNS reference access:',gns_source.size().getInfo(),'source polygons,',
      gns_debris.size().getInfo(),'debris polygons',flush=True)

def download_reference_vectors(asset_id, name):
    path = GNS_ROOT/(name+'.geojson')
    if not path.exists():
        collection = ee.FeatureCollection(asset_id).filterBounds(roi).select([])
        features, token = [], None
        while True:
            params = {'expression':collection,'pageSize':1000}
            if token:
                params['pageToken'] = token
            page = ee.data.computeFeatures(params)
            features.extend(page.get('features',[]))
            token = page.get('nextPageToken')
            print(name,'reference polygons fetched:',len(features),flush=True)
            if not token:
                break
        if not features:
            raise RuntimeError('Empty GNS reference layer: '+asset_id)
        temporary = path.with_suffix('.tmp')
        temporary.write_text(json.dumps({'type':'FeatureCollection','features':features}))
        temporary.replace(path)
    return path

with ThreadPoolExecutor(max_workers=2) as executor:
    reference_paths = list(executor.map(lambda item:download_reference_vectors(*item),
        [(GNS_SOURCE_ASSET,'source'),(GNS_DEBRIS_ASSET,'debris')]))
gns_frames = [gpd.read_file(p).to_crs('EPSG:2193')[['geometry']] for p in reference_paths]
gns_polygons = gpd.GeoDataFrame(pd.concat(gns_frames,ignore_index=True),crs='EPSG:2193')

# Same GNS union target and paint grid used to train/validate the saved models.
gns_label = (ee.Image.constant(0).byte().setDefaultProjection(CRS,GRID)
             .paint(gns_source.merge(gns_debris),1).rename('gns_landslide')
             .clip(roi).unmask(0,False))
label_dir = GNS_ROOT/'label_blocks'
label_dir.mkdir(exist_ok=True)

def download_reference_block(block):
    path = label_dir/(block['block_id']+'.tif')
    if not path.exists():
        for attempt in range(3):
            try:
                url = gns_label.getDownloadURL({'crs':CRS,'crs_transform':block['transform'],
                    'dimensions':[512,512],'format':'GEO_TIFF','filePerBand':False})
                response = requests.get(url,timeout=180)
                response.raise_for_status()
                temporary = path.with_suffix('.tmp')
                temporary.write_bytes(response.content)
                with rasterio.open(temporary) as dataset:
                    assert (dataset.count,dataset.width,dataset.height)==(1,512,512)
                    assert dataset.crs.to_epsg()==2193
                    assert dataset.transform.almost_equals(Affine(*block['transform']))
                temporary.replace(path)
                break
            except Exception:
                if attempt==2:
                    raise
                time.sleep(2*(attempt+1))
    return block['block_id'],str(path)

reference_blocks = {}
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(download_reference_block,b) for b in plan['blocks']]
    for future in as_completed(futures):
        key,path = future.result()
        reference_blocks[key] = path
        print('GNS raster reference ready:',len(reference_blocks),'/',len(plan['blocks']),flush=True)
label_records = [dict(tile,path=reference_blocks[tile['block_id']]) for tile in plan['tiles']]
(GNS_ROOT/'label_manifest.json').write_text(json.dumps(label_records,indent=2))
print('GNS vector and aligned raster references ready for all',len(label_records),'road tiles.')

In [ ]:
# @title Run Random Forest / GNS comparison and show results
# Compare frozen RF predictions with GNS on exactly the same road-selected imagery and reference grid.
# (U-Net removed: road overlaps are compared on the RF's observed coverage.)
import folium
import matplotlib.pyplot as plt

COMPARISON_OUT = ROOT/'comparison_outputs'
pixel_scores = score_prediction_tiles(label_records,
    {'Random Forest':rf_result}, COMPARISON_OUT,
    thresholds={'Random Forest':RF_THRESHOLD},
    evaluation_scope='Road-selected integration pixels; may overlap training/validation; not independent test accuracy')
display(Markdown('## Random Forest and GNS reference comparison\n'
    '**GNS is the mapped landslide reference, not confirmed road blockage truth.** '
    'These road-selected tiles may overlap training/validation, so the following '
    'scores describe integration agreement and are not a new independent test.'))
pixel_table = pd.DataFrame(pixel_scores['models']).T
display(pixel_table[['precision','recall','f1','iou','average_precision','accuracy']].round(4))
print('Shared valid pixels:',pixel_scores['metadata']['shared_valid_pixels'])

# Road overlap: RF and GNS on the same RF-observed road sections
gns_assessment = assess_road_impacts(roads, gns_polygons, rf_result['coverage'])
assessments = {'Random Forest': rf_assessment, 'GNS reference': gns_assessment}
table = rf_roads[['road_id','full_id','osm_id','road_length_m','observed_length_m',
                  'unknown_length_m','coverage_fraction','geometry']].copy()
for name, prefix in [('Random Forest','rf'), ('GNS reference','gns')]:
    layer = assessments[name]['roads'][['road_id','affected_length_m','max_affected_run_m','impact_status']]
    table = table.merge(layer.rename(columns={'affected_length_m':prefix+'_affected_length_m',
        'max_affected_run_m':prefix+'_max_affected_run_m','impact_status':prefix+'_status'}),
        on='road_id', validate='one_to_one')
r_, g_ = table.rf_affected_length_m > 1e-6, table.gns_affected_length_m > 1e-6
table['model_agreement'] = np.select([r_ & g_, r_ & ~g_, ~r_ & g_], ['both','rf_only','gns_only'], default='neither')
table.drop(columns='geometry').to_csv(COMPARISON_OUT/'road_model_agreement.csv', index=False)
length_table = pd.DataFrame({name: {
        'affected_road_parts': int((table[p+'_affected_length_m'] > 1e-6).sum()),
        'affected_length_km': table[p+'_affected_length_m'].sum()/1000,
        'max_continuous_affected_run_m': table[p+'_max_affected_run_m'].max()}
    for name, p in [('Random Forest','rf'), ('GNS reference','gns')]}).T
display(length_table.round(3))
print('Road parts by agreement:', table.model_agreement.value_counts().to_dict())
print('Observed road length (km):', table.observed_length_m.sum()/1000)
print('Unknown road length (km):', table.unknown_length_m.sum()/1000)
print('A larger overlap length is not evidence of a better blockage detector.')

# Static maps: RF and GNS side by side on the same roads and coverage
colors = {'Random Forest':'#8c43d1', 'GNS reference':'#31a354'}
unknown = rf_assessment['unknown_runs']
xmin, ymin, xmax, ymax = roads.total_bounds
pad = max(xmax - xmin, ymax - ymin) * 0.02
fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, sharey=True)
for ax, name in zip(axes, colors):
    a = assessments[name]
    if not a['hazards'].empty:
        a['hazards'].plot(ax=ax, color=colors[name], alpha=0.25)
    roads.plot(ax=ax, color='#84bdc3', linewidth=0.8)
    if not unknown.empty:
        unknown.plot(ax=ax, color='#808080', linewidth=1.5)
    if not a['affected_runs'].empty:
        a['affected_runs'].plot(ax=ax, color=colors[name], linewidth=2)
    ax.set_xlim(xmin - pad, xmax + pad); ax.set_ylim(ymin - pad, ymax + pad)
    ax.set_title(name); ax.set_xlabel('NZTM easting (m)')
    add_map_furniture(ax)
axes[0].set_ylabel('NZTM northing (m)')
fig.suptitle('Same roads and observed coverage: Random Forest versus GNS mapped reference\n'
             'Integration demonstration; no confirmed road closure or new held-out accuracy')
fig.tight_layout()
fig.savefig(COMPARISON_OUT/'rf_gns_maps.png', dpi=180)
plt.show()

# Interactive map
map_view = folium.Map(location=[-42.25, 173.75], zoom_start=10, tiles=None)
folium.TileLayer('https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri, Maxar, Earthstar Geographics and GIS User Community',
    name='Illustrative imagery — dates unverified; not confirmed 2016', overlay=False).add_to(map_view)
folium.GeoJson(table.to_crs(4326).to_json(), name='Road components',
    style_function=lambda feature: {'color': '#65c7d0', 'weight': 2},
    tooltip=folium.GeoJsonTooltip(fields=['road_id','model_agreement','observed_length_m','unknown_length_m',
        'rf_affected_length_m','gns_affected_length_m'])).add_to(map_view)
if not unknown.empty:
    folium.GeoJson(unknown.to_crs(4326).to_json(), name='Unknown / unobserved sections',
        style_function=lambda feature: {'color': '#808080', 'weight': 4}).add_to(map_view)
for name in colors:
    group = folium.FeatureGroup(name=name + ' hazards + affected runs', show=True)
    for key, weight, opacity in [('hazards', 1, 0.18), ('affected_runs', 5, 0)]:
        layer = assessments[name][key]
        if not layer.empty:
            style = {'color': colors[name], 'weight': weight, 'fillOpacity': opacity}
            folium.GeoJson(layer.to_crs(4326).to_json(), style_function=lambda feature, style=style: style).add_to(group)
    group.add_to(map_view)
b = table.to_crs(4326).total_bounds
map_view.fit_bounds([[b[1], b[0]], [b[3], b[2]]])
add_folium_furniture(map_view)
folium.LayerControl(collapsed=False).add_to(map_view)
map_view.save(COMPARISON_OUT/'rf_gns_comparison.html')
map_view.get_root().height = 550
display(map_view)

(COMPARISON_OUT/'rf_model_metadata.json').write_text(json.dumps(RF_METADATA,indent=2))
(COMPARISON_OUT/'comparison_provenance.json').write_text(json.dumps({
    'gns_source_asset':GNS_SOURCE_ASSET, 'gns_debris_asset':GNS_DEBRIS_ASSET,
    'gns_citation':'Jones et al., GNS V3 Kaikoura landslide inventory, DOI:10.17603/ds2-rpyg-0h84',
    'gns_use':'Mapped source/debris union; not confirmed road blockage labels',
    'raster_grid':GRID,'crs':CRS,'rf_threshold':RF_THRESHOLD,
    'road_commit':ROAD_COMMIT,'road_sha256':ROAD_SHA256,
    'rf_parity_max_probability_error':rf_parity_error,
    'evaluation_scope':'Road-selected integration demonstration, not independent held-out accuracy',
},indent=2))
shutil.copy(ROOT/'imagery_config.json',COMPARISON_OUT/'imagery_config.json')
shutil.copy(ROOT/'parity_check.json',COMPARISON_OUT/'parity_check.json')
shutil.copy(RF_OUT/'inference_config.json',COMPARISON_OUT/'rf_inference_config.json')
shutil.copy(RF_OUT/'inference_manifest.json',COMPARISON_OUT/'rf_inference_manifest.json')
shutil.copy(rf_path, COMPARISON_OUT/rf_path.name)
comparison_bundle = shutil.make_archive(str(ROOT/'kaikoura_rf_gns_comparison'),'zip',COMPARISON_OUT)
print('Comparison finished. Tables, maps, reference overlaps and metrics:',comparison_bundle)
files.download(comparison_bundle)

---
# Part 3 – Fastest passable route per vehicle (A*)

From `Vehicle_Routing_Passability`: Phase 3 (Kaikōura → Hapuku) and case 3 (Culverden → Kaikōura).
The Phase 1 demo route has been removed; Phase 3 builds the road graph itself.

The hand-off cell below copies the two CSVs from Parts 1 and 2 into Colab's working folder and downloads the Kaikōura AOI shapefile from GitHub.

In [ ]:
# Hand-off: routing reads its inputs from the working folder, as in the standalone routing notebook.
import shutil, requests
from pathlib import Path

WORK = Path.cwd()
shutil.copy(Path('/content/xband_roads/handoff/by_osm_id.csv'), WORK/'by_osm_id.csv')               # Part 1
shutil.copy(Path('/content/kaikoura_unet_roads/outputs/landslide_by_osm_id_rf.csv'),
            WORK/'landslide_by_osm_id_rf.csv')                                                    # Part 2
AOI_BASE = 'https://raw.githubusercontent.com/Loading-my-brain/x-band/main/kaikoura_road_route/Kaikoura'
for ext in ('shp', 'shx', 'dbf', 'prj', 'cpg'):
    path = WORK/f'Kaikoura.{ext}'
    if not path.exists():
        response = requests.get(f'{AOI_BASE}.{ext}', timeout=120)
        response.raise_for_status()
        path.write_bytes(response.content)
print('Routing inputs ready in', WORK, ':', sorted(p.name for p in WORK.glob('*.csv')), '+ Kaikoura.shp')

Notes on the Vehicle Routing module:

* Routes could be initiated from quite a ways away from the event point. Example: our initial AOI doesn't contain the entirety of the routes taken by Emergency Responders during the Kaikoura incident. Solution: expand our AOI.
* This routing mechanism would be given the starting point and goal point. Determining the starting point based on present conditions is currently out of scope.
* Google Maps data over time can be used to determine roads that have encountered a sudden drop in traffic, possibly indicating a roadblock / passage hazard. This would be in consideration for the identification module, but could be used as training data for the routing module. Other sources are TomTom Traffic Flow, HERE Traffic, and NZTA traffic monitoring system. Caveats: if a cell tower goes down, the data can vanish when we need it. Also slow traffic, time of day (midnight) doesn't mean blocked traffic. And the road queue points to the actual blockage - it isn't where the blockage is.
* If we were to determine off-road capabilities, we would want to take into account slope from elevation, land cover, water, and existing tracks.
* Who owns the blocked roads map - routing or identification? How is that data formatted?
* How to include speeds for determining the best route forward? A longer path with higher speeds may provide a faster time than a shorter path with slower roads.

### Note for Craig: Work on getting off-road maps with speed indicators added to the map

In [ ]:
"""
Phase 1: A* on OSM drive network for Kaikoura AOI.
Requires: osmnx>=2.0, networkx, geopandas>=1.0, pyproj, scipy, matplotlib
"""
import heapq
import math

import geopandas as gpd
import matplotlib.pyplot as plt
import networkx as nx
import osmnx as ox
from pyproj import Transformer

# ---------------- CONFIG ----------------
AOI_SHP = "Kaikoura.shp"
BUFFER_M = 5000                  # buffer so routes can leave the AOI edge
CRS_NZTM = "EPSG:2193"
START_LONLAT = ROUTE_START_LONLAT   # from Project inputs
END_LONLAT = ROUTE_END_LONLAT       # from Project inputs
OUT_PNG = "astar_route.png"
OUT_GPKG = "astar_route.gpkg"
CF = (
    '["highway"~"motorway|trunk|primary|secondary|tertiary|unclassified|residential|service|track"]'
    '["area"!~"yes"]["access"!~"private|no"]'
)
# ----------------------------------------

In [ ]:
def load_aoi(path, buffer_m):
    """
    Load the AOI shapefile and prepare the polygon used to download OSM data.

    Steps:
      1. Read the shapefile and reproject to NZTM (metres).
      2. Dissolve all features into one geometry.
      3. Buffer by `buffer_m` metres (so routes can use roads just outside the AOI).
      4. Reproject the buffered polygon to WGS84 (OSMnx requires lat/lon input).

    Args:
        path: path to the AOI shapefile.
        buffer_m: buffer distance in metres.

    Returns:
        aoi: GeoDataFrame of the original (unbuffered) AOI in NZTM, used for plotting.
        poly_wgs84: shapely polygon of the buffered AOI in EPSG:4326, used for the OSM query.
    """
    aoi = gpd.read_file(path)
    if aoi.crs is None:
      aoi = aoi.set_crs("EPSG:4326")   # assumption: shapefile is already NZTM
    aoi = aoi.to_crs(CRS_NZTM)
    aoi_geom = aoi.geometry.union_all()
    buffered = gpd.GeoSeries([aoi_geom.buffer(buffer_m)], crs=CRS_NZTM)
    poly_wgs84 = buffered.to_crs("EPSG:4326").iloc[0]
    return aoi, poly_wgs84


def build_graph(poly_wgs84):
    """
    Download the OSM drivable road network inside the polygon and prepare it for routing.

    Steps:
      1. Download the drive network (nodes = intersections, edges = road segments).
      2. Add `speed_kph` to each edge (OSM maxspeed, else a default per highway class).
      3. Add `travel_time` (seconds) = length / speed. This is the A* edge cost.
      4. Keep only the largest strongly connected component, so every node can
         reach every other node (avoids dead-end pockets that break routing).
      5. Reproject the graph to NZTM so node x/y are in metres (needed for the heuristic).

    Args:
        poly_wgs84: shapely polygon in EPSG:4326.

    Returns:
        Projected networkx MultiDiGraph with `length`, `speed_kph`, `travel_time` on edges.
    """
    ox.settings.useful_tags_way += ["tracktype", "surface", "smoothness", "4wd_only", "width", "access", "ford", "bridge", "maxweight"]
    G = ox.graph_from_polygon(poly_wgs84, custom_filter=CF, simplify=True)
    G = ox.routing.add_edge_speeds(G)          # speed_kph (maxspeed or highway-class default)
    G = ox.routing.add_edge_travel_times(G)    # travel_time (s)
    G = ox.truncate.largest_component(G, strongly=True)
    return ox.project_graph(G, to_crs=CRS_NZTM)


def make_heuristic(G, goal):
    """
    Build the A* heuristic function h(n) = estimated remaining travel time to `goal`.

    h(n) = straight-line distance from n to goal / fastest speed on the network.
    This never overestimates the true remaining time (admissible) because no road
    is shorter than a straight line and no vehicle is faster than the max speed.
    It also satisfies the triangle inequality (consistent), so a node is never
    needed to be re-expanded once closed.

    Args:
        G: projected graph (node x/y in metres, edges with `speed_kph`).
        goal: node id of the destination.

    Returns:
        Function h(node_id) -> seconds.
    """
    vmax = max(d["speed_kph"] for _, _, d in G.edges(data=True)) / 3.6  # m/s
    gx, gy = G.nodes[goal]["x"], G.nodes[goal]["y"]

    def h(n):
        """Straight-line distance from node n to the goal, converted to seconds at max speed."""
        return math.hypot(G.nodes[n]["x"] - gx, G.nodes[n]["y"] - gy) / vmax

    return h


def astar(G, src, dst, weight="travel_time"):
    """
    A* shortest path from `src` to `dst`.

    Priority of each node in the open set: f(n) = g(n) + h(n), where
      g(n) = cheapest known cost from src to n,
      h(n) = heuristic estimate from n to dst.
    The lowest-f node is expanded next (via a min-heap). When `dst` is popped,
    the path is optimal. Parallel edges between two nodes: the cheapest is used.

    Args:
        G: projected networkx MultiDiGraph.
        src: start node id.
        dst: end node id.
        weight: edge attribute used as cost (default travel time in seconds).

    Returns:
        path: list of node ids from src to dst.
        cost: total cost of the path (seconds if weight="travel_time").
        explored: node ids in the order they were expanded (for visualising the search).

    Raises:
        networkx.NetworkXNoPath: if dst is unreachable from src.
    """
    h = make_heuristic(G, dst)
    g = {src: 0.0}          # best known cost to each node
    parent = {}             # predecessor on best known path
    closed = set()          # nodes already expanded
    explored = []           # expansion order
    pq = [(h(src), 0.0, src)]  # (f, g, node)
    while pq:
        _, gc, u = heapq.heappop(pq)
        if u in closed:     # stale heap entry; a cheaper one was already processed
            continue
        closed.add(u)
        explored.append(u)
        if u == dst:        # goal reached: rebuild path by walking parents back to src
            path = [u]
            while path[-1] != src:
                path.append(parent[path[-1]])
            return path[::-1], gc, explored
        for v, edges in G[u].items():
            w = min(d.get(weight, math.inf) for d in edges.values())
            ng = gc + w
            if ng < g.get(v, math.inf):   # found a cheaper way to reach v
                g[v] = ng
                parent[v] = u
                heapq.heappush(pq, (ng + h(v), ng, v))
    raise nx.NetworkXNoPath(f"No path between {src} and {dst}")


def snap(G, lonlat):
    """
    Snap a lon/lat coordinate to the nearest graph node.

    Converts the WGS84 point to NZTM, then finds the closest node in the
    projected graph.

    Args:
        G: projected graph.
        lonlat: (lon, lat) tuple in EPSG:4326.

    Returns:
        node: nearest node id.
        dist: distance in metres from the input point to that node
              (large values mean the point is far from any road).
    """
    x, y = Transformer.from_crs("EPSG:4326", CRS_NZTM, always_xy=True).transform(*lonlat)
    node, dist = ox.distance.nearest_nodes(G, x, y, return_dist=True)
    return node, dist


def plot_result(G, aoi, route, explored, src, dst, out_png):
    """
    Plot the road network, AOI boundary, A* search footprint, and final route, then save as PNG.

    Layers: grey = road network, black line = AOI boundary, orange dots = nodes
    A* expanded (shows how focused the search was), red line = final route,
    green/blue dots = start/end.

    Args:
        G: projected graph.
        aoi: GeoDataFrame of the AOI (NZTM).
        route: list of node ids on the final path.
        explored: node ids expanded by A*, in order.
        src, dst: start and end node ids.
        out_png: output image path.
    """
    fig, ax = ox.plot_graph(
        G, node_size=0, edge_color="#bbbbbb", edge_linewidth=0.5,
        show=False, close=False, bgcolor="white",
    )
    aoi.boundary.plot(ax=ax, color="black", linewidth=1)
    ax.scatter(
        [G.nodes[n]["x"] for n in explored],
        [G.nodes[n]["y"] for n in explored],
        s=3, c="orange", label=f"explored ({len(explored)})", zorder=2,
    )
    ax.plot(
        [G.nodes[n]["x"] for n in route],
        [G.nodes[n]["y"] for n in route],
        c="red", linewidth=2, label="A* route", zorder=3,
    )
    for n, lbl, c in ((src, "start", "green"), (dst, "end", "blue")):
        ax.scatter(G.nodes[n]["x"], G.nodes[n]["y"], s=60, c=c, label=lbl, zorder=4)
    add_map_furniture(ax)
    ax.legend(loc="lower right")
    fig.savefig(out_png, dpi=200, bbox_inches="tight")

## Phase 3 - Passability-aware routing (by\_osm\_id.csv)

Added cells; Phase 1 above is unchanged. No off-road routing.

* **Input:** `by_osm_id.csv` (upstream, final): one row per OSM way with `passable_pinzgauer` / `passable_mhov` = yes / no / unknown and a reason. Width, pinch and share columns are reported only, never used to re-decide passability.
* **Join:** edge `osmid` → CSV `osm_id`. A simplified edge holding several ways takes the worst status (no > unknown > unverified > yes).
* **Cost:** `travel_time` + fixed delay per unknown way (5 min bridge/tunnel, 2 min unmeasured) + a very large penalty per "no" way. One A\* per vehicle then gives the fastest route that avoids every "no" way, or, if none exists, the next best route crossing the fewest "no" ways.
* **Status:** CLEAR, CLEAR\_WITH\_UNKNOWN or BLOCKED, with a blockage table (location, reason, km along the route) for every "no" and "unknown" way on the route.
* **Limitation:** roads not in the CSV are treated as passable and reported as unverified km.
* **Compute:** graph cached to GraphML after the first download; reuses `G` from Phase 1 if it is already in memory.

In [ ]:
### Phase 3 setup - passability-aware routing from by_osm_id.csv
"""
Joins the upstream passability CSV (one row per OSM way, final yes / no / unknown per vehicle)
onto the Phase 1 road graph and routes with the unchanged astar().

Edge cost per vehicle (seconds):
    cost_<veh> = travel_time + unknown delay + PEN_NO_S per "no" way
  unknown delay = fixed time per unknown way, by reason (PEN_UNKNOWN_S)
  PEN_NO_S      = so large that the route always uses the fewest "no" ways, then the fastest
A way split across several graph edges has its penalty shared between them, so it is charged once per pass.
cost >= travel_time, so astar()'s heuristic stays admissible.

Route status:
  CLEAR               - no "no" or "unknown" ways on the route
  CLEAR_WITH_UNKNOWN  - no "no" ways, but some "unknown" ways (penalised, listed in the blockage table)
  BLOCKED             - no route avoids every "no" way; this is the next best route and its "no" ways are listed
Roads not in the CSV are treated as passable and reported as "unverified" (key limitation: the CSV
only covers the surveyed corridor).
"""
import ast
import os

import pandas as pd
from shapely.geometry import LineString

PASS_CSV = "by_osm_id.csv"
GRAPH_CACHE = "kaikoura_drive.graphml"    # delete this file if the AOI or CF changes
PASS_VEHICLES = tuple(VEHICLE_WIDTHS)     # from Project inputs
PEN_UNKNOWN_S = {
    "bridge/tunnel: lidar cannot see the deck": 300,
    "too little of the road could be measured": 120,
}
PEN_UNKNOWN_DEFAULT_S = 120
PEN_NO_S = 1e6
VALIDATE = True                           # check each A* cost against NetworkX Dijkstra

PASS_START_LONLAT = ROUTE_START_LONLAT    # from Project inputs (default: Kaikoura township)
PASS_END_LONLAT = ROUTE_END_LONLAT        # from Project inputs (default: Hapuku, north on SH1)
OUT_PASS_PNG = "pass_routes.png"
OUT_PASS_GPKG = "pass_routes.gpkg"
OUT_PASS_CSV = "pass_blockages.csv"

STATUS_RANK = {"yes": 0, "unverified": 1, "unknown": 2, "no": 3}
BLOCK_COLS = ["vehicle", "osm_id", "name", "highway", "status", "reason", "is_bridge", "is_tunnel",
              "length_m", "width_median_m", "drive_pinch_m", "km_from", "km_to", "lon", "lat",
              "x_nztm", "y_nztm"]


def load_or_build_graph(poly_wgs84, path=GRAPH_CACHE):
    """Phase 1 graph from the GraphML cache if present, else build_graph() once and cache it."""
    if os.path.exists(path):
        return ox.load_graphml(path)
    G = build_graph(poly_wgs84)
    ox.save_graphml(G, path)
    return G


def way_ids(osmid):
    """OSM way ids on an edge (simplified edges can hold a list; GraphML can hold it as a string)."""
    if isinstance(osmid, str):
        osmid = ast.literal_eval(osmid) if osmid.startswith("[") else int(osmid)
    if not isinstance(osmid, (list, tuple, set)):
        osmid = [osmid]
    return [int(w) for w in osmid]


def load_passability(path=PASS_CSV, vehicles=PASS_VEHICLES):
    """CSV as {osm_id: row dict}; checks ids are unique and statuses are yes / no / unknown."""
    df = pd.read_csv(path)
    if df["osm_id"].duplicated().any():
        raise ValueError(f"duplicate osm_id in {path}")
    for v in vehicles:
        col = f"passable_{v}"
        df[col] = df[col].astype(str).str.strip().str.lower()
        bad = set(df[col]) - {"yes", "no", "unknown"}
        if bad:
            raise ValueError(f"{col} has unexpected values {bad}")
        df[f"reason_{v}"] = df[f"reason_{v}"].where(df[f"reason_{v}"].notna(), None)
    return df.set_index("osm_id").to_dict("index")


def apply_passability(G, rec, vehicles=PASS_VEHICLES):
    """
    Write status_<veh>, pen_<veh> (unknown delay, s) and cost_<veh> (s) on every edge, in place.

    Returns:
        dict with CSV match stats and, per vehicle, edge counts by status.
    """
    seg = {}
    for u, v, d in G.edges(data=True):
        for w in way_ids(d.get("osmid")):
            seg.setdefault(w, set()).add(frozenset((u, v)))
    n_seg = {w: len(s) for w, s in seg.items()}

    counts = {veh: dict.fromkeys(STATUS_RANK, 0) for veh in vehicles}
    for _, _, d in G.edges(data=True):
        ways = way_ids(d.get("osmid"))
        for veh in vehicles:
            status, pen_u, pen_n = "yes", 0.0, 0.0
            for w in ways:
                r = rec.get(w)
                s = "unverified" if r is None else r[f"passable_{veh}"]
                if s == "unknown":
                    pen_u += PEN_UNKNOWN_S.get(r[f"reason_{veh}"], PEN_UNKNOWN_DEFAULT_S) / n_seg[w]
                elif s == "no":
                    pen_n += PEN_NO_S / n_seg[w]
                if STATUS_RANK[s] > STATUS_RANK[status]:
                    status = s
            d[f"status_{veh}"] = status
            d[f"pen_{veh}"] = pen_u
            d[f"cost_{veh}"] = d["travel_time"] + pen_u + pen_n
            counts[veh][status] += 1
    matched = set(rec) & set(seg)
    return {"csv_ways": len(rec), "matched": len(matched), "missing": sorted(set(rec) - matched),
            "edges": G.number_of_edges(), "counts": counts}


def route_blockages(rgdf, rec, veh):
    """One row per "no" / "unknown" CSV way on the route, "no" first, then in route order."""
    km_to = rgdf["length"].cumsum().to_numpy() / 1000
    km_from = km_to - rgdf["length"].to_numpy() / 1000
    mid = gpd.GeoSeries(rgdf.geometry.interpolate(0.5, normalized=True).values, crs=rgdf.crs)
    mid_ll = mid.to_crs("EPSG:4326")
    rows = []
    for i, (osmid, status) in enumerate(zip(rgdf["osmid"], rgdf[f"status_{veh}"])):
        if status not in ("no", "unknown"):
            continue
        for w in way_ids(osmid):
            r = rec.get(w)
            if r is None or r[f"passable_{veh}"] not in ("no", "unknown"):
                continue
            rows.append({
                "vehicle": veh, "osm_id": w, "name": r["name"], "highway": r["highway"],
                "status": r[f"passable_{veh}"], "reason": r[f"reason_{veh}"],
                "is_bridge": r["is_bridge"], "is_tunnel": r["is_tunnel"], "length_m": r["length_m"],
                "width_median_m": r["width_median_m"], "drive_pinch_m": r["drive_pinch_m"],
                "km_from": round(km_from[i], 2), "km_to": round(km_to[i], 2),
                "lon": round(mid_ll.iloc[i].x, 6), "lat": round(mid_ll.iloc[i].y, 6),
                "x_nztm": mid.iloc[i].x, "y_nztm": mid.iloc[i].y,
            })
    df = pd.DataFrame(rows, columns=BLOCK_COLS).drop_duplicates("osm_id")
    order = df["status"].map({"no": 0, "unknown": 1})
    return df.assign(_o=order).sort_values(["_o", "km_from"]).drop(columns="_o").reset_index(drop=True)


def summarise_route(rgdf, blk, veh):
    """Route status and blockage metrics."""
    n_no = int((blk["status"] == "no").sum())
    n_unk = int((blk["status"] == "unknown").sum())
    status = "BLOCKED" if n_no else ("CLEAR_WITH_UNKNOWN" if n_unk else "CLEAR")
    unver = rgdf[f"status_{veh}"] == "unverified"
    km = rgdf["length"].sum() / 1000
    return {
        "status": status, "km": km, "drive_min": rgdf["travel_time"].sum() / 60,
        "unknown_delay_min": rgdf[f"pen_{veh}"].sum() / 60,
        "n_no": n_no, "n_unknown": n_unk,
        "km_no": blk.loc[blk["status"] == "no", "length_m"].sum() / 1000,
        "km_unknown": blk.loc[blk["status"] == "unknown", "length_m"].sum() / 1000,
        "km_unverified": rgdf.loc[unver, "length"].sum() / 1000,
    }


def pass_route(G, src, dst, rec, veh):
    """A* on cost_<veh>; returns dict with route, route_gdf, blockages, summary and explored nodes."""
    w = f"cost_{veh}"
    route, cost, explored = astar(G, src, dst, weight=w)
    if VALIDATE:
        ref = nx.shortest_path_length(G, src, dst, weight=w)
        assert math.isclose(cost, ref, rel_tol=1e-9), (veh, cost, ref)
    rgdf = ox.routing.route_to_gdf(G, route, weight=w)
    blk = route_blockages(rgdf, rec, veh)
    return {"route": route, "route_gdf": rgdf, "blockages": blk, "explored": explored,
            "summary": summarise_route(rgdf, blk, veh)}


def flagged_ways(G, rec, veh, status="no"):
    """Where each CSV way with `status` sits in the graph (lon/lat of one edge midpoint) - for picking test points."""
    rows, seen = [], set()
    to_ll = Transformer.from_crs(CRS_NZTM, "EPSG:4326", always_xy=True)
    for u, v, d in G.edges(data=True):
        for w in way_ids(d.get("osmid")):
            r = rec.get(w)
            if w in seen or r is None or r[f"passable_{veh}"] != status:
                continue
            seen.add(w)
            geom = d.get("geometry") or LineString([(G.nodes[u]["x"], G.nodes[u]["y"]), (G.nodes[v]["x"], G.nodes[v]["y"])])
            p = geom.interpolate(0.5, normalized=True)
            lon, lat = to_ll.transform(p.x, p.y)
            rows.append({"osm_id": w, "name": r["name"], "reason": r[f"reason_{veh}"],
                         "length_m": r["length_m"], "lon": round(lon, 5), "lat": round(lat, 5)})
    return pd.DataFrame(rows, columns=["osm_id", "name", "reason", "length_m", "lon", "lat"])


def gpkg_safe(gdf, cols):
    """Keep `cols` that exist; list-valued OSM attributes become strings so GeoPackage can store them."""
    out = gdf[[c for c in cols if c in gdf.columns]].copy()
    for c in out.columns:
        if c != "geometry" and out[c].dtype == object:
            out[c] = out[c].map(lambda x: str(x) if isinstance(x, (list, tuple, set)) else x)
    return out


def print_pass_result(veh, res, base):
    """Console report for one vehicle."""
    s = res["summary"]
    print(f"\n=== {veh}: {s['status']} ===")
    print(f"Route {s['km']:.2f} km | driving {s['drive_min']:.1f} min + unknown delay "
          f"{s['unknown_delay_min']:.1f} min = {s['drive_min'] + s['unknown_delay_min']:.1f} min")
    print(f"'no' ways: {s['n_no']} ({s['km_no']:.2f} km) | 'unknown' ways: {s['n_unknown']} "
          f"({s['km_unknown']:.2f} km) | unverified (not in CSV): {s['km_unverified']:.2f} km "
          f"of {s['km']:.2f} km")
    print(f"Fastest route ignoring the CSV: {base['km']:.2f} km, {base['drive_min']:.1f} min, "
          f"{base['n_no']} 'no' / {base['n_unknown']} 'unknown' ways for {veh}")
    if s["status"] == "BLOCKED":
        print("No route avoids every 'no' way. Next best route shown; clearing the 'no' ways below opens it.")
    blk = res["blockages"]
    if len(blk):
        cols = [c for c in ["concern", "status", "osm_id", "name", "reason", "length_m", "drive_pinch_m",
                            "km_from", "km_to", "lon", "lat"] if c in blk.columns]
        print(blk[cols].to_string(index=False))


def plot_pass_routes(G, aoi, results, base_gdf, src, dst, out_png, pad_m=2000):
    """One panel per vehicle: network status, fastest-ignoring-CSV route, chosen route, blockages."""
    edges = ox.graph_to_gdfs(G, nodes=False)
    allg = pd.concat([base_gdf.geometry] + [r["route_gdf"].geometry for r in results.values()])
    xmin, ymin, xmax, ymax = allg.total_bounds
    fig, axes = plt.subplots(1, len(results), figsize=(8 * len(results), 8), squeeze=False)
    for ax, (veh, r) in zip(axes[0], results.items()):
        edges.plot(ax=ax, color="#cccccc", linewidth=0.5, zorder=1)
        for s, c in (("unknown", "orange"), ("no", "red")):
            sub = edges[edges[f"status_{veh}"] == s]
            if len(sub):
                sub.plot(ax=ax, color=c, linewidth=1.2, zorder=2, label=f"network: {s}")
        aoi.boundary.plot(ax=ax, color="black", linewidth=1, zorder=1)
        base_gdf.plot(ax=ax, color="black", linewidth=1.2, linestyle="--", zorder=3, label="fastest (ignores CSV)")
        r["route_gdf"].plot(ax=ax, color="blue", linewidth=2.5, alpha=0.8, zorder=4,
                            label=f"chosen route ({r['summary']['status']})")
        b = r["blockages"]
        concern = b["concern"] if "concern" in b.columns else pd.Series("Width", index=b.index)
        for k, marks in (("Width", {"unknown": "o", "no": "X"}), ("Landslide risk", {"unknown": "^", "no": "v"})):
            for s, c in (("unknown", "orange"), ("no", "red")):
                sb = b[(b["status"] == s) & (concern == k)]
                if len(sb):
                    ax.scatter(sb["x_nztm"], sb["y_nztm"], marker=marks[s], s=90, c=c, edgecolors="black",
                               zorder=5, label=f"{k}: {s} ({len(sb)})")
        for n, lbl, c in ((src, "start", "green"), (dst, "end", "blue")):
            ax.scatter(G.nodes[n]["x"], G.nodes[n]["y"], s=80, c=c, edgecolors="black", zorder=6, label=lbl)
        ax.set_xlim(xmin - pad_m, xmax + pad_m)
        ax.set_ylim(ymin - pad_m, ymax + pad_m)
        ax.set_aspect("equal")
        s = r["summary"]
        ax.set_title(f"{veh}: {s['status']} | {s['km']:.1f} km, "
                     f"{s['drive_min'] + s['unknown_delay_min']:.0f} min incl. unknown delay")
        add_map_furniture(ax)
        ax.legend(loc="lower right", fontsize=8)
    fig.savefig(out_png, dpi=200, bbox_inches="tight")
    return fig

In [ ]:
### Phase 3 self-test - small synthetic network (no downloads)
"""
Ways: 101 = unknown bridge split over two edges (0-4, 4-1); 102 (1-2) yes for pinzgauer, no for mhov;
103 (0-3) yes; 104 (3-2) not in the CSV (unverified). Checks the unknown delay is charged once per way,
"no" ways are avoided when a detour exists, and the fewest-"no" route is returned as BLOCKED otherwise.
"""
import tempfile


def _p3_selftest():
    rows = [
        (101, "Bridge Rd", "trunk", True, "unknown", "bridge/tunnel: lidar cannot see the deck",
         "unknown", "bridge/tunnel: lidar cannot see the deck"),
        (102, "SH1", "trunk", False, "yes", None, "no", "sustained pinch narrower than vehicle + clearance"),
        (103, "Detour Rd", "unclassified", False, "yes", None, "yes", None),
        (104, "Back Rd", "unclassified", False, "yes", None, "yes", None),
    ]
    df = pd.DataFrame(rows, columns=["osm_id", "name", "highway", "is_bridge", "passable_pinzgauer",
                                     "reason_pinzgauer", "passable_mhov", "reason_mhov"])
    df = df.assign(is_tunnel=False, length_m=100.0, width_median_m=None, drive_pinch_m=None)

    def make(detour_s, csv_rows):
        T = nx.MultiDiGraph(crs=CRS_NZTM)
        for n, (x, y) in {0: (0, 0), 4: (500, 0), 1: (1000, 0), 2: (2000, 0), 3: (1000, 1500)}.items():
            T.add_node(n, x=x, y=y)
        for a, b, w, t, L in ((0, 4, 101, 18, 500), (4, 1, 101, 18, 500), (1, 2, 102, 36, 1000),
                              (0, 3, 103, detour_s, 1800), (3, 2, 104, detour_s, 1800)):
            for u, v in ((a, b), (b, a)):
                T.add_edge(u, v, osmid=w, length=L, travel_time=t, speed_kph=100, highway="trunk")
        with tempfile.TemporaryDirectory() as tmp:
            p = os.path.join(tmp, "t.csv")
            csv_rows.to_csv(p, index=False)
            rec = load_passability(p)
        apply_passability(T, rec)
        return T, rec

    def run(T, rec, veh):
        return pass_route(T, 0, 2, rec, veh)

    ##### 1: unknown bridge (+5 min) loses to a 3 min unverified detour
    T, rec = make(90, df[df.osm_id != 104])
    r = run(T, rec, "pinzgauer")
    assert r["route"] == [0, 3, 2] and r["summary"]["status"] == "CLEAR", r["route"]
    assert math.isclose(r["summary"]["km_unverified"], 1.8)

    ##### 2: slow detour - pinzgauer takes the bridge once (5 min delay), mhov avoids its 'no'
    T, rec = make(300, df[df.osm_id != 104])
    r = run(T, rec, "pinzgauer")
    assert r["route"] == [0, 4, 1, 2] and r["summary"]["status"] == "CLEAR_WITH_UNKNOWN"
    assert math.isclose(r["summary"]["unknown_delay_min"], 5.0) and len(r["blockages"]) == 1
    r = run(T, rec, "mhov")
    assert r["route"] == [0, 3, 2] and r["summary"]["status"] == "CLEAR", r["route"]

    ##### 3: detour also 'no' for mhov - fastest of the one-'no' routes, BLOCKED, 'no' listed first
    df3 = df.copy()
    df3.loc[df3.osm_id == 104, ["passable_mhov", "reason_mhov"]] = ["no", "sustained pinch narrower than vehicle + clearance"]
    T, rec = make(300, df3)
    r = run(T, rec, "mhov")
    b = r["blockages"]
    assert r["route"] == [0, 4, 1, 2] and r["summary"]["status"] == "BLOCKED"
    assert list(b["osm_id"]) == [102, 101] and list(b["status"]) == ["no", "unknown"], b
    print("Phase 3 self-test passed: unknown charged once per way, 'no' avoided when possible, "
          "BLOCKED fallback lists 'no' ways first")


_p3_selftest()

In [ ]:
### Phase 4 setup - landslide penalties (runs straight after "Phase 3 setup")
"""
Adds the landslide CSV on top of the road-width CSV. No other cell needs editing:
apply_passability() and pass_route() are wrapped, so Phase 3 run, case 2 and case 3 all pick it up.

    cost_<veh> = travel_time + width unknown delay + landslide delay + PEN_NO_S per "no" way

Penalties are >= 0, so astar() and its heuristic are unchanged and still exact.
Ways not in the landslide CSV (outside the study box) get no landslide cost.
Set USE_LANDSLIDE = False in Project inputs and re-run to compare routes without it.
"""
SLIDE_CSV = "landslide_by_osm_id_rf.csv"     # Random Forest; "landslide_by_osm_id.csv" = U-Net
# USE_LANDSLIDE and SLIDE_DELAY_MIN: set in the Project inputs cell at the top
PEN_UNKNOWN_S.update({reason: 60 * minutes for reason, minutes in SLIDE_DELAY_MIN.items()})
rec_slide = load_passability(SLIDE_CSV)      # same loader and checks as by_osm_id.csv


def apply_landslide(G, rec_slide, vehicles=PASS_VEHICLES):
    """Add landslide delay to an edge set that apply_passability() has just reset."""
    seg = {}
    for u, v, d in G.edges(data=True):
        for w in way_ids(d.get("osmid")):
            seg.setdefault(w, set()).add(frozenset((u, v)))
    n_seg = {w: len(s) for w, s in seg.items()}
    for _, _, d in G.edges(data=True):
        ways = way_ids(d.get("osmid"))
        for veh in vehicles:
            status, pen_u, pen_n = "yes", 0.0, 0.0
            for w in ways:
                r = rec_slide.get(w)
                if r is None:
                    continue
                s = r[f"passable_{veh}"]
                if s == "unknown":
                    pen_u += PEN_UNKNOWN_S.get(r[f"reason_{veh}"], PEN_UNKNOWN_DEFAULT_S) / n_seg[w]
                elif s == "no":
                    pen_n += PEN_NO_S / n_seg[w]
                if STATUS_RANK[s] > STATUS_RANK[status]:
                    status = s
            d[f"slide_status_{veh}"] = status
            d[f"slide_pen_{veh}"] = pen_u
            d[f"pen_{veh}"] += pen_u                 # printed "unknown delay" includes landslides
            d[f"cost_{veh}"] += pen_u + pen_n
            if STATUS_RANK[status] > STATUS_RANK[d[f"status_{veh}"]]:
                d[f"status_{veh}"] = status          # worst of width and landslide
    matched = set(rec_slide) & set(seg)
    return {"csv_ways": len(rec_slide), "matched": len(matched), "missing": sorted(set(rec_slide) - matched)}


# Wrap the Phase 3 functions once (the guard stops a re-run of this cell from double-counting).
if "_apply_passability_width" not in globals():
    _apply_passability_width, _pass_route_width = apply_passability, pass_route


def apply_passability(G, rec, vehicles=PASS_VEHICLES):
    """Width costs (Phase 3), then landslide costs on top when USE_LANDSLIDE is True."""
    info = _apply_passability_width(G, rec, vehicles)
    if USE_LANDSLIDE:
        info["landslide"] = apply_landslide(G, rec_slide, vehicles)
        s = info["landslide"]
        print(f"Landslide CSV ({SLIDE_CSV}): {s['csv_ways']} ways, found in graph: {s['matched']}")
    return info


def pass_route(G, src, dst, rec, veh):
    """Phase 3 route, with landslide ways added to the blockage table and summary.
    Every blockage row is labelled concern = "Width" or "Landslide risk"."""
    res = _pass_route_width(G, src, dst, rec, veh)
    res["blockages"] = res["blockages"].assign(concern="Width")
    if USE_LANDSLIDE:
        slide_blk = route_blockages(res["route_gdf"], rec_slide, veh).assign(concern="Landslide risk")
        blk = pd.concat([res["blockages"], slide_blk], ignore_index=True)
        order = blk["status"].map({"no": 0, "unknown": 1})
        res["blockages"] = blk.assign(_o=order).sort_values(["_o", "km_from"]) \
                              .drop(columns="_o").reset_index(drop=True)
        res["summary"] = summarise_route(res["route_gdf"], res["blockages"], veh)
    return res


print(f"Landslide layer ready: {SLIDE_CSV} | USE_LANDSLIDE = {USE_LANDSLIDE}")

In [ ]:
### Phase 3 run - fastest passable route per vehicle, with blockages
"""
Graph: reuses G from the Phase 1 run cell if it exists, else loads the GraphML cache, else builds it once.
One A* per vehicle on cost_<veh>, plus one A* on travel_time for comparison.
Outputs: pass_routes.png, pass_routes.gpkg (route_<veh>, blockages_<veh>, fastest_ignore_csv), pass_blockages.csv.
"""
if "G" not in globals():
    aoi, poly = load_aoi(AOI_SHP, BUFFER_M)
    G = load_or_build_graph(poly)
elif not os.path.exists(GRAPH_CACHE):
    ox.save_graphml(G, GRAPH_CACHE)
print(f"Graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

rec = load_passability(PASS_CSV)
info = apply_passability(G, rec)
print(f"CSV ways: {info['csv_ways']}, found in graph: {info['matched']}, not in graph: {len(info['missing'])}")
for veh, c in info["counts"].items():
    print(f"  {veh} edges: " + ", ".join(f"{k} {n}" for k, n in c.items()))

src, d_src = snap(G, PASS_START_LONLAT)
dst, d_dst = snap(G, PASS_END_LONLAT)
print(f"Snap distance (m): start={d_src:.0f}, end={d_dst:.0f}")
if max(d_src, d_dst) > 500:
    print("Warning: a point is more than 500 m from the nearest road node")

base_route, _, _ = astar(G, src, dst, weight="travel_time")
base_gdf = ox.routing.route_to_gdf(G, base_route, weight="travel_time")

pass_results, base_summary = {}, {}
for veh in PASS_VEHICLES:
    pass_results[veh] = pass_route(G, src, dst, rec, veh)
    base_summary[veh] = summarise_route(base_gdf, route_blockages(base_gdf, rec, veh), veh)
    print_pass_result(veh, pass_results[veh], base_summary[veh])

blk_all = pd.concat([r["blockages"] for r in pass_results.values()], ignore_index=True)
blk_all.drop(columns=["x_nztm", "y_nztm"]).to_csv(OUT_PASS_CSV, index=False)
keep = ["osmid", "name", "highway", "length", "travel_time", "geometry"]
if os.path.exists(OUT_PASS_GPKG):
    os.remove(OUT_PASS_GPKG)
gpkg_safe(base_gdf, keep).to_file(OUT_PASS_GPKG, layer="fastest_ignore_csv", driver="GPKG")
for veh, r in pass_results.items():
    gpkg_safe(r["route_gdf"], keep + [f"status_{veh}", f"pen_{veh}"]) \
        .to_file(OUT_PASS_GPKG, layer=f"route_{veh}", driver="GPKG")
    b = r["blockages"]
    if len(b):
        gpd.GeoDataFrame(b.drop(columns=["x_nztm", "y_nztm"]),
                         geometry=gpd.points_from_xy(b["x_nztm"], b["y_nztm"]), crs=CRS_NZTM) \
            .to_file(OUT_PASS_GPKG, layer=f"blockages_{veh}", driver="GPKG")





plot_pass_routes(G, aoi, pass_results, base_gdf, src, dst, OUT_PASS_PNG)
print(f"\nSaved {OUT_PASS_PNG}, {OUT_PASS_GPKG}, {OUT_PASS_CSV}")

##### Where the 'no' ways are (to pick a test end point beyond one)
for veh in PASS_VEHICLES:
    print(f"\n'no' ways for {veh}:")
    print(flagged_ways(G, rec, veh).to_string(index=False))

In [ ]:
### Phase 3 case 2 - longer trial (Kaikoura -> Clarence) on a satellite map
"""
Reuses G and the passability already joined in the run cell above: only snapping and A* are repeated.
Outputs: case2_routes.png (same panels as above), case2_blockages.csv, case2_map.html (interactive satellite map).
Satellite tiles: Esri World Imagery (needs internet when the map is viewed).
"""
import folium

CASE2_START_LONLAT = (173.6810, -42.4030)   # Kaikoura township
CASE2_END_LONLAT = (173.9330, -42.1700)     # Clarence, about 35 km north along the SH1 coast
CASE2_PREFIX = "case2"

SAT_TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
SAT_ATTR = "Tiles &copy; Esri, Maxar, Earthstar Geographics"
LABEL_TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}"
STREET_TILES = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}"
VEH_COLOUR = {"pinzgauer": "#00e5ff", "mhov": "#ff4fd8"}
STATUS_COLOUR = {"no": "#ff2d2d", "unknown": "#ffa500"}
CONCERN_RING = {"Width": "black", "Landslide risk": "#8b4513"}   # marker outline: width = thin black, landslide = thick brown


def run_pass_case(G, rec, start_lonlat, end_lonlat, prefix):
    """Snap, route every vehicle, print the report, save the PNG and blockage CSV. Returns (results, base_gdf, src, dst)."""
    src, d_src = snap(G, start_lonlat)
    dst, d_dst = snap(G, end_lonlat)
    print(f"Snap distance (m): start={d_src:.0f}, end={d_dst:.0f}")
    if max(d_src, d_dst) > 500:
        print("Warning: a point is more than 500 m from the nearest road node")
    base_route, _, _ = astar(G, src, dst, weight="travel_time")
    base_gdf = ox.routing.route_to_gdf(G, base_route, weight="travel_time")
    results = {}
    for veh in PASS_VEHICLES:
        results[veh] = pass_route(G, src, dst, rec, veh)
        base = summarise_route(base_gdf, route_blockages(base_gdf, rec, veh), veh)
        print_pass_result(veh, results[veh], base)
    pd.concat([r["blockages"] for r in results.values()], ignore_index=True) \
        .drop(columns=["x_nztm", "y_nztm"]).to_csv(f"{prefix}_blockages.csv", index=False)
    plot_pass_routes(G, aoi, results, base_gdf, src, dst, f"{prefix}_routes.png")
    return results, base_gdf, src, dst


def _geojson(gdf, cols, style, tooltip, name, show=True):
    """Route or network lines as a folium layer in WGS84, with list attributes made printable."""
    g = gpkg_safe(gdf, cols + ["geometry"]).to_crs("EPSG:4326")
    fields = [c for c in tooltip if c in g.columns]
    return folium.GeoJson(g.to_json(), name=name, show=show, style_function=lambda _: style,
                          tooltip=folium.GeoJsonTooltip(fields=fields, aliases=[tooltip[f] for f in fields]))


def satellite_map(G, results, base_gdf, src, dst, out_html):
    """Interactive satellite map: chosen route per vehicle, fastest-ignoring-CSV route, blockages, network status."""
    m = folium.Map(tiles=None)
    folium.TileLayer(SAT_TILES, attr=SAT_ATTR, name="Satellite").add_to(m)
    folium.TileLayer(LABEL_TILES, attr=SAT_ATTR, name="Place names", overlay=True).add_to(m)
    folium.TileLayer(STREET_TILES, attr=SAT_ATTR, name="Street map", show=False).add_to(m)

    edges = ox.graph_to_gdfs(G, nodes=False)
    for veh in results:
        for s in ("unknown", "no"):
            sub = edges[edges[f"status_{veh}"] == s]
            if len(sub):
                _geojson(sub, ["name", f"status_{veh}"], {"color": STATUS_COLOUR[s], "weight": 3, "opacity": 0.9},
                         {"name": "Road", f"status_{veh}": "Status"},
                         f"Network {s} ({veh})", show=False).add_to(m)

    _geojson(base_gdf, ["name", "travel_time"], {"color": "white", "weight": 3, "dashArray": "8 6", "opacity": 0.9},
             {"name": "Road"}, "Fastest, ignoring CSV").add_to(m)
    for veh, r in results.items():
        s = r["summary"]
        _geojson(r["route_gdf"], ["name", f"status_{veh}"],
                 {"color": VEH_COLOUR[veh], "weight": 7 if veh == "mhov" else 4, "opacity": 0.95},
                 {"name": "Road", f"status_{veh}": "Status"},
                 f"{veh}: {s['status']}, {s['km']:.1f} km, {s['drive_min'] + s['unknown_delay_min']:.0f} min").add_to(m)
        fg = folium.FeatureGroup(name=f"{veh} blockages ({len(r['blockages'])})")
        for b in r["blockages"].itertuples():
            concern = getattr(b, "concern", "Width")
            pop = (f"<b>{veh}: {b.status.upper()} - {concern}</b><br>{b.name}<br>{b.reason}<br>"
                   f"OSM way {b.osm_id}, {b.length_m:.0f} m<br>km {b.km_from}-{b.km_to} along route")
            folium.CircleMarker([b.lat, b.lon], radius=9 if b.status == "no" else 6,
                                color=CONCERN_RING[concern], weight=1 if concern == "Width" else 3,
                                fill=True, fill_color=STATUS_COLOUR[b.status], fill_opacity=1,
                                tooltip=f"{concern}: {b.status}",
                                popup=folium.Popup(pop, max_width=320)).add_to(fg)
        fg.add_to(m)

    to_ll = Transformer.from_crs(CRS_NZTM, "EPSG:4326", always_xy=True)
    for n, lbl, c in ((src, "Start", "green"), (dst, "End", "blue")):
        lon, lat = to_ll.transform(G.nodes[n]["x"], G.nodes[n]["y"])
        folium.Marker([lat, lon], tooltip=lbl, icon=folium.Icon(color=c)).add_to(m)

    allg = pd.concat([base_gdf.geometry] + [r["route_gdf"].geometry for r in results.values()])
    w, s_, e, n_ = gpd.GeoSeries(allg.values, crs=CRS_NZTM).to_crs("EPSG:4326").total_bounds
    m.fit_bounds([[s_, w], [n_, e]], padding=(30, 30))
    add_folium_furniture(m)
    folium.LayerControl(collapsed=False).add_to(m)
    legend = ("<div style='position:fixed;bottom:52px;left:12px;z-index:9999;background:rgba(0,0,0,.7);color:#fff;"
              "padding:8px 12px;border-radius:6px;font:12px sans-serif;line-height:1.6'>"
              f"<span style='color:{VEH_COLOUR['pinzgauer']}'>&#9644;</span> Pinzgauer route<br>"
              f"<span style='color:{VEH_COLOUR['mhov']}'>&#9644;</span> MHOV route<br>"
              "<span style='color:#fff'>- - -</span> Fastest, ignoring CSV<br>"
              f"<span style='color:{STATUS_COLOUR['no']}'>&#9679;</span> No &nbsp;"
              f"<span style='color:{STATUS_COLOUR['unknown']}'>&#9679;</span> Unknown<br>"
              "Thin black ring: Width &nbsp; Thick brown ring: Landslide risk</div>")
    m.get_root().html.add_child(folium.Element(legend))
    m.save(out_html)
    return m


# case2_results, case2_base, case2_src, case2_dst = run_pass_case(G, rec, CASE2_START_LONLAT, CASE2_END_LONLAT, CASE2_PREFIX)
# case2_map = satellite_map(G, case2_results, case2_base, case2_src, case2_dst, f"{CASE2_PREFIX}_map.html")
# print(f"\nSaved {CASE2_PREFIX}_routes.png, {CASE2_PREFIX}_blockages.csv, {CASE2_PREFIX}_map.html")
# case2_map

In [ ]:
### Case 3 - setup: Culverden -> Kaikoura (inland, about 110 km by road)
"""
Culverden sits outside the Kaikoura AOI, so this case may need a bigger graph.
  - If both points snap within CASE3_MAX_SNAP_M of G, G is reused (no download).
  - Otherwise a corridor graph is built ONCE from a box around both points (+ CASE3_BUFFER_M),
    with the same build_graph() and CF as Phase 1, and cached to CASE3_GRAPH_CACHE.
The passability CSV is joined onto that graph with apply_passability(), as before.
Note: the CSV only covers the surveyed Kaikoura corridor, so most of this route will be "unverified".
Needs the case 2 setup cell (satellite_map, VEH_COLOUR, ...) to have been run.
"""
from shapely.geometry import box
from IPython.display import display

# CASE3_START_LONLAT / CASE3_END_LONLAT: set in the Project inputs cell at the top
CASE3_PREFIX = "case3"
CASE3_BUFFER_M = 10000                      # box margin, so the Inland Road and the SH1 coast alternative are both included
CASE3_MAX_SNAP_M = 1000                     # further than this from G -> build the corridor graph
CASE3_GRAPH_CACHE = "culverden_kaikoura_drive.graphml"   # delete if the points or buffer change

d_start = snap(G, CASE3_START_LONLAT)[1]
d_end = snap(G, CASE3_END_LONLAT)[1]
to_nztm = Transformer.from_crs("EPSG:4326", CRS_NZTM, always_xy=True)
xs, ys = zip(*(to_nztm.transform(*p) for p in (CASE3_START_LONLAT, CASE3_END_LONLAT)))
case3_box = gpd.GeoDataFrame(geometry=[box(min(xs), min(ys), max(xs), max(ys)).buffer(CASE3_BUFFER_M, join_style=2)],
                             crs=CRS_NZTM)

if max(d_start, d_end) <= CASE3_MAX_SNAP_M:
    G3, case3_frame = G, aoi
    print("Both points are inside the existing graph: reusing G")
else:
    print(f"Start is {d_start / 1000:.1f} km from G: using the Culverden-Kaikoura corridor graph")
    G3 = load_or_build_graph(case3_box.to_crs("EPSG:4326").geometry.iloc[0], CASE3_GRAPH_CACHE)
    case3_frame = case3_box
    info3 = apply_passability(G3, rec)
    print(f"CSV ways found in corridor graph: {info3['matched']} of {info3['csv_ways']}")
print(f"Graph: {len(G3.nodes)} nodes, {len(G3.edges)} edges")

In [ ]:
### Case 3 - route both vehicles (no plots)
case3_src, d_src = snap(G3, CASE3_START_LONLAT)
case3_dst, d_dst = snap(G3, CASE3_END_LONLAT)
print(f"Snap distance (m): start={d_src:.0f}, end={d_dst:.0f}")
if max(d_src, d_dst) > 500:
    print("Warning: a point is more than 500 m from the nearest road node")
case3_base_route, _, _ = astar(G3, case3_src, case3_dst, weight="travel_time")
case3_base = ox.routing.route_to_gdf(G3, case3_base_route, weight="travel_time")
case3_results = {veh: pass_route(G3, case3_src, case3_dst, rec, veh) for veh in PASS_VEHICLES}
pd.concat([r["blockages"] for r in case3_results.values()], ignore_index=True) \
    .drop(columns=["x_nztm", "y_nztm"]).to_csv(f"{CASE3_PREFIX}_blockages.csv", index=False)
print(f"Routed {', '.join(PASS_VEHICLES)}; saved {CASE3_PREFIX}_blockages.csv")

In [ ]:
### Case 3 - pinzgauer report and blockage table
veh = "pinzgauer"
r = case3_results[veh]
print_pass_result(veh, {**r, "blockages": r["blockages"].iloc[:0]},
                  summarise_route(case3_base, route_blockages(case3_base, rec, veh), veh))
display(r["blockages"].drop(columns=["vehicle", "x_nztm", "y_nztm"]))

In [ ]:
### Case 3 - mhov report and blockage table
veh = "mhov"
r = case3_results[veh]
print_pass_result(veh, {**r, "blockages": r["blockages"].iloc[:0]},
                  summarise_route(case3_base, route_blockages(case3_base, rec, veh), veh))
display(r["blockages"].drop(columns=["vehicle", "x_nztm", "y_nztm"]))

In [ ]:
### Case 3 - static map, both vehicles
fig = plot_pass_routes(G3, case3_frame, case3_results, case3_base, case3_src, case3_dst, f"{CASE3_PREFIX}_routes.png")
plt.show()

In [ ]:
### Case 3 - satellite map, both vehicles (layers can be switched on and off top right)
case3_map = satellite_map(G3, case3_results, case3_base, case3_src, case3_dst, f"{CASE3_PREFIX}_map.html")
case3_map

In [ ]:
### Case 3 - satellite map, pinzgauer only
satellite_map(G3, {"pinzgauer": case3_results["pinzgauer"]}, case3_base, case3_src, case3_dst, f"{CASE3_PREFIX}_map_pinzgauer.html")

In [ ]:
### Case 3 - satellite map, mhov only
satellite_map(G3, {"mhov": case3_results["mhov"]}, case3_base, case3_src, case3_dst, f"{CASE3_PREFIX}_map_mhov.html")

---

# Part 4 – Off-road routes (extra options after the main routing)

Parts 1–3 only drive on mapped roads. Part 4 asks: **would leaving the road for a stretch be quicker or safer than the road route?** It runs after Part 3 and changes nothing above it.

**How it works** (adapted from the Phase 2 off-road cells in `Vehicle_Routing_Latest`, Craig & Neeravh)

1. **Terrain grid:** a grid of 20 m cells over the area of the Part 3 routes (+2 km). Each cell gets an elevation (Kaikōura 2016–17 lidar, gaps filled from LINZ national 1 m lidar, then the 8 m DEM) and a land cover (ESA WorldCover 2021).
2. **Travel time between cells (per vehicle):** a move is NO-GO if it is steeper than the vehicle can climb, the ground slopes too much sideways, the land cover is impassable (bush, water, wetland), or there is no elevation. Otherwise speed = land-cover speed × a slope factor.
3. **Off-road legs:** from the route start and from both ends of every road segment that Part 3 flagged (width or landslide), a leg can cross country to any road node within 3 km, or to the route end. Each leg is added to a copy of the road graph.
4. **Same A\*, same costs:** the Part 3 `pass_route()` runs on that copy, so road and off-road options compete on the same footing: driving time + width delays + landslide delays.

**What each off-road leg is checked for**

| Check | Off-road legs | Notes |
| --- | --- | --- |
| Slope (up/down and sideways) | Yes | Vehicle limits in Project inputs, × `OFFROAD_SAFETY` |
| Land cover (bush, water, wetland) | Yes | Speeds and NO-GO classes are educated guesses |
| **Landslide risk** | **Yes** | Steered around predicted slips; if a leg still crosses one, it is labelled **Landslide risk** and charged the same delay as a road |
| **Width** | **No** | There is no width data off-road: no OpenStreetMap road and no lidar cross-sections. Every off-road leg is labelled **Off-road: no width data** |

Off-road legs cannot follow a road the width check says the vehicle cannot fit through.

**Settings:** `RUN_OFFROAD`, `OFFROAD_RES_M`, `OFFROAD_MARGIN_M`, `OFFROAD_LINK_RADIUS_M`, `OFFROAD_SAFETY` and `VEHICLE_SLOPES` are in the **Project inputs** cell (2nd code cell).

In [ ]:
# ---- Off-road routes (Part 4, runs after the main routing) ------------------
# Off-road = leaving the mapped roads across open ground. Kaikoura only, like the rest.
RUN_OFFROAD = True             # False = skip Part 4
OFFROAD_RES_M = 20             # terrain cell size (m); 10 = finer but about 4x slower
OFFROAD_MARGIN_M = 2000        # terrain window = area of the Part 3 routes + this margin on every side
OFFROAD_LINK_RADIUS_M = 3000   # an off-road leg can rejoin any road node within this straight-line distance
OFFROAD_SAFETY = 0.5           # vehicle spec slope limits x this (spec figures are for firm, dry ground)
VEHICLE_SLOPES = {             # spec maxima as rise/run (sources in Part 4)
    "pinzgauer": {"grade_max": 1.00, "side_max": 0.949},   # 100% grade, 43.5 deg side slope
    "mhov":      {"grade_max": 0.60, "side_max": 0.40},    # 60% grade, 40% side slope
}

In [ ]:
### Part 4 setup - off-road terrain data (elevation + land cover on one NZTM grid)
"""
Adapted from the Phase 2 off-road cells in Vehicle_Routing_Latest (Craig & Neeravh).
Elevation: Kaikoura 2016-17 lidar first, gaps filled from LINZ national 1 m lidar, then national 8 m DEM.
Land cover: ESA WorldCover 2021 (10 m). Both are public; only the bytes for the window are read.
Settings that people change are in the Project inputs cell; land-cover speeds below are model assumptions.
"""
import json, math, os, urllib.request
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterio.merge import merge
from rasterio.transform import from_origin
from rasterio.warp import reproject, Resampling, transform_bounds
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import dijkstra
from shapely.geometry import LineString, box

os.environ.update({"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR", "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif,.tiff",
                   "GDAL_HTTP_MULTIRANGE": "YES", "GDAL_HTTP_MERGE_CONSECUTIVE_RANGES": "YES"})

OR_DEM_COLLECTIONS = [   # tried in this order to fill each cell
    "https://nz-elevation.s3.ap-southeast-2.amazonaws.com/canterbury/kaikoura_2016-2017/dem_1m/2193/collection.json",
    "https://nz-elevation.s3.ap-southeast-2.amazonaws.com/new-zealand/new-zealand/dem_1m/2193/collection.json",
    "https://nz-elevation.s3.ap-southeast-2.amazonaws.com/new-zealand/new-zealand-contour/dem_8m/2193/collection.json",
]
OR_DEM_LABELS = ["Kaikoura 2016-17 lidar", "national 1 m lidar", "national 8 m DEM"]
WC_URL = ("https://esa-worldcover.s3.eu-central-1.amazonaws.com/v200/2021/map/"
          "ESA_WorldCover_10m_2021_v200_{tile}_Map.tif")

##### Off-road speed by ESA WorldCover class (km/h); None = impassable. Educated guesses - edit freely.
LC_SPEED_KPH = {
    10: None,   # tree cover (native bush / plantation): impassable off-track
    20: 5,      # shrubland (matagouri, scrub)
    30: 30,     # grassland (pasture, tussock)
    40: 20,     # cropland
    50: 10,     # built-up (fences, buildings, yards)
    60: 15,     # bare / sparse vegetation (scree, gravel riverbeds)
    70: None,   # snow and ice
    80: None,   # permanent water
    90: None,   # herbaceous wetland
    95: None,   # mangroves
    100: 10,    # moss and lichen
}
ROAD_LC = 254                 # cells a road passes through: always passable, slope-free between road cells
LC_SPEED_KPH[ROAD_LC] = 20    # speed when crossing / joining a road from the terrain grid (km/h)
OR_V_TOP = max(k for k in LC_SPEED_KPH.values() if k) / 3.6   # fastest off-road speed (m/s)

_OR_STAC_CACHE = {}

def or_stac_tiles(collection_url):
    """(url, NZTM bounds) of every GeoTIFF in a LINZ STAC collection; cached."""
    if collection_url not in _OR_STAC_CACHE:
        base = collection_url.rsplit("/", 1)[0] + "/"
        coll = json.load(urllib.request.urlopen(collection_url))
        hrefs = [l["href"] for l in coll["links"] if l["rel"] == "item"]

        def one(h):
            item = json.load(urllib.request.urlopen(base + h.lstrip("./")))
            href = next(a["href"] for a in item["assets"].values() if a["href"].endswith((".tif", ".tiff")))
            url = href if href.startswith("http") else base + href.lstrip("./")
            return url, transform_bounds("EPSG:4326", CRS_NZTM, *item["bbox"])

        with ThreadPoolExecutor(32) as ex:
            _OR_STAC_CACHE[collection_url] = list(ex.map(one, hrefs))
    return _OR_STAC_CACHE[collection_url]


def _or_read_tiles(tiles, bounds, res, rows, cols):
    """Mosaic the tiles touching `bounds` at `res` metres (averaged from finer data); NaN = no data."""
    xmin, ymin, xmax, ymax = bounds
    hits = [u for u, b in tiles if b[0] < xmax and b[2] > xmin and b[1] < ymax and b[3] > ymin]
    if not hits:
        return np.full((rows, cols), np.nan)
    srcs = [rasterio.open(u) for u in hits]
    try:
        arr, _ = merge(srcs, bounds=bounds, res=res, nodata=-9999, resampling=Resampling.average)
    finally:
        for s in srcs:
            s.close()
    out = arr[0].astype("float64")[:rows, :cols]
    out[out <= -9998] = np.nan
    return out


def or_load_dem(bounds, res):
    """Elevation grid for `bounds`, filled from OR_DEM_COLLECTIONS in order. Returns (dem, source code 1-3, 0 = none)."""
    xmin, ymin, xmax, ymax = bounds
    rows, cols = int(round((ymax - ymin) / res)), int(round((xmax - xmin) / res))
    dem = np.full((rows, cols), np.nan)
    src = np.zeros((rows, cols), dtype="uint8")
    for i, url in enumerate(OR_DEM_COLLECTIONS, start=1):
        gap = ~np.isfinite(dem)
        if not gap.any():
            break
        fill = _or_read_tiles(or_stac_tiles(url), bounds, res, rows, cols)
        use = gap & np.isfinite(fill)
        dem[use], src[use] = fill[use], i
    print("DEM sources: " + ", ".join(f"{l} {(src == i).mean():.0%}" for i, l in enumerate(OR_DEM_LABELS, start=1))
          + f", none {(src == 0).mean():.0%}")
    return dem, src


def or_load_landcover(transform, shape, bounds):
    """ESA WorldCover classes on the grid (most common 10 m class per cell); 0 = no data."""
    w, s, e, n = transform_bounds(CRS_NZTM, "EPSG:4326", *bounds)
    names = sorted({f"{'N' if la >= 0 else 'S'}{abs(la):02d}{'E' if lo >= 0 else 'W'}{abs(lo):03d}"
                    for la in range(int(math.floor(s / 3) * 3), int(math.floor(n / 3) * 3) + 1, 3)
                    for lo in range(int(math.floor(w / 3) * 3), int(math.floor(e / 3) * 3) + 1, 3)})
    out = np.zeros(shape, dtype="uint8")
    for name in names:
        with rasterio.open(WC_URL.format(tile=name)) as src:
            tmp = np.zeros(shape, dtype="uint8")
            reproject(rasterio.band(src, 1), tmp, dst_transform=transform, dst_crs=CRS_NZTM,
                      resampling=Resampling.mode, src_nodata=0, dst_nodata=0)
            out = np.where(out == 0, tmp, out)
    return out


def or_load_terrain(bounds, res, G_roads):
    """Terrain window for NZTM `bounds`: grid, DEM, land cover, road cells marked passable."""
    xmin = math.floor(bounds[0] / res) * res; ymin = math.floor(bounds[1] / res) * res
    xmax = math.ceil(bounds[2] / res) * res;  ymax = math.ceil(bounds[3] / res) * res
    rows, cols = int(round((ymax - ymin) / res)), int(round((xmax - xmin) / res))
    transform = from_origin(xmin, ymax, res, res)
    bnds = (xmin, ymin, xmax, ymax)
    dem, dem_src = or_load_dem(bnds, res)
    lc = or_load_landcover(transform, (rows, cols), bnds)
    t = {"transform": transform, "shape": (rows, cols), "bounds": bnds, "res": res,
         "dem": dem, "dem_src": dem_src, "lc": lc}
    return or_mark_roads(t, G_roads)


def _or_edge_geom(G_roads, u, v, d):
    return d.get("geometry") or LineString([(G_roads.nodes[u]["x"], G_roads.nodes[u]["y"]),
                                            (G_roads.nodes[v]["x"], G_roads.nodes[v]["y"])])


def _or_burn(t, shapes):
    if not shapes:
        return np.zeros(t["shape"], dtype=bool)
    return rasterize([(s, 1) for s in shapes], out_shape=t["shape"], transform=t["transform"],
                     all_touched=True, dtype="uint8").astype(bool)


def or_mark_roads(t, G_roads):
    """
    Road cells: every cell a road passes through is passable (so roads in bush can still be joined).
    Exception per vehicle: roads the width check says it can NOT fit ('no') stay as their land cover,
    so an off-road leg cannot sneak along a road that is too narrow.
    """
    edges = [(u, v, d) for u, v, d in G_roads.edges(data=True) if not d.get("offroad")]
    t["lc_raw"] = t["lc"].copy()
    t["road"] = _or_burn(t, [_or_edge_geom(G_roads, u, v, d) for u, v, d in edges])
    t["road_no"] = {veh: _or_burn(t, [_or_edge_geom(G_roads, u, v, d) for u, v, d in edges
                                      if str(d.get(f"status_{veh}")) == "no"]) for veh in PASS_VEHICLES}
    road = t["road"]
    rows, cols = t["shape"]
    print(f"Terrain {cols} x {rows} cells @ {t['res']} m | elevation {np.isfinite(t['dem']).mean():.0%} "
          f"| land cover {(t['lc'] > 0).mean():.0%} | road cells {road.mean():.1%}")
    return t

In [ ]:
### Part 4 cost model - off-road travel time between neighbouring cells, per vehicle
"""
Every cell links to its 8 neighbours. A move is NO-GO when the slope along it (grade) or across it
(side slope) is beyond the vehicle's derated limit, when either cell has impassable land cover, or when
there is no elevation. Speed = land-cover speed x a slope factor that falls to zero at the grade limit:
f = 1 - grade/grade_max uphill, 1 - 0.5 x |grade|/grade_max downhill (after FM 5-33).
Moves between two road cells skip the slope tests (roads are benched into the hillside).
The predicted landslide cells (Part 2 Random Forest) are NOT blocked here: they are flagged and charged
per off-road leg in the next cell, with the same delays as landslide-flagged roads.
"""
MOVES = [(-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)]


def or_limits(vehicle):
    """(grade_max, side_max) as rise/run after OFFROAD_SAFETY."""
    v = VEHICLE_SLOPES[vehicle]
    return v["grade_max"] * OFFROAD_SAFETY, v["side_max"] * OFFROAD_SAFETY


def _or_slices(dr, dc, rows, cols):
    return (slice(max(0, -dr), rows - max(0, dr)), slice(max(0, -dc), cols - max(0, dc)),
            slice(max(0, dr), rows - max(0, -dr)), slice(max(0, dc), cols - max(0, -dc)))


def build_offroad_graph(t, vehicle):
    """Directed sparse graph (cell -> cell) of off-road travel times in seconds for one vehicle."""
    grade_max, side_max = or_limits(vehicle)
    dem, res = t["dem"], t["res"]
    rows, cols = dem.shape
    road = t["road"] & ~t["road_no"][vehicle]             # this vehicle's usable road cells
    lc = np.where(road, ROAD_LC, t["lc_raw"])
    lut = np.full(256, np.nan)
    for code, kph in LC_SPEED_KPH.items():
        if kph:
            lut[code] = kph / 3.6
    v_lc = lut[lc]
    v_lc[~np.isfinite(dem)] = np.nan
    gy, gx = np.gradient(dem, res)
    gn = -gy
    idx = np.arange(rows * cols).reshape(rows, cols)
    road_ok = road & np.isfinite(dem)
    v_road = LC_SPEED_KPH[ROAD_LC] / 3.6
    src_all, dst_all, w_all = [], [], []
    for dr, dc in MOVES:
        ra, ca, rb, cb = _or_slices(dr, dc, rows, cols)
        L = res * math.hypot(dr, dc)
        ux, uy = dc / math.hypot(dr, dc), -dr / math.hypot(dr, dc)
        grade = (dem[rb, cb] - dem[ra, ca]) / L
        side = 0.5 * (np.abs(gx[ra, ca] * uy - gn[ra, ca] * ux) + np.abs(gx[rb, cb] * uy - gn[rb, cb] * ux))
        f = np.where(grade >= 0, 1 - grade / grade_max, 1 - 0.5 * np.abs(grade) / grade_max)
        va, vb = v_lc[ra, ca] * f, v_lc[rb, cb] * f
        with np.errstate(invalid="ignore", divide="ignore"):
            ok = (np.abs(grade) <= grade_max) & (side <= side_max) & (va > 0) & (vb > 0)
            w = (L / 2) / va + (L / 2) / vb
        on_road = road_ok[ra, ca] & road_ok[rb, cb]      # road-to-road: slope-free at the road-cell speed
        w = np.where(on_road, L / v_road, w)
        ok = (ok & np.isfinite(w)) | on_road
        src_all.append(idx[ra, ca][ok]); dst_all.append(idx[rb, cb][ok]); w_all.append(w[ok])
    n = rows * cols
    return coo_matrix((np.concatenate(w_all), (np.concatenate(src_all), np.concatenate(dst_all))),
                      shape=(n, n)).tocsr()


def or_xy_to_cell(t, x, y):
    col, row = ~t["transform"] * (x, y)
    r, c = int(math.floor(row)), int(math.floor(col))
    rows, cols = t["shape"]
    return r * cols + c if 0 <= r < rows and 0 <= c < cols else None


def or_cell_to_xy(t, i):
    rows, cols = t["shape"]
    return t["transform"] * ((i % cols) + 0.5, (i // cols) + 0.5)


def or_trace(pred, target, source):
    path = [target]
    while path[-1] != source:
        p = pred[path[-1]]
        if p < 0:
            return None
        path.append(p)
    return path[::-1]

In [ ]:
### Part 4 linking - off-road legs on a copy of the road graph, each checked against the landslide layer
"""
Where an off-road leg can start: the route start, and both ends of every road segment flagged on the
Part 3 routes (width 'no' / 'unknown' or landslide risk, on the chosen route or the fastest-ignoring-checks
route). It can end at any road node within OFFROAD_LINK_RADIUS_M, or at the route end.

Every leg is checked against the Part 2 Random Forest landslide layer and charged like a road:
  longest run inside predicted landslides >= SLIDE_RUN_BLOCK_M -> SLIDE_DELAY_MIN["... run >= 50 m"]
  any overlap below that                                     -> SLIDE_DELAY_MIN["... overlap < 50 m"]
  no overlap, but part of the leg was never looked at        -> SLIDE_DELAY_MIN["... not fully observed"]
(the Random Forest was only run on 1.28 km tiles touching the team's roads, so legs far from roads are
partly unassessed). There is NO width check off-road: no road edge, no lidar cross-sections.
The search for each leg steers around predicted landslide cells (each costs a share of the long-run delay),
so a leg only crosses a slip when there is no reasonable way round; the leg is then charged as above.
"""
OFFROAD_MAX_LEG_S = 2 * 3600       # ignore off-road legs longer than 2 h (keeps Dijkstra fast)
_OR_REASON_KEY = {                 # leg reason -> the SLIDE_DELAY_MIN key whose delay it is charged
    "landslide: predicted run >= 50 m": "landslide: predicted run >= 50 m",
    "landslide: predicted overlap < 50 m": "landslide: predicted overlap < 50 m",
    "landslide: off-road leg not fully assessed": "landslide: road not fully observed",
}


def or_landslide_layers(t):
    """Rasterise the Random Forest landslide polygons (t['slide']) and the area it looked at (t['seen'])."""
    hz = [g for g in rf_assessment["hazards"].to_crs(CRS_NZTM).geometry if g is not None and not g.is_empty]
    cov = [g for g in rf_result["coverage"].to_crs(CRS_NZTM).geometry if g is not None and not g.is_empty]
    t["slide"] = _or_burn(t, hz)
    t["seen"] = _or_burn(t, cov)
    print(f"Random Forest landslide cells: {t['slide'].mean():.1%} of the window | "
          f"assessed by the model: {t['seen'].mean():.0%} of the window")
    return t


def or_leg_landslide(t, cells):
    """Overlap (m), longest continuous run (m) inside predicted landslides, assessed share, share on road cells."""
    res = t["res"]
    slide = t["slide"].ravel()[cells]
    run = best = 0
    for s in slide:
        run = run + 1 if s else 0
        best = max(best, run)
    return {"slide_overlap_m": float(slide.sum() * res), "slide_run_m": float(best * res),
            "slide_seen_share": float(t["seen"].ravel()[cells].mean()),
            "road_cell_share": float(t["road"].ravel()[cells].mean())}


def or_leg_reason(m):
    if m["slide_run_m"] >= SLIDE_RUN_BLOCK_M:
        return "landslide: predicted run >= 50 m"
    if m["slide_overlap_m"] > 0:
        return "landslide: predicted overlap < 50 m"
    if m["slide_seen_share"] < 1:
        return "landslide: off-road leg not fully assessed"
    return None


def or_choice_graph(M, t):
    """Grid used to choose leg paths: M plus a per-cell surcharge for entering a predicted landslide cell."""
    per_cell = (60.0 * SLIDE_DELAY_MIN["landslide: predicted run >= 50 m"] / max(1.0, SLIDE_RUN_BLOCK_M / t["res"])
                if USE_LANDSLIDE else 0.0)
    C = M.copy().tocsr()
    C.data = C.data + per_cell * t["slide"].ravel()[C.indices]
    return C


def _or_drive_seconds(M, cells):
    """True off-road drive time along a cell path (no landslide surcharge)."""
    if len(cells) < 2:
        return 0.0
    return float(np.asarray(M[np.array(cells[:-1]), np.array(cells[1:])]).sum())


def _or_add_leg(Gv, u, v, cells, t, seconds, veh):
    """Add one off-road edge u -> v with the same cost attributes Part 3 routes on."""
    pts = [(Gv.nodes[u]["x"], Gv.nodes[u]["y"])] + [or_cell_to_xy(t, c) for c in cells] + [(Gv.nodes[v]["x"], Gv.nodes[v]["y"])]
    geom = LineString(pts)
    straight = math.dist(pts[0], pts[-1])
    seconds = max(float(seconds), straight / OR_V_TOP, 1e-3)   # never faster than a straight line at top speed (A* stays exact)
    m = or_leg_landslide(t, cells)
    reason = or_leg_reason(m)
    pen = 60.0 * SLIDE_DELAY_MIN[_OR_REASON_KEY[reason]] if (reason and USE_LANDSLIDE) else 0.0
    Gv.add_edge(u, v, highway="offroad", offroad=True, name="(off-road)", travel_time=seconds,
                length=float(geom.length), speed_kph=straight / seconds * 3.6, geometry=geom,
                slide_reason=reason, **m,
                **{f"cost_{veh}": seconds + pen, f"pen_{veh}": pen, f"status_{veh}": "offroad",
                   f"slide_status_{veh}": "unknown" if reason else "yes"})


def add_offroad_legs(Gv, t, M, veh, from_nodes, end_node=None):
    """Add off-road legs from each node in from_nodes to nearby road nodes, and from nearby road nodes to end_node."""
    road_nodes = [(n, c) for n, d in Gv.nodes(data=True)
                  if (c := or_xy_to_cell(t, d["x"], d["y"])) is not None]
    jobs = [(u, "out") for u in from_nodes] + ([(end_node, "in")] if end_node is not None else [])
    C = or_choice_graph(M, t)
    CT = C.T.tocsr()
    added = 0
    for u, direction in jobs:
        cu = or_xy_to_cell(t, Gv.nodes[u]["x"], Gv.nodes[u]["y"])
        if cu is None:
            continue
        dist, pred = dijkstra(C if direction == "out" else CT, directed=True, indices=cu,
                              return_predecessors=True, limit=OFFROAD_MAX_LEG_S)
        ux, uy = Gv.nodes[u]["x"], Gv.nodes[u]["y"]
        for n, c in road_nodes:
            if n == u or not np.isfinite(dist[c]):
                continue
            if math.dist((ux, uy), (Gv.nodes[n]["x"], Gv.nodes[n]["y"])) > OFFROAD_LINK_RADIUS_M:
                continue
            cells = or_trace(pred, c, cu)
            if cells is None:
                continue
            if direction == "in":
                cells = cells[::-1]                 # leg runs road node n -> end node u
            seconds = _or_drive_seconds(M, cells)
            if direction == "out":
                _or_add_leg(Gv, u, n, cells, t, seconds, veh)
            else:
                _or_add_leg(Gv, n, u, cells, t, seconds, veh)
            added += 1
    return added


def or_flagged_nodes(route, veh):
    """Both end nodes of every road segment on `route` whose width or landslide status is 'no' / 'unknown'."""
    out = set()
    for a, b in zip(route[:-1], route[1:]):
        d = min(G[a][b].values(), key=lambda e: e.get(f"cost_{veh}", math.inf))
        if d.get(f"status_{veh}") in ("no", "unknown"):
            out.update((a, b))
    return out

In [ ]:
### Part 4 run - best route per vehicle when off-road legs are allowed (Phase 3 route: start -> end)
"""
Uses G, src, dst, base_route, pass_results, rec from Part 3 and rf_assessment / rf_result from Part 2.
The Part 3 road graph G is not changed: each vehicle gets its own copy with off-road legs added.
Outputs: offroad_routes.gpkg (route_<veh>, offroad_legs_<veh>), offroad_legs.csv.
"""
if not RUN_OFFROAD:
    print("Part 4 skipped (RUN_OFFROAD = False in Project inputs)")
else:
    ##### 1. One terrain window around the Part 3 routes (+ margin), shared by both vehicles
    route_geoms = [base_gdf.geometry.union_all()] + [r["route_gdf"].geometry.union_all() for r in pass_results.values()]
    xmin, ymin, xmax, ymax = gpd.GeoSeries(route_geoms, crs=CRS_NZTM).total_bounds
    or_bounds = (xmin - OFFROAD_MARGIN_M, ymin - OFFROAD_MARGIN_M, xmax + OFFROAD_MARGIN_M, ymax + OFFROAD_MARGIN_M)
    or_t = or_load_terrain(or_bounds, OFFROAD_RES_M, G)
    or_landslide_layers(or_t)

    offroad_results, leg_tables = {}, []
    for veh in PASS_VEHICLES:
        g_lim, s_lim = or_limits(veh)
        print(f"\n=== {veh}: off-road limits {math.degrees(math.atan(g_lim)):.0f} deg grade, "
              f"{math.degrees(math.atan(s_lim)):.0f} deg side slope (spec x {OFFROAD_SAFETY}) ===")
        ##### 2. Off-road legs on a copy of the road graph
        Gv = G.copy()
        M = build_offroad_graph(or_t, veh)
        starts = {src} | or_flagged_nodes(pass_results[veh]["route"], veh) | or_flagged_nodes(base_route, veh)
        n_legs = add_offroad_legs(Gv, or_t, M, veh, starts, end_node=dst)
        print(f"Off-road legs added: {n_legs} (from the start and {len(starts) - 1} flagged-road ends)")
        ##### 3. Same A*, same costs, same Dijkstra check as Part 3
        res = pass_route(Gv, src, dst, rec, veh)
        res["graph"] = Gv
        offroad_results[veh] = res
        ##### 4. Compare with the road-only route and list the off-road legs
        s_road, s_off = pass_results[veh]["summary"], res["summary"]
        t_road = s_road["drive_min"] + s_road["unknown_delay_min"]
        t_off = s_off["drive_min"] + s_off["unknown_delay_min"]
        rg = res["route_gdf"]
        is_off = rg["offroad"].eq(True) if "offroad" in rg.columns else pd.Series(False, index=rg.index)
        legs = rg[is_off]
        print(f"Road only (Part 3): {s_road['km']:.1f} km, {t_road:.1f} min incl. delays")
        print(f"Off-road allowed:   {s_off['km']:.1f} km, {t_off:.1f} min incl. delays "
              f"({legs['length'].sum() / 1000:.2f} km off-road in {len(legs)} legs)")
        if legs.empty:
            print("No off-road leg beats the roads for this vehicle: the road route stays best.")
            continue
        table = pd.DataFrame({
            "vehicle": veh,
            "leg": range(1, len(legs) + 1),
            "length_m": legs["length"].round(0).to_numpy(),
            "drive_min": (legs["travel_time"] / 60).round(1).to_numpy(),
            "landslide_delay_min": (legs[f"pen_{veh}"] / 60).round(1).to_numpy(),
            "slide_overlap_m": legs["slide_overlap_m"].to_numpy(),
            "slide_run_m": legs["slide_run_m"].to_numpy(),
            "slide_assessed_pct": (100 * legs["slide_seen_share"]).round(0).to_numpy(),
            "on_road_cells_pct": (100 * legs["road_cell_share"]).round(0).to_numpy(),
            "concern": ["Off-road: no width data"
                        + ("; Landslide risk" if o > 0 else "")
                        + ("; Landslide not fully assessed" if sh < 1 else "")
                        for o, sh in zip(legs["slide_overlap_m"], legs["slide_seen_share"])],
            "reason": legs["slide_reason"].fillna("").to_numpy(),
        })
        leg_tables.append(table)
        display(table)
        if len(res["blockages"]):
            print("Road segments still used on this route:")
            display(res["blockages"].drop(columns=["vehicle", "x_nztm", "y_nztm"], errors="ignore"))

    ##### 5. Save
    if os.path.exists("offroad_routes.gpkg"):
        os.remove("offroad_routes.gpkg")
    keep = ["name", "highway", "length", "travel_time", "offroad", "slide_reason", "slide_overlap_m",
            "slide_run_m", "slide_seen_share", "geometry"]
    for veh, res in offroad_results.items():
        rg = res["route_gdf"]
        gpkg_safe(rg, [c for c in keep if c in rg.columns]).to_file("offroad_routes.gpkg", layer=f"route_{veh}", driver="GPKG")
    if leg_tables:
        pd.concat(leg_tables, ignore_index=True).to_csv("offroad_legs.csv", index=False)
    print("\nSaved offroad_routes.gpkg" + (", offroad_legs.csv" if leg_tables else ""))

In [ ]:
### Part 4 map - road-only route vs route with off-road legs, over terrain and predicted landslides
from matplotlib.colors import LightSource

if RUN_OFFROAD:
    xmin, ymin, xmax, ymax = or_t["bounds"]
    ext = (xmin, xmax, ymin, ymax)
    dem = or_t["dem"]
    shade = LightSource(azdeg=315, altdeg=45).hillshade(
        np.nan_to_num(dem, nan=np.nanmin(dem) if np.isfinite(dem).any() else 0.0), dx=or_t["res"], dy=or_t["res"])
    nogo_lc = np.isin(or_t["lc_raw"], [k for k, v in LC_SPEED_KPH.items() if not v]) & ~or_t["road"]
    nogo = nogo_lc | ~np.isfinite(dem)
    hz = rf_assessment["hazards"].to_crs(CRS_NZTM).cx[xmin:xmax, ymin:ymax]
    fig, axes = plt.subplots(1, len(PASS_VEHICLES), figsize=(9 * len(PASS_VEHICLES), 9))
    for ax, veh in zip(np.atleast_1d(axes), PASS_VEHICLES):
        res, Gv = offroad_results[veh], offroad_results[veh]["graph"]
        ax.imshow(shade, cmap="gray", extent=ext, alpha=0.8)
        ax.imshow(np.ma.masked_where(~nogo, nogo), cmap="Greys", vmin=0, vmax=1.5, extent=ext, alpha=0.35)
        if len(hz):
            hz.plot(ax=ax, color="orange", alpha=0.6, zorder=2)
        for u, v, d in G.edges(data=True):
            ax.plot(*_or_edge_geom(G, u, v, d).xy, color="#555555", lw=0.5, zorder=2)
        pass_results[veh]["route_gdf"].plot(ax=ax, color="blue", lw=4, alpha=0.45, zorder=3)
        rg = res["route_gdf"]
        off = rg["offroad"].eq(True) if "offroad" in rg.columns else pd.Series(False, index=rg.index)
        if (~off).any():
            rg[~off].plot(ax=ax, color="red", lw=2, zorder=4)
        if off.any():
            rg[off].plot(ax=ax, color="magenta", lw=2.5, linestyle="--", zorder=5)
        for n, c, lbl in ((src, "green", "start"), (dst, "blue", "end")):
            ax.scatter(G.nodes[n]["x"], G.nodes[n]["y"], s=80, c=c, edgecolors="black", zorder=6)
        s_road, s_off = pass_results[veh]["summary"], res["summary"]
        ax.set_title(f"{VEHICLE_WIDTHS[veh]['label']}\nroad only {s_road['drive_min'] + s_road['unknown_delay_min']:.0f} min"
                     f" | off-road allowed {s_off['drive_min'] + s_off['unknown_delay_min']:.0f} min")
        ax.plot([], [], color="blue", lw=4, alpha=0.45, label="Part 3 route (roads only)")
        ax.plot([], [], color="red", lw=2, label="off-road allowed: road legs")
        ax.plot([], [], color="magenta", lw=2.5, ls="--", label="off-road legs (no width data)")
        ax.fill_between([], [], color="orange", alpha=0.6, label="predicted landslide (Random Forest)")
        ax.fill_between([], [], color="grey", alpha=0.35, label="NO-GO land cover / no elevation")
        ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_aspect("equal")
        add_map_furniture(ax)
        ax.legend(loc="lower right", fontsize=8)
    fig.tight_layout()
    fig.savefig("offroad_routes.png", dpi=150, bbox_inches="tight")
    plt.show()

## Off-road routes: limitations, risks and next steps

**Read every off-road leg as "possible on paper", not "confirmed drivable".**

**Data gaps**

* **No width or clearance data off-road.** Fences, gates, gullies, boulders and trees narrower than a vehicle are not in any of our datasets. *Next step:* look for data that could give this, such as LINZ topographic layers (fences, tracks, gates), drone imagery or photogrammetry of candidate legs, or local knowledge from farmers and DOC.
* **Landslide checks are partial off-road.** The Random Forest only ran on 1.28 km tiles touching the team's roads, so legs that go further from a road are partly unassessed (`slide_assessed_pct` in the leg table). *Next step:* run the Random Forest over the whole off-road window.
* **Land cover is from 2021, not 2016,** at 10 m resolution. Recent slips, cleared bush and new fences are not reflected.
* **Elevation is averaged to 20 m cells.** Short steep banks, ditches and road cuttings smaller than a cell are smoothed out, so a leg may need to climb a bank the model cannot see.
* **Rivers and fords are only modelled as "permanent water"** in the land cover. Braided gravel riverbeds count as passable bare ground, regardless of flow on the day.

**Model assumptions**

* **Land-cover speeds and NO-GO classes are educated guesses,** not measurements (`LC_SPEED_KPH`).
* **Slope limits come from manufacturer specs** for firm, dry ground, halved by `OFFROAD_SAFETY`. Wet ground, loose scree and loaded vehicles do worse.
* **Ground strength is not modelled:** soft or saturated soil, liquefaction and fresh debris can stop a heavy vehicle on gentle slopes.
* **Off-road legs only start at the route start and at flagged roads,** and only rejoin roads within 3 km. Wider searches are possible but slower.

**Operational and safety risks**

* **Aftershocks** can trigger new slips on slopes the model rates as passable.
* **Private land, stock and access permission** apply to every off-road leg.
* **Communications and recovery are harder** off-road: there is no road for a breakdown truck.
* **Environmental and cultural sites** (wāhi tapu, DOC land, waterways) are not checked.

**Suggested next steps**

1. Ground-truth a few off-road legs against aerial imagery (the leg table lists their lengths and landslide checks).
2. Add LINZ fences, tracks and rivers, and the full-window Random Forest, then re-run.
3. Bring in the other mitigations from `Vehicle_Routing_Latest` (bridges and weight limits, rivers and fording, damaged-road speeds, re-planning as reports arrive) once these routes are agreed.